# 11c. LightGBM Reranking — Full (Facial Skincare)

This notebook estimates the Full condition on the fixed personalized-retrieval candidate pool for the 2,288-case benchmark. Its 56-feature prior-aware registry combines candidate-common evidence with strict All Prior history depth and concentration, functional-family affinity, recency, and Brand-history association. The previously-reviewed-item block remains excluded.

Separate LambdaRank models are fitted on the personalized pool at the five fixed candidate depths using five user-group-disjoint outer folds, inner temporal validation, NDCG@5 early stopping, and out-of-fold scoring. Strict cold cases reuse the same-family Base out-of-fold scores and ranks. The notebook exports candidate rankings, per-query and aggregate metrics, fold and model contracts, runtime summaries, feature manifests, learning-curve diagnostics, fallback-identity checks, and leakage-control checks.


In [1]:
# ==== Environment Setup ====
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!pip -q install -U pyarrow tqdm lightgbm scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 675.5/675.5 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 117.7 MB/s eta 0:00:00


## 1. Environment / Paths / Config

In [5]:
# ==== Environment / Paths / Config ====
# Imports, core config, paths, and Notebook 10-compatible runtime framework

import ast
import gc
import hashlib
import json
import math
import os
import platform
import random
import re
import socket
import time
from collections import Counter, defaultdict
from contextlib import contextmanager
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 220)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)

NOTEBOOK_NAME = '11c_personalized_rerank_lightgbm_face.ipynb'
CATEGORY_ID = "face"
CATEGORY_FOLDER = "facial_skincare"
CATEGORY_LABEL = "Facial Skincare"
EXPERIMENT_CONDITION = 'full_pipeline_personalization'
CANDIDATE_SOURCE_LABEL = 'notebook09_personalized_winner_top1000_face'
STAGE = 'stage2_personalized_rerank_lightgbm_temporal'

# Notebook 09 personalized retrieval winner candidate pool.
STAGE1_QUERY_METHOD = "C"
DEFAULT_CANDIDATE_POOL_TYPE = None
DEFAULT_RETRIEVAL_METHOD = None
DEFAULT_RETRIEVAL_METHOD_LABEL = None
CANDIDATE_POOL_TYPE = None
RETRIEVAL_METHOD = None
RETRIEVAL_METHOD_LABEL = None

RERANKING_METHOD = "lightgbm"
BASELINE_RERANK_METHOD = "stage1_baseline"
OUTPUT_RERANK_METHOD = "lightgbm_rerank"
EXPECTED_RERANK_METHODS = [BASELINE_RERANK_METHOD, OUTPUT_RERANK_METHOD]

NONPERSONALIZED_RETRIEVAL_NAME_BY_LEGACY = {}


def canonical_nonpersonalized_retrieval_method(value):
    text = "" if value is None else str(value).strip()
    return NONPERSONALIZED_RETRIEVAL_NAME_BY_LEGACY.get(text, text)


POOL_K = 1000
POOL_DEPTHS = [100, 300, 500, 700, 1000]
MAX_POOL_DEPTH = int(max(POOL_DEPTHS))
REPORT_POOL_DEPTH = 1000
RUNTIME_REPORT_POOL_DEPTH = REPORT_POOL_DEPTH

# Stage-2 reporting follows the thesis/Notebook 10 convention.
RERANK_EVAL_KS = [1, 5, 10]
EVAL_KS = [1, 5, 10, 100, 300, 500, 700, 1000]
PRIMARY_STAGE2_METRIC = "NDCG@5"
PRIMARY_STAGE2_METRICS = ["NDCG@5"]
SECONDARY_STAGE2_METRICS = ["HitRate@5", "NDCG@1", "HitRate@1", "MRR@5"]
SUPPLEMENTARY_STAGE2_METRICS = ["NDCG@10", "HitRate@10", "MRR@10", "NDCG@100"]

COMMON_STAGE2_COMPARISON_METRICS = [
    "NDCG@5",
    "HitRate@5",
    "NDCG@1",
    "HitRate@1",
    "MRR@5",
    "NDCG@10",
    "HitRate@10",
    "MRR@10",
]
PREFERENCE_ALIGNMENT_K = 5

REGIME_ORDER = ["cold", "weak", "moderate", "strong"]
EXPECTED_REGIMES = list(REGIME_ORDER)
REGIME_SOURCE_PRIORITY = ("regime_cand", "regime")
REGIME_COLD_PRIOR_REVIEW_N = 0
REGIME_WEAK_PRIOR_REVIEW_MIN = 1
REGIME_WEAK_PRIOR_REVIEW_MAX = 4
REGIME_MODERATE_PRIOR_REVIEW_MIN = 5
REGIME_MODERATE_PRIOR_REVIEW_MAX = 9
REGIME_STRONG_PRIOR_REVIEW_MIN = 10

# Supervised LightGBM should learn the value of personalization from features rather than
# receiving hand-set regime caps. Keep explicit regime personalization weights uniform and
# expose regime/history-density variables as model features and diagnostics.
REGIME_PERSONALIZATION_WEIGHT_POLICY = "uniform_no_manual_regime_cap_for_supervised_ltr"
REGIME_PERSONALIZATION_WEIGHT = {regime: 1.0 for regime in REGIME_ORDER}
USE_EXPLICIT_REGIME_PERSONALIZATION_WEIGHTS = False

REQUIRED_CANDIDATE_COLUMNS = [
    "case_id",
    "query_id",
    "user_id",
    "regime",
    "gt_item_id",
    "candidate_item_id",
    "candidate_rank",
    "candidate_score",
    "candidate_pool_type",
    "retrieval_method",
    "retrieval_method_label",
    "is_gt",
]

DEDUP_HISTORY_BY_USER_ITEM = False

BRAND_QUERY_ENABLED = False
BRAND_CANDIDATE_VISIBLE = True
USER_BRAND_AFFINITY_ENABLED = True
BRAND_RERANKING_ENABLED = bool(USER_BRAND_AFFINITY_ENABLED)
HISTORY_SOURCE = "all_prior"
HISTORICAL_POPULATION_REVIEW_SIGNALS_IN_USER_PROFILE = False
CANDIDATE_BRAND_SOURCE_COLUMN = "brand_facet_text"
QUERY_BRAND_QC_COLUMN = "brand_or_name_leak_flag"
USE_CANDIDATE_SCORE_FEATURE = False
MODEL_CANDIDATE_SCORE_FEATURE_POLICY = "exclude_raw_method_score_keep_within_query_pool_normalized_score"

PROJECT_ROOT = Path("/content/drive/MyDrive/thesis_recsys/categories/facial_skincare")
os.chdir(PROJECT_ROOT)
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
STAGE1_POOL_DIR = OUTPUTS_DIR / "stage1_candidate_pools"
CANDIDATE_POOL_DIR = STAGE1_POOL_DIR / "by_method"
CANDIDATE_POOL_MANIFEST_OUTPUT_KEY = "personalized_winner_long"
CANDIDATE_POOL_PATH = CANDIDATE_POOL_DIR / 'personalized_winner_top1000_face.parquet'
CANDIDATE_POOL_MANIFEST_PATH = STAGE1_POOL_DIR / 'stage1_candidate_pool_export_manifest_face.json'
CANDIDATE_POOL_SUMMARY_PATH = STAGE1_POOL_DIR / 'winner_candidate_pool_summary_face.csv'
QUERY_CACHE_PARQUET = PROJECT_ROOT / 'outputs/query_cache/face_queries.parquet'
QUERY_CACHE_SUMMARY_PATH = PROJECT_ROOT / 'outputs/query_summary/face_query_generation_qc.parquet'
QUERY_CACHE_CONFIG_PATH = PROJECT_ROOT / 'outputs/query_summary/face_queries_config.json'

PROCESSED_ITEMS_DIR = PROJECT_ROOT / "data" / "processed" / "items"
USER_SAMPLING_DIR = PROJECT_ROOT / "data" / "processed" / "user_sampling"
USER_SAMPLE_PARQUET = USER_SAMPLING_DIR / "face_user_regime_sample.parquet"
PRIOR_HISTORY_PARQUET = USER_SAMPLING_DIR / "face_user_prior_review_history.parquet"
TARGET_CASE_METADATA_PARQUET = USER_SAMPLING_DIR / "face_target_case_metadata.parquet"

ITEM_SCHEMA_PARQUET = PROCESSED_ITEMS_DIR / "face_item_schema_full.parquet"
ITEM_SCHEMA_BASE_PARQUET = PROCESSED_ITEMS_DIR / "face_item_schema.parquet"
ITEM_DOCS_PARQUET = PROCESSED_ITEMS_DIR / "face_item_docs.parquet"
GRAPH_EDGES_PARQUET = PROCESSED_ITEMS_DIR / "face_item_graph_edges.parquet"
PRODUCT_NODES_PARQUET = PROCESSED_ITEMS_DIR / "face_item_graph_product_nodes.parquet"
ENTITY_NODES_PARQUET = PROCESSED_ITEMS_DIR / "face_item_graph_entity_nodes.parquet"
ITEMS_FACETS_PARQUET = PROCESSED_ITEMS_DIR / "face_items_facets.parquet"

OUT_DIR = PROJECT_ROOT / 'outputs/stage2_personalized_rerank/lightgbm_full'
OUT_DIR.mkdir(parents=True, exist_ok=True)
SHARED_MODEL_ARTIFACT_DIR = OUT_DIR

P2Q_OOF_PREDICTION_PATH = OUTPUTS_DIR / 'stage2_nonpersonalized_rerank' / 'lightgbm_no_prior' / 'p2q_oof_candidate_predictions.parquet'
P2Q_OOF_PREDICTION_MANIFEST_PATH = P2Q_OOF_PREDICTION_PATH.with_name('p2q_oof_candidate_predictions_manifest.json')

RUNTIME_BRANCH = 'personalized_winner_lightgbm_prior'
RUNTIME_ROWS = []
NOTEBOOK_TIMER_START = time.perf_counter()

print("Libraries loaded.")
print("Random seed:", RANDOM_SEED)
print("Candidate pool path:", CANDIDATE_POOL_PATH)
print("Candidate input exists:", CANDIDATE_POOL_PATH.exists())



Libraries loaded.
Random seed: 42
Candidate pool path: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage1_candidate_pools/by_method/personalized_winner_top1000_face.parquet
Candidate input exists: True


In [12]:
# ==== Environment / Paths / Config — Part 2 ====


def load_json_if_exists(path_obj):
    path = Path(path_obj)
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8-sig") as f:
        return json.load(f)



candidate_pool_manifest = load_json_if_exists(CANDIDATE_POOL_MANIFEST_PATH)
if candidate_pool_manifest:
    manifest_output_paths = candidate_pool_manifest.get("output_paths", {})
    manifest_personalized_winner_path = manifest_output_paths.get(CANDIDATE_POOL_MANIFEST_OUTPUT_KEY)
    if manifest_personalized_winner_path and Path(manifest_personalized_winner_path) != CANDIDATE_POOL_PATH:
        raise RuntimeError("Notebook 09 query-only winner path does not match CANDIDATE_POOL_PATH.")
    manifest_personalized_winner_label = candidate_pool_manifest.get("selected_personalized_method_label")
    manifest_personalized_winner_key = candidate_pool_manifest.get("selected_personalized_method_slug")
    if manifest_personalized_winner_label:
        DEFAULT_CANDIDATE_POOL_TYPE = manifest_personalized_winner_label
        DEFAULT_RETRIEVAL_METHOD = manifest_personalized_winner_label
        DEFAULT_RETRIEVAL_METHOD_LABEL = manifest_personalized_winner_label
        CANDIDATE_POOL_TYPE = manifest_personalized_winner_label
        RETRIEVAL_METHOD = manifest_personalized_winner_label
        RETRIEVAL_METHOD_LABEL = manifest_personalized_winner_label
        NONPERSONALIZED_RETRIEVAL_NAME_BY_LEGACY[manifest_personalized_winner_label] = manifest_personalized_winner_label
    if manifest_personalized_winner_key and manifest_personalized_winner_label:
        NONPERSONALIZED_RETRIEVAL_NAME_BY_LEGACY[manifest_personalized_winner_key] = manifest_personalized_winner_label

if not all([CANDIDATE_POOL_TYPE, RETRIEVAL_METHOD, RETRIEVAL_METHOD_LABEL]):
    raise RuntimeError(
        "Notebook 09 manifest did not provide the query-only retrieval winner identity."
    )


def record_runtime_step(
    step_name,
    runtime_sec,
    *,
    method=None,
    branch=None,
    pool_depth=None,
    n_queries=None,
    n_candidates=None,
    runtime_measurement_type="wall_clock_step",
    derived_from_existing_runtime=False,
    error="",
    **extra_metadata,
):
    n_queries_value = np.nan if n_queries is None else n_queries
    n_candidates_value = np.nan if n_candidates is None else n_candidates
    runtime_sec_value = np.nan if runtime_sec is None else float(runtime_sec)

    row = {
        "category": CATEGORY_FOLDER,
        "category_id": CATEGORY_ID,
        "notebook_name": NOTEBOOK_NAME,
        "method": method or RERANKING_METHOD,
        "branch": branch or RUNTIME_BRANCH,
        "step_name": step_name,
        "pool_depth": pool_depth,
        "n_queries": n_queries_value,
        "n_candidates": n_candidates_value,
        "runtime_sec": runtime_sec_value,
        "runtime_sec_per_query": runtime_sec_value / n_queries_value if pd.notna(runtime_sec_value) and pd.notna(n_queries_value) and n_queries_value else np.nan,
        "runtime_sec_per_candidate": runtime_sec_value / n_candidates_value if pd.notna(runtime_sec_value) and pd.notna(n_candidates_value) and n_candidates_value else np.nan,
        "error": error,
        "runtime_measurement_type": runtime_measurement_type,
        "derived_from_existing_runtime": bool(derived_from_existing_runtime),
    }
    row.update(extra_metadata)
    RUNTIME_ROWS.append(row)
    return row


@contextmanager
def runtime_step(
    step_name,
    *,
    method=None,
    branch=None,
    pool_depth=None,
    n_queries=None,
    n_candidates=None,
    runtime_measurement_type="wall_clock_step",
    derived_from_existing_runtime=False,
    **extra_metadata,
):
    start_time = time.perf_counter()
    error_message = ""
    try:
        yield
    except Exception as exc:
        error_message = f"{type(exc).__name__}: {exc}"
        raise
    finally:
        record_runtime_step(
            step_name,
            time.perf_counter() - start_time,
            method=method,
            branch=branch,
            pool_depth=pool_depth,
            n_queries=n_queries,
            n_candidates=n_candidates,
            runtime_measurement_type=runtime_measurement_type,
            derived_from_existing_runtime=derived_from_existing_runtime,
            error=error_message,
            **extra_metadata,
        )


def _runtime_sum(runtime_steps_df, step_names, pool_depth=REPORT_POOL_DEPTH, method=RERANKING_METHOD):
    if runtime_steps_df is None or runtime_steps_df.empty:
        return 0.0
    rows = runtime_steps_df.copy()
    if "method" in rows.columns:
        rows = rows[rows["method"].astype(str).eq(str(method))]
    if "pool_depth" in rows.columns:
        rows = rows[pd.to_numeric(rows["pool_depth"], errors="coerce").eq(int(pool_depth))]
    rows = rows[rows["step_name"].astype(str).isin(list(step_names))]
    vals = pd.to_numeric(rows["runtime_sec"], errors="coerce").dropna()
    return float(vals.sum()) if len(vals) else 0.0


def _metric_from_summary(summary_df, method_name, metric_col):
    if summary_df is None or summary_df.empty or metric_col not in summary_df.columns:
        return np.nan
    rows = summary_df.copy()
    if "rerank_method" in rows.columns:
        rows = rows[rows["rerank_method"].astype(str).eq(method_name)]
    if rows.empty:
        return np.nan
    vals = pd.to_numeric(rows[metric_col], errors="coerce").dropna()
    return float(vals.iloc[0]) if len(vals) else np.nan


def export_runtime_logs(output_dir, report_pool_depth=REPORT_POOL_DEPTH):
    output_dir = Path(output_dir)
    export_start = time.perf_counter()

    report_rows = per_query_report_df[
        per_query_report_df["rerank_method"].astype(str).eq(OUTPUT_RERANK_METHOD)
    ]
    candidate_report_rows = reranked_candidates_report_df[
        reranked_candidates_report_df["rerank_method"].astype(str).eq(OUTPUT_RERANK_METHOD)
    ]

    n_queries_at_report = float(report_rows["query_id"].nunique())
    n_candidates_at_report = float(len(candidate_report_rows))

    export_runtime_sec = time.perf_counter() - export_start
    existing_export_row = any(
        str(row.get("step_name", "")) == "export_outputs"
        and int(row.get("pool_depth") or report_pool_depth) == int(report_pool_depth)
        for row in RUNTIME_ROWS
    )
    if not existing_export_row:
        record_runtime_step(
            "export_outputs",
            export_runtime_sec,
            pool_depth=report_pool_depth,
            n_queries=n_queries_at_report,
            n_candidates=n_candidates_at_report,
        )

    total_runtime_sec = time.perf_counter() - NOTEBOOK_TIMER_START
    record_runtime_step(
        "total_notebook",
        total_runtime_sec,
        pool_depth=report_pool_depth,
        n_queries=n_queries_at_report,
        n_candidates=n_candidates_at_report,
        runtime_measurement_type="wall_clock_total",
    )

    runtime_steps_df = pd.DataFrame(RUNTIME_ROWS)
    runtime_steps_df["pool_depth"] = pd.to_numeric(runtime_steps_df["pool_depth"], errors="coerce")

    online_steps = ["feature_preparation", "model_scoring", "ranking_sorting"]
    offline_steps = ["model_fit_or_tuning"]
    evaluation_export_steps = ["evaluation", "export_outputs"]

    online_runtime_sec = _runtime_sum(runtime_steps_df, online_steps, report_pool_depth)
    offline_runtime_sec = _runtime_sum(runtime_steps_df, offline_steps, report_pool_depth)
    evaluation_export_runtime_sec = _runtime_sum(runtime_steps_df, evaluation_export_steps, report_pool_depth)

    ndcg_at_5 = _metric_from_summary(summary_overall_df, OUTPUT_RERANK_METHOD, "NDCG@5")
    hitrate_at_5 = _metric_from_summary(summary_overall_df, OUTPUT_RERANK_METHOD, "HitRate@5")
    mrr_at_5 = _metric_from_summary(summary_overall_df, OUTPUT_RERANK_METHOD, "MRR@5")
    baseline_ndcg_at_5 = _metric_from_summary(summary_overall_df, BASELINE_RERANK_METHOD, "NDCG@5")
    baseline_hitrate_at_5 = _metric_from_summary(summary_overall_df, BASELINE_RERANK_METHOD, "HitRate@5")
    delta_ndcg_at_5 = ndcg_at_5 - baseline_ndcg_at_5 if pd.notna(ndcg_at_5) and pd.notna(baseline_ndcg_at_5) else np.nan
    delta_hitrate_at_5 = hitrate_at_5 - baseline_hitrate_at_5 if pd.notna(hitrate_at_5) and pd.notna(baseline_hitrate_at_5) else np.nan

    runtime_method_summary_at1000_df = pd.DataFrame([{
        "category": CATEGORY_FOLDER,
        "category_id": CATEGORY_ID,
        "notebook_name": NOTEBOOK_NAME,
        "branch": RUNTIME_BRANCH,
        "method": RERANKING_METHOD,
        "rerank_method": OUTPUT_RERANK_METHOD,
        "pool_depth": int(report_pool_depth),
        "n_queries": n_queries_at_report,
        "n_candidates": n_candidates_at_report,
        "online_operation_runtime_sec": online_runtime_sec,
        "offline_preparation_runtime_sec": offline_runtime_sec,
        "evaluation_export_runtime_sec": evaluation_export_runtime_sec,
        "total_notebook_runtime_sec": total_runtime_sec,
        "runtime_sec_per_query": online_runtime_sec / n_queries_at_report if n_queries_at_report else np.nan,
        "runtime_sec_per_candidate": online_runtime_sec / n_candidates_at_report if n_candidates_at_report else np.nan,
        "queries_per_second": n_queries_at_report / online_runtime_sec if online_runtime_sec else np.nan,
        "candidates_per_second": n_candidates_at_report / online_runtime_sec if online_runtime_sec else np.nan,
        "runtime_scope": "stage2_online_reranking_excluding_stage1_retrieval",
        "candidate_pool_type": CANDIDATE_POOL_TYPE,
        "retrieval_method": RETRIEVAL_METHOD,
        "retrieval_method_label": RETRIEVAL_METHOD_LABEL,
        "reranking_method": RERANKING_METHOD,
        "sample_scope": "native",
        "common_sample_filtering_introduced": False,
        "primary_metric": PRIMARY_STAGE2_METRIC,
        "ndcg_at_5": ndcg_at_5,
        "hitrate_at_5": hitrate_at_5,
        "mrr_at_5": mrr_at_5,
        "baseline_ndcg_at_5": baseline_ndcg_at_5,
        "baseline_hitrate_at_5": baseline_hitrate_at_5,
        "delta_ndcg_at_5_vs_baseline": delta_ndcg_at_5,
        "delta_hitrate_at_5_vs_baseline": delta_hitrate_at_5,
        "delta_ndcg5_per_100sec_online": (delta_ndcg_at_5 / online_runtime_sec) * 100 if pd.notna(delta_ndcg_at_5) and online_runtime_sec else np.nan,
        "delta_hitrate5_per_100sec_online": (delta_hitrate_at_5 / online_runtime_sec) * 100 if pd.notna(delta_hitrate_at_5) and online_runtime_sec else np.nan,
    }])

    runtime_notebook_summary_df = runtime_method_summary_at1000_df.copy()
    runtime_notebook_summary_df["total_logged_step_runtime_sec"] = pd.to_numeric(
        runtime_steps_df["runtime_sec"],
        errors="coerce",
    ).sum()

    runtime_method_components_at1000_df = pd.DataFrame([{
        "category_id": CATEGORY_ID,
        "category_folder": CATEGORY_FOLDER,
        "method": RERANKING_METHOD,
        "rerank_method": OUTPUT_RERANK_METHOD,
        "pool_depth": int(report_pool_depth),
        "n_queries": n_queries_at_report,
        "feature_preparation_runtime_sec": _runtime_sum(runtime_steps_df, ["feature_preparation"], report_pool_depth),
        "model_fit_or_tuning_runtime_sec": _runtime_sum(runtime_steps_df, ["model_fit_or_tuning"], report_pool_depth),
        "model_scoring_runtime_sec": _runtime_sum(runtime_steps_df, ["model_scoring"], report_pool_depth),
        "ranking_sorting_runtime_sec": _runtime_sum(runtime_steps_df, ["ranking_sorting"], report_pool_depth),
        "evaluation_runtime_sec": _runtime_sum(runtime_steps_df, ["evaluation"], report_pool_depth),
        "export_outputs_runtime_sec": _runtime_sum(runtime_steps_df, ["export_outputs"], report_pool_depth),
        "online_operation_runtime_sec": online_runtime_sec,
        "runtime_scope": "stage2_online_reranking_excluding_stage1_retrieval",
        "candidate_pool_type": CANDIDATE_POOL_TYPE,
        "retrieval_method": RETRIEVAL_METHOD,
        "retrieval_method_label": RETRIEVAL_METHOD_LABEL,
        "reranking_method": RERANKING_METHOD,
    }])

    runtime_pool_depth_diagnostic_df = runtime_by_pool_depth_df.copy()
    runtime_pool_depth_diagnostic_df["notebook_name"] = NOTEBOOK_NAME
    runtime_pool_depth_diagnostic_df["branch"] = RUNTIME_BRANCH
    runtime_pool_depth_diagnostic_df["summary_scope"] = "all_pool_depths"
    runtime_pool_depth_diagnostic_df["sample_scope"] = "native"
    runtime_pool_depth_diagnostic_df["candidate_pool_type"] = CANDIDATE_POOL_TYPE
    runtime_pool_depth_diagnostic_df["retrieval_method"] = RETRIEVAL_METHOD
    runtime_pool_depth_diagnostic_df["retrieval_method_label"] = RETRIEVAL_METHOD_LABEL
    runtime_pool_depth_diagnostic_df["reranking_method"] = RERANKING_METHOD

    runtime_steps_df.to_csv(output_dir / "runtime_steps.csv", index=False)
    runtime_notebook_summary_df.to_csv(output_dir / "runtime_notebook_summary.csv", index=False)
    runtime_method_summary_at1000_df.to_csv(output_dir / "runtime_method_summary_at1000.csv", index=False)
    runtime_pool_depth_diagnostic_df.to_csv(output_dir / "runtime_pool_depth_diagnostic.csv", index=False)
    runtime_method_components_at1000_df.to_csv(output_dir / "runtime_method_components_at1000_face.csv", index=False)

    return (
        runtime_steps_df,
        runtime_notebook_summary_df,
        runtime_method_summary_at1000_df,
        runtime_pool_depth_diagnostic_df,
        runtime_method_components_at1000_df,
    )

print("Notebook 10-compatible runtime framework ready.")

Notebook 10-compatible runtime framework ready.


In [13]:
# ==== Held-Out Feature Interpretation Artifacts ====
CONDITION_NAME = 'Full'
MODEL_SOURCE_ARTIFACT_DIR = OUT_DIR
INTERPRETATION_CANDIDATE_SOURCE_FORMAT = "long_flat_parquet_by_method_pool"
INTERPRETATION_CANDIDATE_SCORE_POLICY = "preserved_upstream_score_without_retrieval_recomputation"
FEATURE_INTERPRETATION_FOLD_ASSIGNMENT_PATH = OUT_DIR / "feature_interpretation_fold_assignments.parquet"
FEATURE_GROUP_MAPPING_PATH = OUT_DIR / "feature_group_mapping.csv"
FEATURE_INTERPRETATION_MANIFEST_PATH = OUT_DIR / "feature_interpretation_manifest.json"
FEATURE_INTERPRETATION_CANDIDATE_FILENAME = "heldout_feature_interpretation_pool{pool_depth}.parquet"
FEATURE_PREPROCESSING_CONTRACT = {
    "numeric_dtype": "float32",
    "missing_value_fill": 0.0,
    "candidate_score_normalization": "within_query_minmax_by_pool_depth",
    "candidate_position_input": "raw_stage1_candidate_rank_only",
    "categorical_encoding": "precomputed_numeric_features_only",
    "feature_order_source": "EXPECTED_MODEL_FEATURE_COLUMNS",
}

In [14]:
# ==== Batch-2 Matched LightGBM Training Contract ====
LIGHTGBM_VARIANT = "b"
LIGHTGBM_TRAINING_PLAN_ROLE = "write"  # self-pool Full builds its own split plan (plan contract includes pool candidate_identity_hash)
LIGHTGBM_MATCHED_FITTING_POLICY = "target_present_non_cold_queries_only"
LIGHTGBM_TRAINING_PLAN_VERSION = "lightgbm_method_specific_split_plan_v2_primary_novel_item"
LIGHTGBM_MODEL_CONTRACT_VERSION = "lightgbm_matched_ablation_v2_primary_novel_item"
PRIMARY_REGISTRY_POLICY_VERSION = "benchmark_primary_novel_item_registry_v1"
SPECIFICATION_ROLE = "primary"
ENABLE_EXACT_ITEM_FAMILIARITY = False
LEGACY_RESULTS_REUSABLE = False
REQUIRES_DOWNSTREAM_REAGGREGATION = True
LIGHTGBM_RESUME_VALID_FOLDS = False
# Retain the variable name required by the held-out interpretation export.
SHARED_ALL_PRIOR_MODEL_CONTRACT_VERSION = LIGHTGBM_MODEL_CONTRACT_VERSION
LIGHTGBM_COMMON_TRAINING_DIR = (
    PROJECT_ROOT / "outputs" / "stage2_personalized_rerank" / "lightgbm_full" / "training_contract_self_pool"
)
# Keep the split artifact LightGBM-specific and category-local; do not reuse
# GAM or Transformer sampling membership.
# VERIFIED: PROJECT_ROOT resolves to the Facial Skincare category; shared LightGBM split artifacts stay category-local.

In [15]:
# ==== Environment / Paths / Config — Part 5 ====
# LightGBM config, input validation, and path summary

N_FOLDS = 5
VALID_FRAC_WITHIN_TRAIN = 0.15

LIGHTGBM_OBJECTIVE = "lambdarank"
LIGHTGBM_METRIC = "ndcg"
# Model selection is aligned across categories with the thesis primary metric.
# Reporting cutoffs remain controlled separately by EVAL_KS.
LIGHTGBM_EVAL_AT = [5]
LIGHTGBM_NUM_BOOST_ROUND = 300
LIGHTGBM_EARLY_STOPPING_ROUNDS = 50

VALIDATION_RATIO = 0.20
VALIDATION_GROUP_COLS = ("regime", "sampling_bracket")
VALIDATION_MIN_GROUP_SIZE = 5

required_paths = {
    "candidate_pool_path": CANDIDATE_POOL_PATH,
    "query_cache_parquet": QUERY_CACHE_PARQUET,
    "target_case_metadata_parquet": TARGET_CASE_METADATA_PARQUET,
    "item_schema_parquet": ITEM_SCHEMA_PARQUET,
    "item_schema_base_parquet": ITEM_SCHEMA_BASE_PARQUET,
    "item_docs_parquet": ITEM_DOCS_PARQUET,
    "items_facets_parquet": ITEMS_FACETS_PARQUET,
    "prior_history_parquet": PRIOR_HISTORY_PARQUET,
}
missing_required_paths = [name for name, path_obj in required_paths.items() if path_obj is None or not Path(path_obj).exists()]
if missing_required_paths:
    raise FileNotFoundError(
        "Missing required files:\n" +
        "\n".join(f"- {name}: {required_paths[name]}" for name in missing_required_paths)
    )

optional_paths = {
    "candidate_pool_manifest_path": CANDIDATE_POOL_MANIFEST_PATH,
    "candidate_pool_summary_path": CANDIDATE_POOL_SUMMARY_PATH,
    "query_cache_summary_path": QUERY_CACHE_SUMMARY_PATH,
    "query_cache_config_path": QUERY_CACHE_CONFIG_PATH,
    "graph_edges_parquet": GRAPH_EDGES_PARQUET,
    "product_nodes_parquet": PRODUCT_NODES_PARQUET,
    "entity_nodes_parquet": ENTITY_NODES_PARQUET,
}
optional_path_status_df = pd.DataFrame([
    {"artifact": name, "path": str(path_obj), "exists": bool(Path(path_obj).exists())}
    for name, path_obj in optional_paths.items()
])

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CANDIDATE_POOL_DIR:", CANDIDATE_POOL_DIR)
print("CANDIDATE_POOL_PATH:", CANDIDATE_POOL_PATH)
print("CANDIDATE_POOL_MANIFEST_PATH:", CANDIDATE_POOL_MANIFEST_PATH)
print("CANDIDATE_POOL_SUMMARY_PATH:", CANDIDATE_POOL_SUMMARY_PATH)
print("QUERY_CACHE_PARQUET:", QUERY_CACHE_PARQUET)
print("TARGET_CASE_METADATA_PARQUET:", TARGET_CASE_METADATA_PARQUET)
print("ITEM_SCHEMA_PARQUET:", ITEM_SCHEMA_PARQUET)
print("ITEM_SCHEMA_BASE_PARQUET:", ITEM_SCHEMA_BASE_PARQUET)
print("ITEMS_FACETS_PARQUET:", ITEMS_FACETS_PARQUET)
print("ITEM_DOCS_PARQUET:", ITEM_DOCS_PARQUET)
print("PRIOR_HISTORY_PARQUET:", PRIOR_HISTORY_PARQUET)
print("STAGE1_QUERY_METHOD:", STAGE1_QUERY_METHOD)
print("RETRIEVAL_METHOD:", RETRIEVAL_METHOD)
print("CANDIDATE_POOL_TYPE:", CANDIDATE_POOL_TYPE)
print("POOL_K:", POOL_K)
print("POOL_DEPTHS:", POOL_DEPTHS)
print("EVAL_KS:", EVAL_KS)
print("LIGHTGBM_EVAL_AT:", LIGHTGBM_EVAL_AT)
print("PRIMARY_STAGE2_METRICS:", PRIMARY_STAGE2_METRICS)
print("SECONDARY_STAGE2_METRICS:", SECONDARY_STAGE2_METRICS)
print("COMMON_STAGE2_COMPARISON_METRICS:", COMMON_STAGE2_COMPARISON_METRICS)
print("REGIME_PERSONALIZATION_WEIGHT_POLICY:", REGIME_PERSONALIZATION_WEIGHT_POLICY)
print("REGIME_PERSONALIZATION_WEIGHT:", REGIME_PERSONALIZATION_WEIGHT)
print("OUT_DIR:", OUT_DIR)
print("RERANKING_METHOD:", RERANKING_METHOD)
display(optional_path_status_df)

PROJECT_ROOT: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare
CANDIDATE_POOL_DIR: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage1_candidate_pools/by_method
CANDIDATE_POOL_PATH: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage1_candidate_pools/by_method/personalized_winner_top1000_face.parquet
CANDIDATE_POOL_MANIFEST_PATH: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage1_candidate_pools/stage1_candidate_pool_export_manifest_face.json
CANDIDATE_POOL_SUMMARY_PATH: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage1_candidate_pools/winner_candidate_pool_summary_face.csv
QUERY_CACHE_PARQUET: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/query_cache/face_queries.parquet
TARGET_CASE_METADATA_PARQUET: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/user_sampling/face_target_case_metadata.parquet
ITEM_SCHEMA

,artifact,path,exists
0,candidate_pool_manifest_path,/content/drive/MyDrive/thesis_recsys/categorie...,True
1,candidate_pool_summary_path,/content/drive/MyDrive/thesis_recsys/categorie...,True
2,query_cache_summary_path,/content/drive/MyDrive/thesis_recsys/categorie...,True
3,query_cache_config_path,/content/drive/MyDrive/thesis_recsys/categorie...,True
4,graph_edges_parquet,/content/drive/MyDrive/thesis_recsys/categorie...,True
5,product_nodes_parquet,/content/drive/MyDrive/thesis_recsys/categorie...,True
6,entity_nodes_parquet,/content/drive/MyDrive/thesis_recsys/categorie...,True


## 2. Utility Helpers

In [16]:
# ==== Utility Helpers ====
BASE_FACET_PREFIXES = (
    "product_type_",
    "form_",
    "concern_",
    "benefit_",
    "ingredient_",
    "free_",
    "flavor_",
    "usage_target_",
)

ITEM_SCHEMA_FAMILY_SOURCE_CANDIDATES = {
    "brand": ["brand_facet_text"],
    "product_type": ["product_type_norm", "product_type_norm_text", "family_sub_category", "sub_category_norm", "sub_category_norm_text", "sub_category"],
    "benefit": ["benefit_final", "benefit_final_text", "benefit_norm", "benefit_norm_text", "family_benefit_function", "product_benefits"],
    "ingredient": ["ingredient_final", "ingredient_final_text", "ingredient_norm", "ingredient_norm_text", "family_ingredient_composition", "active_ingredients", "special_ingredients"],
    "form": ["form_norm", "form_norm_text", "form", "form_text", "item_form_norm", "family_item_form", "item_form"],
    "free": ["free_claim_norm", "free_claim_norm_text", "free_claim", "free_claim_text", "family_claims_diet", "diet_type", "material_feature"],
    "flavor": ["flavor_norm", "flavor_norm_text", "flavor", "flavor_text", "family_flavor", "scent"],
    "usage_target": ["usage_target_norm", "usage_target_norm_text", "usage_target", "usage_target_text", "family_usage_target", "age_range"],
    "concern": ["concern_final", "concern_final_text", "concern_claim_norm", "concern_claim_norm_text", "usage_need_norm_text", "family_usage_need", "recommended_use", "specific_uses"],
}

MAIN_FAMILIES = ["brand", "product_type", "form", "concern", "benefit", "ingredient", "free", "flavor", "usage_target"]
SAFE_FACET_FALLBACK_FAMILIES = {"product_type", "form", "free", "flavor", "usage_target"}

def pick_first_existing(cols, candidates, default=None):
    for c in candidates:
        if c in cols:
            return c
    return default

def normalize_space(s):
    return re.sub(r"\s+", " ", "" if s is None else str(s)).strip()

def normalize_text(s):
    s = normalize_space(s).lower()
    s = re.sub(r"[^a-z0-9+\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def tokenize(text):
    return re.findall(r"[a-z0-9+]+", normalize_text(text))

def row_pick(row, names):
    for n in names:
        if n in row and pd.notna(row[n]) and normalize_space(row[n]) != "":
            return str(row[n])
    return ""

def to_ms_timestamp(series: pd.Series) -> pd.Series:
    ts = pd.to_numeric(series, errors="coerce").fillna(0).astype(np.int64)
    if len(ts) > 0 and int(ts.max()) < 10**12:
        ts = ts * 1000
    return ts

def minmax_norm(values):
    arr = np.asarray(values, dtype=np.float64)
    if arr.size == 0:
        return arr
    mn = float(np.nanmin(arr))
    mx = float(np.nanmax(arr))
    if not np.isfinite(mn) or not np.isfinite(mx) or mx - mn < 1e-12:
        return np.zeros_like(arr, dtype=np.float64)
    return ((arr - mn) / (mx - mn + 1e-12)).astype(np.float64)

def minmax_norm_series(series: pd.Series) -> pd.Series:
    arr = minmax_norm(pd.to_numeric(series, errors="coerce").fillna(0.0).to_numpy())
    return pd.Series(arr, index=series.index)

def clamp(x, lo=0.0, hi=1.0):
    return float(max(lo, min(hi, x)))

def ndcg_at_k(rel_list, k=10):
    rel = np.asarray(list(rel_list)[:k], dtype=float)
    if rel.size == 0:
        return 0.0
    gains = (2.0**rel - 1.0)
    discounts = np.log2(np.arange(2, rel.size + 2))
    dcg = float((gains / discounts).sum())
    ideal = np.sort(rel)[::-1]
    idcg = float(((2.0**ideal - 1.0) / discounts).sum())
    return 0.0 if idcg == 0.0 else dcg / idcg

def mrr_at_k(pred_item_ids, true_item_id, k=10):
    ranked = [str(x) for x in list(pred_item_ids)[:k]]
    true_item_id = str(true_item_id)
    for idx, item_id in enumerate(ranked, start=1):
        if item_id == true_item_id:
            return float(1.0 / idx)
    return 0.0

def compute_query_metrics(pred_item_ids, true_item_id, ks=EVAL_KS):
    out = {}
    ranked = [str(x) for x in list(pred_item_ids)]
    true_item_id = str(true_item_id)
    for k in ks:
        topk = ranked[:k]
        rel = [1 if x == true_item_id else 0 for x in topk]
        out[f"HitRate@{k}"] = float(true_item_id in topk)
        out[f"MRR@{k}"] = mrr_at_k(ranked, true_item_id, k=k)
        out[f"NDCG@{k}"] = ndcg_at_k(rel, k=k)
    if 100 not in set(int(k) for k in ks):
        rel100 = [1 if x == true_item_id else 0 for x in ranked[:100]]
        out["NDCG@100"] = ndcg_at_k(rel100, k=100)
    return out

In [17]:
# ==== Utility Helpers — Part 2 ====
def parse_literal_structure(x):
    if x is None:
        return None
    if isinstance(x, float) and pd.isna(x):
        return None
    if isinstance(x, (list, tuple, set, dict)):
        return x
    if not isinstance(x, str):
        return None
    text = x.strip()
    if not text:
        return None
    try:
        return json.loads(text)
    except Exception:
        try:
            return ast.literal_eval(text)
        except Exception:
            return None

def flatten_text_parts(value):
    if value is None:
        return []
    if isinstance(value, float) and pd.isna(value):
        return []
    if isinstance(value, dict):
        out = []
        for k, v in value.items():
            ktxt = normalize_space(k)
            vparts = flatten_text_parts(v)
            if ktxt:
                out.append(ktxt)
            out.extend(vparts)
        return out
    if isinstance(value, (list, tuple, set)):
        out = []
        for v in value:
            out.extend(flatten_text_parts(v))
        return out
    if isinstance(value, np.ndarray):
        return flatten_text_parts(value.tolist())
    txt = normalize_space(value)
    return [txt] if txt else []

def ensure_list(value):
    parsed = parse_literal_structure(value)
    source = parsed if parsed is not None else value
    if source is None:
        return []
    if isinstance(source, float) and pd.isna(source):
        return []
    if isinstance(source, (list, tuple, set)):
        return [x for x in source]
    if isinstance(source, dict):
        out = []
        for k, v in source.items():
            if normalize_space(k):
                out.append(normalize_space(k))
            out.extend(ensure_list(v))
        return out
    text = normalize_space(source)
    if not text:
        return []
    if "|" in text:
        return [normalize_space(x) for x in text.split("|") if normalize_space(x)]
    if "," in text and len(text) < 250:
        return [normalize_space(x) for x in text.split(",") if normalize_space(x)]
    return [text]

def dedupe_keep_order(values):
    seen = set()
    out = []
    for v in values:
        key = normalize_text(v)
        if not key or key in seen:
            continue
        seen.add(key)
        out.append(normalize_space(v))
    return out

def ensure_normalized_label_list(value):
    values = dedupe_keep_order(ensure_list(value))
    out = []
    for v in values:
        nv = normalize_text(v)
        if nv:
            out.append(nv)
    return dedupe_keep_order(out)

def to_normalized_value_set(value) -> set:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return set()
    if isinstance(value, (set, list, tuple)):
        raw_values = list(value)
    else:
        text = normalize_space(value)
        if not text:
            return set()
        raw_values = re.split(r"[|,;/]+", text)
    out = set()
    for raw in raw_values:
        norm = normalize_text(raw)
        if norm:
            out.add(norm)
    return out

def get_item_family_value_set(item_id: str, family: str, item_brand_map: dict, item_family_labels_map: dict) -> set:
    item_id = str(item_id)
    family = str(family)
    if family == "brand":
        return to_normalized_value_set(item_brand_map.get(item_id, ""))
    return to_normalized_value_set(item_family_labels_map.get(item_id, {}).get(family, []))

In [18]:
# ==== Utility Helpers — Part 3 ====
def jaccard_set_similarity(left: set, right: set) -> float:
    if not left or not right:
        return 0.0
    union = left | right
    if not union:
        return 0.0
    return float(len(left & right) / len(union))

def binary_family_match(left: set, right: set) -> float:
    if not left or not right:
        return 0.0
    return float(bool(left & right))

def normalize_family_weight_map(families, family_weights=None):
    families = [str(f) for f in families]
    if not families:
        return {}
    raw = {}
    if family_weights is not None:
        for family in families:
            raw[family] = max(float(family_weights.get(family, 0.0)), 0.0)
    else:
        raw = {family: 1.0 for family in families}
    total = float(sum(raw.values()))
    if total <= 0.0:
        uniform = 1.0 / len(families)
        return {family: float(uniform) for family in families}
    return {family: float(raw[family] / total) for family in families}

PREFERENCE_DIAGNOSTIC_ACTIVE_FAMILIES = [
    "brand",
    "concern",
    "ingredient",
    "skin_type",
    "form",
    "benefit",
]

OPTIONAL_PREFERENCE_DIAGNOSTIC_FAMILIES = [
    "free",
    "scent",
    "product_type",
]

PREFERENCE_DIAGNOSTIC_FAMILY_WEIGHTS = {
    "brand": 1.0,
    "concern": 1.0,
    "ingredient": 1.0,
    "skin_type": 1.0,
    "form": 1.0,
    "benefit": 1.0,
    "free": 1.0,
    "scent": 1.0,
    "product_type": 1.0,
}

PREFERENCE_FAMILY_TO_METRIC_COL = {
    "brand": "brand_match_at_5",
    "concern": "concern_match_at_5",
    "ingredient": "ingredient_match_at_5",
    "skin_type": "skin_type_match_at_5",
    "form": "form_match_at_5",
    "benefit": "benefit_match_at_5",
    "free": "claim_match_at_5",
    "scent": "scent_match_at_5",
    "product_type": "category_or_product_type_match_at_5",
}

PREFERENCE_DIAGNOSTIC_METRICS = [
    "weighted_facet_overlap_at_5",
    "brand_match_at_5",
    "concern_match_at_5",
    "ingredient_match_at_5",
    "skin_type_match_at_5",
    "form_match_at_5",
    "benefit_match_at_5",
    "claim_match_at_5",
    "scent_match_at_5",
    "category_or_product_type_match_at_5",
]

SUPPLEMENTARY_METRIC_COLS = list(PREFERENCE_DIAGNOSTIC_METRICS)
UPLIFT_DIAGNOSTIC_METRICS = list(PREFERENCE_DIAGNOSTIC_METRICS)

for _name in [
    "PREFERENCE_DIAGNOSTIC_ACTIVE_FAMILIES",
    "PREFERENCE_DIAGNOSTIC_FAMILY_WEIGHTS",
    "PREFERENCE_FAMILY_TO_METRIC_COL",
    "PREFERENCE_DIAGNOSTIC_METRICS",
]:
    if _name not in globals():
        raise RuntimeError(f"Missing preference diagnostic config: {_name}")

if "formulation_match_at_5" in PREFERENCE_DIAGNOSTIC_METRICS:
    raise RuntimeError("formulation_match_at_5 must not be required for Facial preference diagnostics.")

EXPORT_DIAGNOSTIC_FAMILY_SPECS = [
    {
        "family": "brand",
        "metric_col": "brand_match_at_5",
        "candidate_col": "candidate_brand",
        "target_col": "target_brand",
        "match_col": "candidate_brand_match",
        "affinity_col": "user_brand_affinity",
        "candidate_source_patterns": ["candidate_brand", "item_brand", "brand"],
        "target_source_patterns": ["target_brand", "gt_brand", "true_brand", "heldout_brand"],
        "match_source_patterns": ["candidate_brand_match", "brand_match"],
        "affinity_source_patterns": ["candidate_brand_prior_share"],
    },
    {
        "family": "concern",
        "metric_col": "concern_match_at_5",
        "candidate_col": "candidate_concern",
        "target_col": "target_concern",
        "match_col": "candidate_concern_match",
        "affinity_col": "user_concern_affinity",
        "candidate_source_patterns": ["candidate_concern", "candidate_concerns", "item_concern", "item_concerns", "concern", "concerns", "skin_concern", "benefit_concern", "concern_final", "family_usage_need", "usage_need", "specific_uses", "recommended_use"],
        "target_source_patterns": ["target_concern", "target_concerns", "gt_concern", "gt_concerns", "true_concern", "true_concerns", "heldout_concern", "heldout_concerns"],
        "match_source_patterns": ["candidate_concern_match", "concern_match", "skin_concern_match", "benefit_concern_match"],
        "affinity_source_patterns": ["user_concern_affinity", "uaff__concern", "profile_concern_affinity"],
    },
    {
        "family": "ingredient",
        "metric_col": "ingredient_match_at_5",
        "candidate_col": "candidate_ingredient",
        "target_col": "target_ingredient",
        "match_col": "candidate_ingredient_match",
        "affinity_col": "user_ingredient_affinity",
        "candidate_source_patterns": ["candidate_ingredient", "candidate_ingredients", "item_ingredient", "item_ingredients", "ingredient", "ingredients", "ingredient_final", "ingredient_norm", "special_ingredients", "active_ingredients"],
        "target_source_patterns": ["target_ingredient", "target_ingredients", "gt_ingredient", "gt_ingredients", "true_ingredient", "true_ingredients", "heldout_ingredient", "heldout_ingredients"],
        "match_source_patterns": ["candidate_ingredient_match", "ingredient_match", "ingredients_match"],
        "affinity_source_patterns": ["user_ingredient_affinity", "uaff__ingredient", "profile_ingredient_affinity"],
    },
    {
        "family": "skin_type",
        "metric_col": "skin_type_match_at_5",
        "candidate_col": "candidate_skin_type",
        "target_col": "target_skin_type",
        "match_col": "candidate_skin_type_match",
        "affinity_col": "user_skin_type_affinity",
        "candidate_source_patterns": ["candidate_skin_type", "item_skin_type", "skin_type", "skin_type_norm", "family_skin_type"],
        "target_source_patterns": ["target_skin_type", "gt_skin_type", "true_skin_type", "heldout_skin_type"],
        "match_source_patterns": ["candidate_skin_type_match", "skin_type_match"],
        "affinity_source_patterns": ["user_skin_type_affinity", "uaff__skin_type", "profile_skin_type_affinity"],
    },
    {
        "family": "form",
        "metric_col": "form_match_at_5",
        "candidate_col": "candidate_form",
        "target_col": "target_form",
        "match_col": "candidate_form_match",
        "affinity_col": "user_form_affinity",
        "candidate_source_patterns": ["candidate_form", "candidate_item_form", "item_form", "form", "form_norm", "family_item_form", "facet_form_text", "product_form"],
        "target_source_patterns": ["target_form", "target_item_form", "gt_form", "gt_item_form", "true_form", "true_item_form", "heldout_form", "heldout_item_form"],
        "match_source_patterns": ["candidate_form_match", "form_match", "item_form_match", "texture_or_form_match"],
        "affinity_source_patterns": ["user_form_affinity", "user_item_form_affinity", "uaff__form", "uaff__item_form"],
    },
    {
        "family": "benefit",
        "metric_col": "benefit_match_at_5",
        "candidate_col": "candidate_benefit",
        "target_col": "target_benefit",
        "match_col": "candidate_benefit_match",
        "affinity_col": "user_benefit_affinity",
        "candidate_source_patterns": ["candidate_benefit", "candidate_benefits", "item_benefit", "item_benefits", "benefit", "benefits", "benefit_final", "benefit_norm", "product_benefits", "family_benefit_function", "facet_benefit_text"],
        "target_source_patterns": ["target_benefit", "target_benefits", "gt_benefit", "gt_benefits", "true_benefit", "true_benefits", "heldout_benefit", "heldout_benefits"],
        "match_source_patterns": ["candidate_benefit_match", "benefit_match", "benefits_match"],
        "affinity_source_patterns": ["user_benefit_affinity", "uaff__benefit", "profile_benefit_affinity"],
    },
    {
        "family": "free",
        "metric_col": "claim_match_at_5",
        "candidate_col": "candidate_claim",
        "target_col": "target_claim",
        "match_col": "candidate_claim_match",
        "affinity_col": "user_claim_affinity",
        "candidate_source_patterns": ["candidate_claim", "candidate_claims", "candidate_free_claim", "item_claim", "item_claims", "item_free_claim", "free_claim", "free_claim_norm", "material_type_free", "material_feature", "claim", "claims"],
        "target_source_patterns": ["target_claim", "target_claims", "target_free_claim", "gt_claim", "gt_claims", "gt_free_claim", "true_claim", "true_claims", "heldout_claim", "heldout_free_claim"],
        "match_source_patterns": ["candidate_claim_match", "candidate_free_claim_match", "claim_match", "free_claim_match"],
        "affinity_source_patterns": ["user_claim_affinity", "user_free_claim_affinity", "uaff__claim", "uaff__free", "uaff__free_claim"],
    },
    {
        "family": "scent",
        "metric_col": "scent_match_at_5",
        "candidate_col": "candidate_scent",
        "target_col": "target_scent",
        "match_col": "candidate_scent_match",
        "affinity_col": "user_scent_affinity",
        "candidate_source_patterns": ["candidate_scent", "item_scent", "scent", "fragrance", "perfume", "family_flavor"],
        "target_source_patterns": ["target_scent", "gt_scent", "true_scent", "heldout_scent"],
        "match_source_patterns": ["candidate_scent_match", "scent_match", "fragrance_match"],
        "affinity_source_patterns": ["user_scent_affinity", "uaff__scent", "uaff__flavor"],
    },
    {
        "family": "product_type",
        "metric_col": "category_or_product_type_match_at_5",
        "candidate_col": "candidate_category_or_product_type",
        "target_col": "target_category_or_product_type",
        "match_col": "candidate_category_or_product_type_match",
        "affinity_col": "user_category_or_product_type_affinity",
        "candidate_source_patterns": ["candidate_category_or_product_type", "candidate_product_type", "candidate_category", "candidate_sub_category", "item_product_type", "item_category", "item_sub_category", "family_sub_category", "facet_category_text", "product_type", "category", "sub_category"],
        "target_source_patterns": ["target_category_or_product_type", "target_product_type", "target_category", "target_sub_category", "gt_product_type", "gt_category", "gt_sub_category", "true_product_type", "true_category", "true_sub_category", "heldout_product_type", "heldout_category", "heldout_sub_category"],
        "match_source_patterns": ["candidate_category_or_product_type_match", "candidate_product_type_match", "candidate_category_match", "product_type_match", "category_match", "category_or_product_type_match"],
        "affinity_source_patterns": ["user_category_or_product_type_affinity", "user_product_type_affinity", "user_category_affinity", "user_sub_category_affinity", "uaff__product_type", "uaff__category", "uaff__sub_category"],
    },
]
PREFERENCE_EXPORT_DIAGNOSTIC_COLUMNS = list(dict.fromkeys(
    ["target_parent_asin", "candidate_parent_asin", "parent_asin"]
    + [spec["candidate_col"] for spec in EXPORT_DIAGNOSTIC_FAMILY_SPECS]
    + [spec["target_col"] for spec in EXPORT_DIAGNOSTIC_FAMILY_SPECS]
    + [spec["match_col"] for spec in EXPORT_DIAGNOSTIC_FAMILY_SPECS]
    + [spec["affinity_col"] for spec in EXPORT_DIAGNOSTIC_FAMILY_SPECS]
))
_PREFERENCE_DIAGNOSTIC_ITEM_MAP_CACHE = None


def _preference_join_values(values):
    if values is None:
        return ""
    return " | ".join(sorted(str(v) for v in values if str(v)))


def _source_value_set_from_row(row, source_patterns):
    values = set()
    for col in source_patterns:
        if col in row.index:
            values.update(to_normalized_value_set(row[col]))
    return values


def build_preference_diagnostic_item_family_map():
    global _PREFERENCE_DIAGNOSTIC_ITEM_MAP_CACHE
    if _PREFERENCE_DIAGNOSTIC_ITEM_MAP_CACHE is not None:
        return _PREFERENCE_DIAGNOSTIC_ITEM_MAP_CACHE

    item_ids = set(str(k) for k in globals().get("item_brand_map", {}).keys()) | set(str(k) for k in globals().get("item_family_labels_map", {}).keys())
    meta_lookup = {}
    if "meta_df" in globals() and isinstance(meta_df, pd.DataFrame) and "item_id" in meta_df.columns:
        meta_keyed = meta_df.copy()
        meta_keyed["_diagnostic_item_id"] = meta_keyed["item_id"].astype(str)
        meta_lookup = meta_keyed.drop_duplicates("_diagnostic_item_id").set_index("_diagnostic_item_id").to_dict(orient="index")
        item_ids |= set(str(k) for k in meta_lookup.keys())

    out = {metric: np.nan for metric in PREFERENCE_DIAGNOSTIC_METRICS}
    for item_id in item_ids:
        family_map = {}
        base_family_map = globals().get("item_family_labels_map", {}).get(str(item_id), {})
        meta_row = pd.Series(meta_lookup.get(str(item_id), {}), dtype="object")
        for spec in EXPORT_DIAGNOSTIC_FAMILY_SPECS:
            family = spec["family"]
            values = set()
            if family == "brand":
                values.update(to_normalized_value_set(globals().get("item_brand_map", {}).get(str(item_id), "")))
            values.update(to_normalized_value_set(base_family_map.get(family, [])))
            if not values and not meta_row.empty:
                values.update(_source_value_set_from_row(meta_row, spec["candidate_source_patterns"]))
            if values:
                family_map[family] = values
        out[str(item_id)] = family_map

    _PREFERENCE_DIAGNOSTIC_ITEM_MAP_CACHE = out
    return out


def preference_diagnostic_available_families():
    diag_map = build_preference_diagnostic_item_family_map()
    available = []
    for spec in EXPORT_DIAGNOSTIC_FAMILY_SPECS:
        family = spec["family"]
        if any(bool(family_map.get(family)) for family_map in diag_map.values()):
            available.append(family)
    return available


def compute_topk_similarity_metrics(
    ranked_item_ids,
    gt_item_id,
    families,
    item_brand_map,
    item_family_labels_map,
    family_weights=None,
    k=PREFERENCE_ALIGNMENT_K,
):
    ranked = [str(x) for x in list(ranked_item_ids)[:k]]
    gt_item_id = str(gt_item_id)
    diag_map = build_preference_diagnostic_item_family_map()
    requested = [str(f) for f in families]
    active_families = [f for f in requested if diag_map.get(gt_item_id, {}).get(f)]
    weight_families = [f for f in PREFERENCE_DIAGNOSTIC_ACTIVE_FAMILIES if f in active_families]
    weight_map = normalize_family_weight_map(weight_families, family_weights=family_weights)

    out = {metric: np.nan for metric in PREFERENCE_DIAGNOSTIC_METRICS}
    if ranked and weight_families:
        weighted_scores = []
        for rec_item_id in ranked:
            weighted_score = 0.0
            for family in weight_families:
                weighted_score += weight_map[family] * jaccard_set_similarity(
                    diag_map.get(gt_item_id, {}).get(family, set()),
                    diag_map.get(rec_item_id, {}).get(family, set()),
                )
            weighted_scores.append(weighted_score)
        if weighted_scores:
            out["weighted_facet_overlap_at_5"] = float(np.mean(weighted_scores))

    for family in active_families:
        metric_col = PREFERENCE_FAMILY_TO_METRIC_COL.get(family)
        if not metric_col:
            continue
        scores = [
            binary_family_match(
                diag_map.get(gt_item_id, {}).get(family, set()),
                diag_map.get(rec_item_id, {}).get(family, set()),
            )
            for rec_item_id in ranked
            if diag_map.get(rec_item_id, {}).get(family)
        ]
        if scores:
            out[metric_col] = float(np.mean(scores))
    return out


def _first_existing_column(df: pd.DataFrame, candidates):
    return next((col for col in candidates if col in df.columns), None)


def _series_value_sets(series: pd.Series):
    return series.map(to_normalized_value_set)


def append_preference_diagnostic_columns(work: pd.DataFrame) -> pd.DataFrame:
    if "candidate_item_id" not in work.columns or "gt_item_id" not in work.columns:
        print("WARNING: Preference diagnostic export skipped because candidate_item_id or gt_item_id is missing.")
        return work
    diag_map = build_preference_diagnostic_item_family_map()
    candidate_ids = work["candidate_item_id"].astype(str)
    target_ids = work["gt_item_id"].astype(str)
    if "candidate_parent_asin" not in work.columns:
        work["candidate_parent_asin"] = candidate_ids
    if "target_parent_asin" not in work.columns:
        work["target_parent_asin"] = target_ids
    if "parent_asin" not in work.columns:
        work["parent_asin"] = candidate_ids

    for spec in EXPORT_DIAGNOSTIC_FAMILY_SPECS:
        family = spec["family"]
        direct_source_col = _first_existing_column(work, spec.get("match_source_patterns", []))
        affinity_source_col = _first_existing_column(work, spec.get("affinity_source_patterns", []))
        candidate_source_col = _first_existing_column(work, spec.get("candidate_source_patterns", []))
        target_source_col = _first_existing_column(work, spec.get("target_source_patterns", []))

        if direct_source_col:
            work[spec["match_col"]] = pd.to_numeric(work[direct_source_col], errors="coerce")
        if affinity_source_col:
            work[spec["affinity_col"]] = pd.to_numeric(work[affinity_source_col], errors="coerce")

        if candidate_source_col:
            candidate_values = _series_value_sets(work[candidate_source_col])
        else:
            candidate_values = candidate_ids.map(lambda item_id: diag_map.get(item_id, {}).get(family, set()))
        if target_source_col:
            target_values = _series_value_sets(work[target_source_col])
        else:
            target_values = target_ids.map(lambda item_id: diag_map.get(item_id, {}).get(family, set()))

        has_candidate = candidate_values.map(bool)
        has_target = target_values.map(bool)
        if has_candidate.any():
            work[spec["candidate_col"]] = candidate_values.map(_preference_join_values)
        if has_target.any():
            work[spec["target_col"]] = target_values.map(_preference_join_values)
        if spec["match_col"] not in work.columns and has_candidate.any() and has_target.any():
            work[spec["match_col"]] = [
                float(bool(c_vals & t_vals)) if c_vals and t_vals else np.nan
                for c_vals, t_vals in zip(candidate_values, target_values)
            ]
    return work


def compute_topk_diagnostics_from_ranked_frame(work: pd.DataFrame, k=PREFERENCE_ALIGNMENT_K):
    rank_col = next((c for c in ["new_rank", "rerank_rank", "reranked_rank", "final_rank", "rank"] if c in work.columns), None)
    top = work.sort_values(rank_col).head(k) if rank_col else work.head(k)
    out = {metric: np.nan for metric in PREFERENCE_DIAGNOSTIC_METRICS}
    weighted_parts = []
    weighted_weights = []
    for spec in EXPORT_DIAGNOSTIC_FAMILY_SPECS:
        match_col = spec["match_col"]
        metric_col = spec["metric_col"]
        family = spec["family"]
        if match_col not in top.columns:
            continue
        values = pd.to_numeric(top[match_col], errors="coerce")
        if values.notna().any():
            out[metric_col] = float(values.mean())
            if family in PREFERENCE_DIAGNOSTIC_ACTIVE_FAMILIES:
                weighted_parts.append(values)
                weighted_weights.append(float(PREFERENCE_DIAGNOSTIC_FAMILY_WEIGHTS.get(family, 1.0)))
    if weighted_parts and sum(weighted_weights) > 0:
        weighted_matrix = pd.concat(weighted_parts, axis=1)
        valid_weights = np.asarray(weighted_weights, dtype=float)
        valid_matrix = weighted_matrix.notna().astype(float)
        row_weight_sum = valid_matrix.mul(valid_weights, axis=1).sum(axis=1)
        weighted_values = weighted_matrix.fillna(0.0).mul(valid_weights, axis=1).sum(axis=1) / row_weight_sum.replace(0, np.nan)
        if weighted_values.notna().any():
            out["weighted_facet_overlap_at_5"] = float(weighted_values.mean())
    return out


def build_reranked_candidates_export_qc(export_df: pd.DataFrame, output_path):
    added_cols = [c for c in PREFERENCE_EXPORT_DIAGNOSTIC_COLUMNS if c in export_df.columns]
    requested_cols = [
        col
        for spec in EXPORT_DIAGNOSTIC_FAMILY_SPECS
        for col in [spec["candidate_col"], spec["target_col"], spec["match_col"], spec["affinity_col"]]
    ]
    missing_requested = [c for c in requested_cols if c not in export_df.columns]
    core_cols = ["query_item_structured_match", "user_item_affinity", "user_item_seen_strength", "candidate_rank", "candidate_score", "candidate_score_norm_pool"]
    missing_core = [c for c in core_cols if c not in export_df.columns]
    active_families = [spec["family"] for spec in EXPORT_DIAGNOSTIC_FAMILY_SPECS if spec["match_col"] in export_df.columns]
    row = {
        "notebook_name": NOTEBOOK_NAME,
        "output_path": str(output_path),
        "exported_row_count": int(len(export_df)),
        "exported_column_count": int(len(export_df.columns)),
        "added_diagnostic_columns": json.dumps(added_cols),
        "missing_requested_diagnostic_columns": json.dumps(missing_requested),
        "missing_core_mechanism_columns": json.dumps(missing_core),
        "active_preference_diagnostic_families": json.dumps(active_families),
        "active_preference_family_weights": json.dumps(PREFERENCE_DIAGNOSTIC_FAMILY_WEIGHTS),
        "notes": "Export-only preference diagnostics; model features, scores, and ranking logic unchanged.",
    }
    for spec in EXPORT_DIAGNOSTIC_FAMILY_SPECS:
        family = spec["family"]
        key_family = "category_or_product_type" if family == "product_type" else "claim" if family == "free" else family
        source_cols = [c for c in [spec["candidate_col"], spec["target_col"], spec["match_col"], spec["affinity_col"]] if c in export_df.columns]
        row[f"source_columns_used_for_{key_family}"] = json.dumps(source_cols)
    return pd.DataFrame([row])


def detect_facet_cols(df: pd.DataFrame):
    return [c for c in df.columns if any(c.startswith(prefix) for prefix in BASE_FACET_PREFIXES)]

def infer_family_from_facet_col(col: str) -> str:
    return col.split("_", 1)[0] if "_" in col else str(col)

def facet_label(col: str) -> str:
    if "_" not in col:
        return normalize_text(col)
    return normalize_text(col.split("_", 1)[1])

def normalized_entropy_from_counter(counter_obj: Counter, vocab_size: int):
    total = float(sum(counter_obj.values()))
    if total <= 0.0:
        return {"count": 0, "entropy": np.nan, "entropy_norm": np.nan}
    probs = np.asarray([v / total for v in counter_obj.values()], dtype=np.float64)
    entropy = float(-np.sum(probs * np.log(probs + 1e-12)))
    denom = math.log(max(int(vocab_size), 2))
    entropy_norm = float(entropy / denom) if denom > 0 else np.nan
    entropy_norm = float(max(0.0, min(1.0, entropy_norm))) if pd.notna(entropy_norm) else np.nan
    return {"count": int(total), "entropy": float(entropy), "entropy_norm": entropy_norm}

In [19]:
# ==== Utility Helpers — Part 4 ====
def aggregate_entropy_norm(entropy_norm_map, default=np.nan):
    vals = [float(v) for v in entropy_norm_map.values() if pd.notna(v)]
    return default if not vals else float(np.mean(vals))

def make_jsonable_dict(d):
    if isinstance(d, dict):
        return {str(k): make_jsonable_dict(v) for k, v in d.items()}
    if isinstance(d, list):
        return [make_jsonable_dict(v) for v in d]
    if isinstance(d, tuple):
        return [make_jsonable_dict(v) for v in d]
    if isinstance(d, (np.floating,)):
        return None if pd.isna(d) else float(d)
    if isinstance(d, (np.integer,)):
        return int(d)
    if pd.isna(d):
        return None
    return d

def load_optional_json(paths):
    for path in paths:
        if path is None:
            continue
        path = Path(path)
        if path.exists():
            return path, json.loads(path.read_text(encoding="utf-8-sig"))
    return None, {}

def nonnull_rate(series: pd.Series) -> float:
    if len(series) == 0:
        return np.nan
    return float(series.notna().mean())

def coalesce_series(df: pd.DataFrame, candidate_cols, default_value=""):
    for col in candidate_cols:
        if col in df.columns:
            return df[col].fillna(default_value)
    return pd.Series([default_value] * len(df), index=df.index)

In [20]:
# ==== Utility Helpers — Part 5 ====
def standardize_stage1_candidates(df: pd.DataFrame) -> tuple[pd.DataFrame, dict, list]:
    required_cols = [
        "case_id",
        "query_id",
        "user_id",
        "regime",
        "sampling_bracket",
        "target_selection_mode",
        "query_method",
        "gt_item_id",
        "query_text",
        "retrieval_method",
        "candidate_parent_asin",
        "candidate_rank",
        "candidate_score",
        "candidate_brand_facet_text",
        "is_target",
    ]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise RuntimeError(f"Stage 1 candidate parquet is missing required columns: {missing_cols}")

    retrieval_method_label_source = (
        df["retrieval_method_label"]
        if "retrieval_method_label" in df.columns
        else df["retrieval_method"]
    )


    out = pd.DataFrame({
        "benchmark_query_set": "",
        "benchmark_query_set_raw": "",
        "split_strategy": "",
        "category_key": CATEGORY_ID,
        "case_id": df["case_id"].astype(str),
        "query_id": df["query_id"].astype(str),
        "user_id": df["user_id"].fillna("").astype(str),
        "regime": df["regime"].fillna("").astype(str).map(normalize_regime_label),
        "sampling_bracket": df["sampling_bracket"].fillna("").astype(str),
        "target_selection_mode": df["target_selection_mode"].fillna("").astype(str),
        "query_method": df["query_method"].fillna(STAGE1_QUERY_METHOD).astype(str),
        "gt_item_id": df["gt_item_id"].astype(str),
        "query_text": df["query_text"].fillna("").astype(str),
        "candidate_item_id": df["candidate_parent_asin"].astype(str),
        "candidate_brand_facet_text": df["candidate_brand_facet_text"].fillna("").astype(str),
        "candidate_rank": pd.to_numeric(df["candidate_rank"], errors="coerce").astype("Int64"),
        "candidate_score": pd.to_numeric(df["candidate_score"], errors="coerce").fillna(0.0),
        "candidate_pool_type": CANDIDATE_POOL_TYPE,
        "retrieval_method": df["retrieval_method"].fillna("").astype(str).map(canonical_nonpersonalized_retrieval_method),
        "retrieval_method_label":
        retrieval_method_label_source.fillna("").astype(str).map(canonical_nonpersonalized_retrieval_method),
        "is_gt": pd.to_numeric(df["is_target"], errors="coerce").fillna(0).astype(np.int8),
    })

    out = out.sort_values(["query_id", "candidate_rank", "candidate_item_id"]).reset_index(drop=True)

    candidate_schema_map = {
        "case_id": "case_id",
        "query_id": "query_id",
        "user_id": "user_id",
        "regime": "regime",
        "sampling_bracket": "sampling_bracket",
        "target_selection_mode": "target_selection_mode",
        "query_method": "query_method",
        "gt_item_id": "gt_item_id",
        "query_text": "query_text",
        "retrieval_method": "retrieval_method",
        "retrieval_method_label": "retrieval_method_label",
        "candidate_item_id": "candidate_parent_asin",
        "candidate_brand_facet_text": "candidate_brand_facet_text",
        "candidate_rank": "candidate_rank",
        "candidate_score": "candidate_score",
        "is_gt": "is_target",
        "candidate_pool_type": None,
        "benchmark_query_set": None,
        "benchmark_query_set_raw": None,
        "split_strategy": None,
        "category_key": None,
    }

    return out, candidate_schema_map, []


In [21]:
# ==== Utility Helpers — Part 6 ====
def build_query_cache_meta(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    required_cols = [
        "case_id", "user_id", "target_parent_asin", "query", "regime",
        "target_selection_mode",
    ]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise RuntimeError(f"Query cache parquet is missing required columns: {missing_cols}")

    query_text = df["query"].fillna("").astype(str)
    if query_text.str.strip().eq("").any():
        raise RuntimeError("The active query column contains empty values.")
    if "query_C" in df.columns and not df["query_C"].fillna("").astype(str).eq(query_text).all():
        raise RuntimeError("query_C must equal query when the compatibility alias is present.")

    prior_n = (
        pd.to_numeric(df["prior_history_n"], errors="coerce").fillna(0).astype(int)
        if "prior_history_n" in df.columns
        else pd.Series(np.zeros(len(df), dtype=int), index=df.index)
    )
    sampling_bracket = (
        df["sampling_bracket"].fillna("").astype(str)
        if "sampling_bracket" in df.columns
        else df["regime"].fillna("").astype(str)
    )
    target_timestamp = (
        to_ms_timestamp(df["target_timestamp_ms"]).astype(np.int64)
        if "target_timestamp_ms" in df.columns
        else pd.Series(np.zeros(len(df), dtype=np.int64), index=df.index)
    )

    out = pd.DataFrame({
        "case_id": df["case_id"].astype(str),
        "query_id": df["case_id"].astype(str) + f"__{STAGE1_QUERY_METHOD}",
        "user_id": df["user_id"].fillna("").astype(str),
        "gt_item_id": df["target_parent_asin"].astype(str),
        "timestamp_ms": target_timestamp,
        "query_text_cache": query_text,
        "regime": df["regime"].fillna("").astype(str).map(normalize_regime_label),
        "sampling_bracket": sampling_bracket,
        "prior_review_n": prior_n,
        "prior_item_n": prior_n,
        "user_total_reviews": prior_n,
        "query_token_len": query_text.map(lambda x: len(tokenize(x))).astype(int),
        "removed_token_count": 0,
        "query_source": "query",
        "facet_hit_count": 0,
        "review_signal_count": 0,
    }).drop_duplicates("query_id").reset_index(drop=True)

    query_cache_schema_map = {
        "case_id": "case_id",
        "query_id": "case_id + '__C' compatibility key",
        "user_id": "user_id",
        "gt_item_id": "target_parent_asin",
        "timestamp_ms": "target_timestamp_ms if present; target_case_metadata.target_timestamp_ms after merge",
        "query_text": "query",
        "regime": "regime",
        "sampling_bracket": "sampling_bracket if present; otherwise regime",
        "prior_review_n": "prior_history_n if present; otherwise derived later from strict prior history",
        "prior_item_n": "prior_history_n if present; otherwise derived later from strict prior history",
        "user_total_reviews": "prior_history_n if present; otherwise derived later from strict prior history",
        "query_token_len": "computed_from_query",
        "removed_token_count": None,
        "query_source": "query",
        "facet_hit_count": None,
        "review_signal_count": None,
    }

    return out, query_cache_schema_map

def merge_target_case_metadata(query_meta_df: pd.DataFrame, target_metadata_path: Path) -> tuple[pd.DataFrame, dict]:
    if target_metadata_path is None or not Path(target_metadata_path).exists():
        raise FileNotFoundError(f"Missing target case metadata parquet: {target_metadata_path}")

    target_case_metadata = pd.read_parquet(target_metadata_path)
    required_target_meta_cols = [
        "case_id",
        "user_id",
        "target_parent_asin",
        "target_timestamp_ms",
    ]
    missing_target_meta_cols = [
        col for col in required_target_meta_cols
        if col not in target_case_metadata.columns
    ]
    if missing_target_meta_cols:
        raise RuntimeError(
            f"Target case metadata missing required columns: {missing_target_meta_cols}"
        )

    target_meta = target_case_metadata[required_target_meta_cols].copy()
    target_meta["case_id"] = target_meta["case_id"].astype(str)
    target_meta["user_id"] = target_meta["user_id"].fillna("").astype(str)
    target_meta["target_parent_asin"] = target_meta["target_parent_asin"].fillna("").astype(str)

    duplicated_case_n = int(target_meta["case_id"].duplicated().sum())
    if duplicated_case_n:
        raise RuntimeError(
            f"Target case metadata contains duplicated case_id values: n={duplicated_case_n}"
        )

    target_timestamp = pd.to_numeric(target_meta["target_timestamp_ms"], errors="coerce")
    missing_target_timestamp_n = int(target_timestamp.isna().sum())
    if missing_target_timestamp_n:
        raise RuntimeError(
            f"Target case metadata contains missing target_timestamp_ms values: n={missing_target_timestamp_n}"
        )

    target_meta["target_timestamp_ms"] = to_ms_timestamp(target_timestamp).astype(np.int64)
    nonpositive_target_timestamp_n = int(target_meta["target_timestamp_ms"].le(0).sum())
    if nonpositive_target_timestamp_n:
        raise RuntimeError(
            "Target case metadata contains non-positive target_timestamp_ms values: "
            f"n={nonpositive_target_timestamp_n}"
        )

    target_meta_for_merge = target_meta.rename(
        columns={
            "user_id": "user_id_from_target_meta",
            "target_parent_asin": "target_parent_asin_from_target_meta",
            "target_timestamp_ms": "target_timestamp_ms_from_target_meta",
        }
    )

    out = query_meta_df.merge(
        target_meta_for_merge,
        on="case_id",
        how="left",
        validate="many_to_one",
    )

    target_meta_timestamp = to_ms_timestamp(out["target_timestamp_ms_from_target_meta"]).astype(np.int64)
    target_meta_present = target_meta_timestamp.gt(0)

    if "target_timestamp_ms" in out.columns:
        query_timestamp_source = out["target_timestamp_ms"]
    elif "timestamp_ms" in out.columns:
        query_timestamp_source = out["timestamp_ms"]
    else:
        query_timestamp_source = pd.Series(np.zeros(len(out), dtype=np.int64), index=out.index)
    query_cache_timestamp = to_ms_timestamp(query_timestamp_source).astype(np.int64)

    merged_target_timestamp = target_meta_timestamp.where(
        target_meta_present,
        query_cache_timestamp,
    )
    missing_after_merge_n = int(merged_target_timestamp.le(0).sum())
    if missing_after_merge_n:
        raise RuntimeError(
            "Missing target_timestamp_ms after target metadata merge and query-cache fallback: "
            f"n={missing_after_merge_n}"
        )

    target_metadata_missing_for_query_n = int((~target_meta_present).sum())

    if "user_id" in out.columns:
        user_mismatch = (
            target_meta_present
            & (
                out["user_id"].fillna("").astype(str) != out["user_id_from_target_meta"].fillna("").astype(str)
            )
        )
        user_mismatch_n = int(user_mismatch.sum())
        if user_mismatch_n:
            raise RuntimeError(
                f"user_id mismatch after target metadata merge: n={user_mismatch_n}"
            )

    if "gt_item_id" in out.columns:
        gt_item_mismatch = (
            target_meta_present
            & (
                out["gt_item_id"].fillna("").astype(str) != out["target_parent_asin_from_target_meta"].fillna("").astype(str)
            )
        )
        gt_item_mismatch_n = int(gt_item_mismatch.sum())
        if gt_item_mismatch_n:
            raise RuntimeError(
                f"gt_item_id mismatch after target metadata merge: n={gt_item_mismatch_n}"
            )

    out["target_timestamp_ms"] = merged_target_timestamp.astype(np.int64)
    out["timestamp_ms"] = out["target_timestamp_ms"].astype(np.int64)

    drop_target_meta_cols = [
        "user_id_from_target_meta",
        "target_parent_asin_from_target_meta",
        "target_timestamp_ms_from_target_meta",
    ]
    out = out.drop(columns=[col for col in drop_target_meta_cols if col in out.columns])

    target_case_metadata_summary = {
        "target_case_metadata_path": str(target_metadata_path),
        "target_case_metadata_rows": int(len(target_meta)),
        "target_case_metadata_unique_case_id": int(target_meta["case_id"].nunique()),
        "query_rows_after_target_merge": int(len(out)),
        "target_metadata_missing_for_query_rows": int(target_metadata_missing_for_query_n),
        "target_timestamp_filled_from_query_cache_rows": int(target_metadata_missing_for_query_n),
        "target_timestamp_missing_after_merge": int(out["timestamp_ms"].isna().sum()),
    }

    return out, target_case_metadata_summary



In [22]:
# ==== Utility Helpers — Part 7 ====
def summarize_metrics(df: pd.DataFrame, group_cols):
    agg_map = {"query_id": "nunique"}
    metric_cols = [c for c in df.columns if re.match(r"^(HitRate|MRR|NDCG)@\d+$", c)]
    for col in metric_cols:
        agg_map[col] = "mean"
    for col in SUPPLEMENTARY_METRIC_COLS:
        if col in df.columns:
            agg_map[col] = "mean"
    out = (
        df.groupby(group_cols, dropna=False)
        .agg(agg_map)
        .reset_index()
        .rename(columns={"query_id": "n_queries"})
        .sort_values(group_cols)
        .reset_index(drop=True)
    )
    return out

def summarize_delta(df: pd.DataFrame, group_cols):
    agg_map = {"query_id": "nunique"}
    delta_cols = [c for c in df.columns if c.startswith("delta_")]
    for col in delta_cols:
        agg_map[col] = "mean"
    out = (
        df.groupby(group_cols, dropna=False)
        .agg(agg_map)
        .reset_index()
        .rename(columns={"query_id": "n_queries"})
        .sort_values(group_cols)
        .reset_index(drop=True)
    )
    return out

def weighted_mean_from_feature_map(feature_map, weight_map):
    vals = []
    weights = []
    for k, v in feature_map.items():
        w = float(weight_map.get(k, 0.0))
        vals.append(float(v))
        weights.append(w)
    if not weights or sum(weights) <= 0:
        return 0.0
    return float(np.average(vals, weights=weights))

def sanitize_feature_name(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_]+", "_", str(name)).strip("_").lower()

def build_time_based_valid_queries(train_query_meta: pd.DataFrame, valid_frac: float = 0.15):
    if len(train_query_meta) < 8:
        return [], train_query_meta["query_id"].astype(str).tolist()
    work = train_query_meta.sort_values(["timestamp_ms", "query_id"]).reset_index(drop=True)
    n_valid = max(1, int(round(len(work) * valid_frac)))
    n_valid = min(n_valid, max(len(work) - 4, 1))
    valid_ids = work.tail(n_valid)["query_id"].astype(str).tolist()
    fit_ids = work.iloc[:-n_valid]["query_id"].astype(str).tolist()
    if not fit_ids:
        fit_ids = work["query_id"].astype(str).tolist()
        valid_ids = []
    return valid_ids, fit_ids

def prepare_group_arrays(df: pd.DataFrame, query_col: str = "query_id"):
    df = df.sort_values([query_col, "candidate_rank", "candidate_item_id"])
    groups = df.groupby(query_col).size().astype(int).tolist()
    return df, groups

def classify_regime_from_prior_review_n(prior_review_n):
    try:
        n = int(float(prior_review_n))
    except Exception:
        n = 0

    if n == REGIME_COLD_PRIOR_REVIEW_N:
        return "cold"

    if REGIME_WEAK_PRIOR_REVIEW_MIN <= n <= REGIME_WEAK_PRIOR_REVIEW_MAX:
        return "weak"

    if REGIME_MODERATE_PRIOR_REVIEW_MIN <= n <= REGIME_MODERATE_PRIOR_REVIEW_MAX:
        return "moderate"

    if n >= REGIME_STRONG_PRIOR_REVIEW_MIN:
        return "strong"

    return "other"

def normalize_regime_label(value):
    if pd.isna(value):
        return ""

    regime = str(value).strip().lower()

    if regime in {"cold"}:
        return "cold"

    if regime in {"weak"}:
        return "weak"

    if regime in {"moderate", "modest"}:
        return "moderate"

    if regime in {"strong"}:
        return "strong"

    return regime


def ordered_regime_values(values):
    present = []
    seen = set()

    for raw in values:
        regime = normalize_regime_label(raw)

        if not regime or regime in seen:
            continue

        seen.add(regime)
        present.append(regime)

    ordered = [regime for regime in REGIME_ORDER if regime in seen]
    ordered.extend([regime for regime in present if regime not in ordered])

    return ordered


def ordered_regime_counts(values):
    series = pd.Series(list(values), dtype="object").map(normalize_regime_label)
    series = series[series.astype(str).str.len() > 0]

    counts = series.value_counts(dropna=False).to_dict()

    ordered_counts = {
        regime: int(counts[regime])
        for regime in REGIME_ORDER
        if regime in counts
    }

    ordered_counts.update({
        str(regime): int(count)
        for regime, count in counts.items()
        if str(regime) not in ordered_counts
    })

    return ordered_counts


def apply_regime_order(df, regime_col="regime"):
    if regime_col not in df.columns:
        return df

    out = df.copy(deep=False)
    normalized = out[regime_col].map(normalize_regime_label)

    categories = ordered_regime_values(normalized.tolist())
    out[regime_col] = pd.Categorical(
        normalized,
        categories=categories,
        ordered=True,
    )

    return out


In [23]:
# ==== Facial Skincare Family / Facet Overrides ====
BRAND_FEATURE_SCALE = 0.60
CONCERN_FEATURE_SCALE = 0.30

FACIAL_ATTRIBUTE_FAMILIES = (
    "brand",
    "product_type",
    "form",
    "concern",
    "benefit",
    "ingredient",
    "free",
    "flavor",
    "usage_target",
)

FACIAL_BASE_FACET_PREFIXES = (
    "product_type_",
    "form_",
    "concern_",
    "benefit_",
    "ingredient_",
    "free_",
    "flavor_",
    "usage_target_",
)

FACE_ITEM_SCHEMA_FAMILY_SOURCE_CANDIDATES = {
    "brand": ["brand_facet_text"],
    "product_type": ["product_type_norm", "product_type_norm_text", "family_sub_category", "sub_category_norm", "sub_category_norm_text", "sub_category"],
    "benefit": ["benefit_final", "benefit_final_text", "benefit_norm", "benefit_norm_text", "family_benefit_function", "product_benefits"],
    "ingredient": ["ingredient_final", "ingredient_final_text", "ingredient_norm", "ingredient_norm_text", "family_ingredient_composition", "active_ingredients", "special_ingredients"],
    "form": ["form_norm", "form_norm_text", "form", "form_text", "item_form_norm", "family_item_form", "item_form"],
    "free": ["free_claim_norm", "free_claim_norm_text", "free_claim", "free_claim_text", "family_claims_diet", "diet_type", "material_feature"],
    "flavor": ["flavor_norm", "flavor_norm_text", "flavor", "flavor_text", "family_flavor", "scent"],
    "usage_target": ["usage_target_norm", "usage_target_norm_text", "usage_target", "usage_target_text", "family_usage_target", "age_range"],
    "concern": ["concern_final", "concern_final_text", "concern_claim_norm", "concern_claim_norm_text", "usage_need_norm_text", "family_usage_need", "recommended_use", "specific_uses"],
}

ATTRIBUTE_FAMILIES = FACIAL_ATTRIBUTE_FAMILIES
INSPECT_ONLY_FACET_FAMILIES = tuple()
BASE_FACET_PREFIXES = FACIAL_BASE_FACET_PREFIXES
SUPPLEMENTARY_FAMILY_PRIORITY = ("brand", "concern", "ingredient", "free", "product_type")

MAIN_FAMILIES = list(FACIAL_ATTRIBUTE_FAMILIES)
SAFE_FACET_FALLBACK_FAMILIES = {"product_type", "form", "free", "flavor", "usage_target"}
ITEM_SCHEMA_FAMILY_SOURCE_CANDIDATES = FACE_ITEM_SCHEMA_FAMILY_SOURCE_CANDIDATES

FAMILY_BASE_MULTIPLIER = {family: 1.0 for family in ATTRIBUTE_FAMILIES}
FAMILY_BASE_MULTIPLIER["brand"] = BRAND_FEATURE_SCALE
FAMILY_BASE_MULTIPLIER["concern"] = CONCERN_FEATURE_SCALE
FAMILY_BASE_MULTIPLIER["usage_target"] = 0.70
FAMILY_BASE_MULTIPLIER["flavor"] = 0.60

FAMILY_STRENGTH_WEIGHTS = {
    "product_type": 1.50,
    "benefit": 1.20,
    "ingredient": 1.20,
    "form": 1.00,
    "free": 0.80,
    "usage_target": 0.70,
    "flavor": 0.60,
    "brand": BRAND_FEATURE_SCALE,
    "concern": CONCERN_FEATURE_SCALE,
}

SUPPLEMENTARY_METRIC_COLS = list(PREFERENCE_DIAGNOSTIC_METRICS)
UPLIFT_DIAGNOSTIC_METRICS = list(PREFERENCE_DIAGNOSTIC_METRICS)

FACET_PREFIX_TO_FAMILY = {prefix: prefix[:-1] for prefix in BASE_FACET_PREFIXES}
ORDERED_FACET_PREFIXES = tuple(sorted(BASE_FACET_PREFIXES, key=len, reverse=True))

EXCLUDED_FACET_EXACT = {
    "product_type_norm",
    "product_type_norm_text",
    "form_norm",
    "form_norm_text",
    "benefit_final",
    "benefit_final_text",
    "benefit_norm",
    "benefit_norm_text",
    "ingredient_final",
    "ingredient_final_text",
    "ingredient_norm",
    "ingredient_norm_text",
    "free_claim_norm",
    "free_claim_norm_text",
    "flavor_norm",
    "flavor_norm_text",
    "usage_target_norm",
    "usage_target_norm_text",
    "concern_final",
    "concern_final_text",
    "usage_need_norm_text",
}
EXCLUDED_FACET_SUFFIXES = ("_norm", "_final", "_text", "_norm_text", "_final_text")

def is_sequence_like_value(v):
    return isinstance(v, (list, tuple, set, dict, np.ndarray, pd.Series))

COMMON_FACET_ROLE_TO_MODEL_FAMILY = {
    "brand": "brand",
    "category_or_product_type": "product_type",
    "form_texture": "form",
    "ingredient_or_composition": "ingredient",
    "need_benefit_concern": "concern",
    "claim_constraint": "claim",
    "target_context": "skin_type",
    "sensory": "scent",
}


In [24]:
# ==== Facial-Specific Feature Family Registry ====
FACIAL_ATTRIBUTE_FAMILIES = (
    "brand",
    "product_type",
    "form",
    "formulation",
    "texture",
    "skin_type",
    "concern",
    "benefit",
    "ingredient",
    "free",
    "usage_target",
    "scent",
    "claim",
    "category",
)

FACIAL_BASE_FACET_PREFIXES = (
    "rr_product_form_texture_",
    "rr_concern_",
    "rr_skin_type_",
    "rr_benefit_",
    "rr_ingredient_",
    "product_type_",
    "formulation_",
    "form_",
    "texture_",
    "skin_type_",
    "concern_",
    "benefit_",
    "ingredient_",
    "free_",
    "usage_target_",
    "scent_",
    "flavor_",
    "claim_",
    "category_",
)

FACE_ITEM_SCHEMA_FAMILY_SOURCE_CANDIDATES = {
    "brand": ["brand_facet_text"],
    "product_type": ["product_type_norm", "product_type_norm_text", "family_sub_category", "sub_category_norm", "sub_category_norm_text", "sub_category", "facet_category_text", "category"],
    "form": ["form_norm", "form_norm_text", "form", "form_text", "item_form_norm", "family_item_form", "item_form", "facet_form_text"],
    "formulation": ["formulation_norm", "formulation_norm_text", "formulation", "formulation_text", "facet_form_text"],
    "texture": ["texture_norm", "texture_norm_text", "texture", "texture_text", "rr_product_form_texture", "review_reputation_product_form_texture"],
    "skin_type": ["skin_type_norm", "skin_type_norm_text", "skin_type", "skin_type_text", "facet_skin_type_text", "rr_skin_type", "review_reputation_skin_type"],
    "concern": ["concern_final", "concern_final_text", "concern_claim_norm", "concern_claim_norm_text", "usage_need_norm_text", "family_usage_need", "recommended_use", "specific_uses", "rr_concern", "review_reputation_concern"],
    "benefit": ["benefit_final", "benefit_final_text", "benefit_norm", "benefit_norm_text", "family_benefit_function", "product_benefits", "facet_benefit_text", "rr_benefit", "review_reputation_benefit"],
    "ingredient": ["ingredient_final", "ingredient_final_text", "ingredient_norm", "ingredient_norm_text", "family_ingredient_composition", "active_ingredients", "special_ingredients", "facet_ingredient_text", "rr_ingredient", "review_reputation_ingredient"],
    "free": ["free_claim_norm", "free_claim_norm_text", "free_claim", "free_claim_text", "family_claims_diet", "diet_type", "material_feature"],
    "usage_target": ["usage_target_norm", "usage_target_norm_text", "usage_target", "usage_target_text", "family_usage_target", "age_range"],
    "scent": ["scent", "scent_text", "flavor_norm", "flavor_norm_text", "flavor", "flavor_text", "facet_scent_text"],
    "claim": ["claim_norm", "claim_norm_text", "claim", "claim_text", "facet_claim_text"],
    "category": ["category", "category_text", "facet_category_text", "main_category", "sub_category"],
}
FACIAL_ITEM_SCHEMA_FAMILY_SOURCE_CANDIDATES = FACE_ITEM_SCHEMA_FAMILY_SOURCE_CANDIDATES

ATTRIBUTE_FAMILIES = FACIAL_ATTRIBUTE_FAMILIES
BASE_FACET_PREFIXES = FACIAL_BASE_FACET_PREFIXES
MAIN_FAMILIES = list(FACIAL_ATTRIBUTE_FAMILIES)
SAFE_FACET_FALLBACK_FAMILIES = set(FACIAL_ATTRIBUTE_FAMILIES)
ITEM_SCHEMA_FAMILY_SOURCE_CANDIDATES = FACE_ITEM_SCHEMA_FAMILY_SOURCE_CANDIDATES
SUPPLEMENTARY_FAMILY_PRIORITY = ("brand", "concern", "ingredient", "skin_type", "product_type")

FAMILY_BASE_MULTIPLIER = {family: 1.0 for family in ATTRIBUTE_FAMILIES}
FAMILY_BASE_MULTIPLIER.update({
    "brand": float(globals().get("BRAND_FEATURE_SCALE", 0.60)),
    "concern": float(globals().get("CONCERN_FEATURE_SCALE", 0.30)),
    "usage_target": 0.70,
    "scent": 0.60,
    "claim": 0.70,
    "texture": 0.90,
    "skin_type": 1.10,
})
FAMILY_STRENGTH_WEIGHTS = {
    "brand": float(globals().get("BRAND_FEATURE_SCALE", 0.60)),
    "product_type": 1.50,
    "form": 1.00,
    "formulation": 1.00,
    "texture": 0.90,
    "skin_type": 1.10,
    "concern": float(globals().get("CONCERN_FEATURE_SCALE", 0.30)),
    "benefit": 1.20,
    "ingredient": 1.20,
    "free": 0.80,
    "usage_target": 0.70,
    "scent": 0.60,
    "claim": 0.70,
    "category": 0.80,
}

FACET_PREFIX_TO_FAMILY = {
    "rr_product_form_texture_": "texture",
    "rr_concern_": "concern",
    "rr_skin_type_": "skin_type",
    "rr_benefit_": "benefit",
    "rr_ingredient_": "ingredient",
    "product_type_": "product_type",
    "formulation_": "formulation",
    "form_": "form",
    "texture_": "texture",
    "skin_type_": "skin_type",
    "concern_": "concern",
    "benefit_": "benefit",
    "ingredient_": "ingredient",
    "free_": "free",
    "usage_target_": "usage_target",
    "scent_": "scent",
    "flavor_": "scent",
    "claim_": "claim",
    "category_": "category",
}
ORDERED_FACET_PREFIXES = tuple(sorted(BASE_FACET_PREFIXES, key=len, reverse=True))

FACET_FAMILY_ALIAS_MAP = {
    "rr_product_form_texture": "texture",
    "review_reputation_product_form_texture": "texture",
    "rr_concern": "concern",
    "review_reputation_concern": "concern",
    "rr_skin_type": "skin_type",
    "review_reputation_skin_type": "skin_type",
    "rr_benefit": "benefit",
    "review_reputation_benefit": "benefit",
    "rr_ingredient": "ingredient",
    "review_reputation_ingredient": "ingredient",
    "flavor": "scent",
}

print("Facial feature families:", list(FACIAL_ATTRIBUTE_FAMILIES))


Facial feature families: ['brand', 'product_type', 'form', 'formulation', 'texture', 'skin_type', 'concern', 'benefit', 'ingredient', 'free', 'usage_target', 'scent', 'claim', 'category']


In [25]:
# ==== Utility Helpers — Part 10 ====
def infer_family_from_facet_col(col):
    base = str(col)
    if "__" in base:
        return base.split("__", 1)[0]
    for prefix in ORDERED_FACET_PREFIXES:
        if base.startswith(prefix):
            return FACET_PREFIX_TO_FAMILY[prefix]
    return base.split("_", 1)[0]


def facet_label(col):
    base = str(col)
    if "__" in base:
        return normalize_space(base.split("__", 1)[1].replace("_", " "))
    for prefix in ORDERED_FACET_PREFIXES:
        if base.startswith(prefix):
            base = base[len(prefix):]
            break
    return normalize_space(base.replace("_", " "))


def normalize_family_key(x):
    s = str(x).strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s


def normalize_facet_slug(x):
    s = str(x).strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s


def build_long_facet_objects(items_facets: pd.DataFrame):
    required_cols = [
        "parent_asin",
        "facet_role",
        "facet_value",
        "facet_value_norm",
        "is_brand",
        "is_review_derived",
        "is_profile_safe",
    ]
    missing_cols = [col for col in required_cols if col not in items_facets.columns]
    if missing_cols:
        raise RuntimeError(f"items_facets is missing Notebook 02 contract columns: {missing_cols}")

    facet_long_df = items_facets[required_cols].copy()
    facet_long_df["item_id"] = facet_long_df["parent_asin"].astype(str).str.strip()
    facet_long_df["facet_role"] = facet_long_df["facet_role"].fillna("").astype(str).str.strip()
    facet_long_df["family"] = facet_long_df["facet_role"].map(COMMON_FACET_ROLE_TO_MODEL_FAMILY).fillna("")
    facet_long_df["label_raw"] = facet_long_df["facet_value_norm"].where(
        facet_long_df["facet_value_norm"].fillna("").astype(str).str.strip().ne(""),
        facet_long_df["facet_value"],
    )
    facet_long_df["label"] = facet_long_df["label_raw"].map(normalize_text)
    facet_long_df["label_slug"] = facet_long_df["label"].map(normalize_facet_slug)
    for col in ["is_brand", "is_review_derived", "is_profile_safe"]:
        facet_long_df[col] = facet_long_df[col].fillna(False).astype(bool)

    unmapped_profile_roles = sorted(
        facet_long_df.loc[facet_long_df["is_profile_safe"] & facet_long_df["family"].eq(""), "facet_role"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )
    if unmapped_profile_roles:
        raise RuntimeError(f"Profile-safe Notebook 02 roles are not mapped: {unmapped_profile_roles}")

    facet_long_df = facet_long_df.loc[
        facet_long_df["item_id"].ne("")
        & facet_long_df["family"].ne("")
        & facet_long_df["label"].ne("")
        & facet_long_df["label_slug"].ne("")
    ].copy()

    facet_long_df["facet_col"] = facet_long_df["family"] + "__" + facet_long_df["label_slug"]
    facet_long_df = facet_long_df.drop_duplicates(
        ["item_id", "family", "label", "is_profile_safe"]
    ).reset_index(drop=True)

    facet_columns = sorted(facet_long_df["facet_col"].unique().tolist())
    facet_col_to_label = (
        facet_long_df[["facet_col", "label"]]
        .drop_duplicates("facet_col")
        .set_index("facet_col")["label"]
        .to_dict()
    )
    facet_family_to_cols = defaultdict(list)
    for family, group in facet_long_df[["family", "facet_col"]].drop_duplicates().groupby("family", sort=False):
        facet_family_to_cols[str(family)] = sorted(group["facet_col"].tolist())

    return facet_long_df, facet_columns, facet_col_to_label, facet_family_to_cols

print("Facial skincare long-format facet utilities ready.")

Facial skincare long-format facet utilities ready.


In [26]:
# ==== Facial Facet Family Aliases ====
if "normalize_family_key" in globals() and "FACET_FAMILY_ALIAS_MAP" in globals():
    _normalize_family_key_base = normalize_family_key

    def normalize_family_key(x):
        base = _normalize_family_key_base(x)
        return FACET_FAMILY_ALIAS_MAP.get(base, base)

print("Facial facet family aliases ready.")


Facial facet family aliases ready.


In [27]:
# ==== Common Feature Registry and Temporal Contract ====
TEMPORAL_FEATURE_VERSION = "temporal_recency_v2_compact_prequery_only"
COMMON_FEATURE_REGISTRY_VERSION = "lightgbm_primary_novel_item_registry_v1"
USE_TEMPORAL_RECENCY_FEATURES = True
TEMPORAL_WINDOWS_DAYS = [30, 90, 180]
MS_PER_DAY = 1000 * 60 * 60 * 24
RECENCY_FEATURE_FILL_DAYS = np.float32(9999.0)
RECENCY_COUNT_FILL = np.float32(0.0)

TEMPORAL_ARTIFACT_DIR = PROJECT_ROOT / "data" / "processed" / "temporal"
ITEM_REVIEW_TIME_INDEX_PATH = TEMPORAL_ARTIFACT_DIR / "face_item_review_timestamp_index.parquet"
ITEM_REVIEW_DAILY_COUNTS_PATH = TEMPORAL_ARTIFACT_DIR / "face_item_daily_review_counts.parquet"
ENTITY_REVIEW_DAILY_COUNTS_PATH = TEMPORAL_ARTIFACT_DIR / "face_entity_daily_review_counts.parquet"
ITEM_TEMPORAL_SUMMARY_PATH = TEMPORAL_ARTIFACT_DIR / "face_item_temporal_summary.parquet"
TEMPORAL_ARTIFACT_MANIFEST_PATH = TEMPORAL_ARTIFACT_DIR / "temporal_artifact_manifest_face.json"
TEMPORAL_LEAKAGE_RULE = "review_timestamp_ms < target_timestamp_ms"

# category and that the manifest points to the same files. Do not substitute a
# cross-category artifact or a post-target aggregate.
TEMPORAL_ARTIFACT_PATHS = {
    "item_review_time_index": ITEM_REVIEW_TIME_INDEX_PATH,
    "item_review_daily_counts": ITEM_REVIEW_DAILY_COUNTS_PATH,
    "entity_review_daily_counts": ENTITY_REVIEW_DAILY_COUNTS_PATH,
    "item_temporal_summary": ITEM_TEMPORAL_SUMMARY_PATH,
    "temporal_artifact_manifest": TEMPORAL_ARTIFACT_MANIFEST_PATH,
}

COMMON_FACET_ROLE_ORDER = ['category_or_product_type', 'form_texture', 'ingredient_or_composition', 'need_benefit_concern', 'claim_constraint', 'target_context', 'sensory']
missing_common_role_mappings = [
    role for role in COMMON_FACET_ROLE_ORDER
    if role not in COMMON_FACET_ROLE_TO_MODEL_FAMILY
]
if missing_common_role_mappings:
    raise RuntimeError(f"Missing common-role mappings: {missing_common_role_mappings}")
COMMON_ROLE_TO_MODEL_FAMILY = {
    role: COMMON_FACET_ROLE_TO_MODEL_FAMILY[role]
    for role in COMMON_FACET_ROLE_ORDER
}
NON_BRAND_FAMILIES = list(dict.fromkeys(COMMON_ROLE_TO_MODEL_FAMILY.values()))
MAIN_FAMILIES = ["brand", *NON_BRAND_FAMILIES]

COMMON_CANDIDATE_BASE_FEATURE_COLS = [
    "candidate_rank",
    "query_token_len",
    "item_title_token_overlap",
    "item_doc_token_overlap",
]
PRIMARY_COMMON_USER_BASE_FEATURE_COLS = [
    "prior_review_n_log1p",
    "prior_item_n_log1p",
    "user_entropy_norm_mean",
]
EXACT_ITEM_FAMILIARITY_FEATURE_COLS = [
    "user_item_seen_strength",
    "user_item_recency_days",
    "user_item_recent_count_180d",
]
COMMON_USER_BASE_FEATURE_COLS = list(PRIMARY_COMMON_USER_BASE_FEATURE_COLS)
COMMON_DYNAMIC_FEATURE_COLS = ["candidate_score_norm_pool"]
assert "candidate_score" not in COMMON_CANDIDATE_BASE_FEATURE_COLS
assert "candidate_score_norm_pool" in COMMON_DYNAMIC_FEATURE_COLS

COMMON_FAMILY_CANDIDATE_FEATURE_COLS = [
    f"qmatch__{family}" for family in NON_BRAND_FAMILIES
]
COMMON_FAMILY_USER_FEATURE_COLS = (
    [f"uaff__{family}" for family in NON_BRAND_FAMILIES]
    + [f"ucountlog__{family}" for family in NON_BRAND_FAMILIES]
)

COMMON_TEMPORAL_ITEM_FEATURE_COLS = [
    "item_prequery_review_count",
    "item_recent_review_share_180d",
    "item_last_review_gap_days",
    "item_first_review_age_days",
]
PRIMARY_COMMON_TEMPORAL_USER_FEATURE_COLS = [
    "user_last_interaction_gap_days",
]
COMMON_TEMPORAL_USER_FEATURE_COLS = list(PRIMARY_COMMON_TEMPORAL_USER_FEATURE_COLS)
FAMILY_RECENCY_FEATURE_FAMILIES = list(NON_BRAND_FAMILIES)
COMMON_TEMPORAL_FAMILY_FEATURE_COLS = []
for family in FAMILY_RECENCY_FEATURE_FAMILIES:
    COMMON_TEMPORAL_FAMILY_FEATURE_COLS.extend([
        f"user_{family}_recency_days",
        f"user_{family}_recent_count_180d",
    ])

BRAND_CANDIDATE_FEATURE_COLS = ["candidate_brand_present"]
BRAND_ALL_PRIOR_STATIC_FEATURE_COLS = [
    "candidate_brand_prior_interaction_count",
    "candidate_brand_prior_unique_item_count",
    "candidate_brand_prior_share",
    "user_prior_unique_brand_count",
    "user_prior_brand_entropy_norm",
]
BRAND_ALL_PRIOR_TEMPORAL_FEATURE_COLS = [
    "candidate_brand_last_interaction_age_days",
    "candidate_brand_recent_interaction_count_180d",
]
# Candidate-side brand availability is category-common information, not user prior.
# Only history-derived brand-affinity variables belong in USER_BRAND_FEATURE_COLUMNS.
USER_BRAND_FEATURE_COLUMNS = (
    BRAND_ALL_PRIOR_STATIC_FEATURE_COLS
    + BRAND_ALL_PRIOR_TEMPORAL_FEATURE_COLS
)
USER_BRAND_FEATURE_COLUMNS_P2Q = []
assert not USER_BRAND_FEATURE_COLUMNS_P2Q

P2Q_STATIC_FEATURE_COLUMNS = list(dict.fromkeys(
    COMMON_CANDIDATE_BASE_FEATURE_COLS
    + COMMON_FAMILY_CANDIDATE_FEATURE_COLS
    + COMMON_TEMPORAL_ITEM_FEATURE_COLS
    + BRAND_CANDIDATE_FEATURE_COLS
))
SHARED_ALL_PRIOR_STATIC_FEATURE_COLUMNS = list(dict.fromkeys(
    COMMON_CANDIDATE_BASE_FEATURE_COLS
    + COMMON_USER_BASE_FEATURE_COLS
    + COMMON_FAMILY_CANDIDATE_FEATURE_COLS
    + COMMON_FAMILY_USER_FEATURE_COLS
    + COMMON_TEMPORAL_ITEM_FEATURE_COLS
    + COMMON_TEMPORAL_USER_FEATURE_COLS
    + COMMON_TEMPORAL_FAMILY_FEATURE_COLS
    + BRAND_CANDIDATE_FEATURE_COLS
    + BRAND_ALL_PRIOR_STATIC_FEATURE_COLS
    + BRAND_ALL_PRIOR_TEMPORAL_FEATURE_COLS
))

P2Q_FEATURE_COLUMNS = P2Q_STATIC_FEATURE_COLUMNS + COMMON_DYNAMIC_FEATURE_COLS
if set(BRAND_CANDIDATE_FEATURE_COLS).intersection(USER_BRAND_FEATURE_COLUMNS):
    raise RuntimeError("Candidate-side brand features must not be classified as user-prior brand features.")
if not set(BRAND_CANDIDATE_FEATURE_COLS).issubset(P2Q_FEATURE_COLUMNS):
    raise RuntimeError("P2-Q must retain category-common candidate-side brand availability features.")
P2Q_FORBIDDEN_USER_PRIOR_FEATURE_COLUMNS = list(dict.fromkeys(
    COMMON_USER_BASE_FEATURE_COLS
    + EXACT_ITEM_FAMILIARITY_FEATURE_COLS
    + COMMON_FAMILY_USER_FEATURE_COLS
    + COMMON_TEMPORAL_USER_FEATURE_COLS
    + COMMON_TEMPORAL_FAMILY_FEATURE_COLS
    + USER_BRAND_FEATURE_COLUMNS
    + ["regime_cold", "regime_weak", "regime_moderate", "regime_strong"]
))
P2Q_USER_PRIOR_FEATURE_COLUMNS = sorted(
    set(P2Q_FEATURE_COLUMNS).intersection(P2Q_FORBIDDEN_USER_PRIOR_FEATURE_COLUMNS)
)
if P2Q_USER_PRIOR_FEATURE_COLUMNS:
    raise RuntimeError(
        f"P2-Q model feature contract contains user-prior features: {P2Q_USER_PRIOR_FEATURE_COLUMNS}"
    )

SHARED_ALL_PRIOR_FEATURE_COLUMNS = (
    SHARED_ALL_PRIOR_STATIC_FEATURE_COLUMNS + COMMON_DYNAMIC_FEATURE_COLS
)
PRIMARY_SHARED_ALL_PRIOR_FEATURE_COLUMNS = list(SHARED_ALL_PRIOR_FEATURE_COLUMNS)
NOVELTY_SENSITIVITY_FEATURE_COLUMNS = [
    *PRIMARY_SHARED_ALL_PRIOR_FEATURE_COLUMNS,
    *EXACT_ITEM_FAMILIARITY_FEATURE_COLS,
]
if ENABLE_EXACT_ITEM_FAMILIARITY:
    SHARED_ALL_PRIOR_FEATURE_COLUMNS = list(NOVELTY_SENSITIVITY_FEATURE_COLUMNS)
feature_columns_P2P = SHARED_ALL_PRIOR_FEATURE_COLUMNS
feature_columns_Full = SHARED_ALL_PRIOR_FEATURE_COLUMNS
feature_dtypes_P2P = {column: "float32" for column in feature_columns_P2P}
feature_dtypes_Full = {column: "float32" for column in feature_columns_Full}
assert feature_columns_P2P == feature_columns_Full
assert feature_dtypes_P2P == feature_dtypes_Full

EXPECTED_STATIC_FEATURE_COLUMNS = (
    P2Q_STATIC_FEATURE_COLUMNS
    if not USER_BRAND_AFFINITY_ENABLED
    else SHARED_ALL_PRIOR_STATIC_FEATURE_COLUMNS
)
EXPECTED_MODEL_FEATURE_COLUMNS = (
    P2Q_FEATURE_COLUMNS
    if not USER_BRAND_AFFINITY_ENABLED
    else SHARED_ALL_PRIOR_FEATURE_COLUMNS
)
COMMON_STATIC_FEATURE_COLS = EXPECTED_STATIC_FEATURE_COLUMNS
COMMON_BASE_FEATURE_COLS = EXPECTED_STATIC_FEATURE_COLUMNS
COMMON_FAMILY_FEATURE_COLS = (
    COMMON_FAMILY_CANDIDATE_FEATURE_COLS
    + ([] if not USER_BRAND_AFFINITY_ENABLED else COMMON_FAMILY_USER_FEATURE_COLS)
)
COMMON_TEMPORAL_FEATURE_COLS = (
    COMMON_TEMPORAL_ITEM_FEATURE_COLS
    + ([] if not USER_BRAND_AFFINITY_ENABLED else COMMON_TEMPORAL_USER_FEATURE_COLS)
    + ([] if not USER_BRAND_AFFINITY_ENABLED else COMMON_TEMPORAL_FAMILY_FEATURE_COLS)
    + ([] if not USER_BRAND_AFFINITY_ENABLED else BRAND_ALL_PRIOR_TEMPORAL_FEATURE_COLS)
)
COMMON_TIME_FEATURE_COLS = COMMON_TEMPORAL_FEATURE_COLS
COMMON_MODEL_FEATURES = EXPECTED_STATIC_FEATURE_COLUMNS

if len(P2Q_FEATURE_COLUMNS) != 17:
    raise RuntimeError("The LightGBM P2-Q registry must contain exactly 17 features.")
if len(PRIMARY_SHARED_ALL_PRIOR_FEATURE_COLUMNS) != 56:
    raise RuntimeError("The primary LightGBM All-Prior registry must contain exactly 56 features.")
if len(NOVELTY_SENSITIVITY_FEATURE_COLUMNS) != 59:
    raise RuntimeError("The LightGBM novelty sensitivity registry must contain exactly 59 features.")
if not set(P2Q_FEATURE_COLUMNS).issubset(PRIMARY_SHARED_ALL_PRIOR_FEATURE_COLUMNS):
    raise RuntimeError("The LightGBM prior-aware registry does not preserve every P2-Q base feature.")
if SPECIFICATION_ROLE not in {"primary", "novelty_sensitivity"}:
    raise RuntimeError(f"Unknown LightGBM specification role: {SPECIFICATION_ROLE}")
if (SPECIFICATION_ROLE == "primary") != (not ENABLE_EXACT_ITEM_FAMILIARITY):
    raise RuntimeError("LightGBM specification role and exact-item switch disagree.")
if set(PRIMARY_SHARED_ALL_PRIOR_FEATURE_COLUMNS).intersection(EXACT_ITEM_FAMILIARITY_FEATURE_COLS):
    raise RuntimeError("Previously-reviewed-item diagnostics entered the primary LightGBM registry.")
if set(NOVELTY_SENSITIVITY_FEATURE_COLUMNS) - set(PRIMARY_SHARED_ALL_PRIOR_FEATURE_COLUMNS) != set(EXACT_ITEM_FAMILIARITY_FEATURE_COLS):
    raise RuntimeError("The LightGBM novelty sensitivity extension contains unexpected features.")
if len(EXPECTED_MODEL_FEATURE_COLUMNS) != len(set(EXPECTED_MODEL_FEATURE_COLUMNS)):
    raise RuntimeError("Model feature contract contains duplicated columns.")

LIGHTGBM_RANKER_PARAMS_BASE = {
    "objective": LIGHTGBM_OBJECTIVE,
    "metric": LIGHTGBM_METRIC,
    "eval_at": list(LIGHTGBM_EVAL_AT),
    "n_estimators": LIGHTGBM_NUM_BOOST_ROUND,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_child_samples": 20,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "n_jobs": -1,
    "force_col_wise": True,
}

FORBIDDEN_MODEL_FEATURE_COLS = {
    "timestamp_ms",
    "target_timestamp_ms",
    "case_id",
    "query_id",
    "user_id",
    "target_parent_asin",
    "candidate_parent_asin",
    "target_item_id",
    "candidate_item_id",
    "gt_item_id",
    "is_gt",
    "label",
    "target_rank_desc",
    "target_selection_mode",
    "query_text",
    "raw_query_text",
    "review_text",
    "review_body",
    "raw_review_text",
    "target_review_text",
    "heldout_review_text",
}

FORBIDDEN_MODEL_FEATURE_PREFIXES = (
    "target_",
    "heldout_",
    "raw_review_",
    "review_text",
    "review_body",
)

print("COMMON_FEATURE_REGISTRY_VERSION:", COMMON_FEATURE_REGISTRY_VERSION)
print("TEMPORAL_FEATURE_VERSION:", TEMPORAL_FEATURE_VERSION)
print("Temporal feature columns:", len(COMMON_TEMPORAL_FEATURE_COLS))
print("Expected model feature columns:", len(EXPECTED_MODEL_FEATURE_COLUMNS))

COMMON_FEATURE_REGISTRY_VERSION: lightgbm_primary_novel_item_registry_v1
TEMPORAL_FEATURE_VERSION: temporal_recency_v2_compact_prequery_only
Temporal feature columns: 21
Expected model feature columns: 56


In [28]:
# ==== Temporal Artifact Loading ====
def _load_json_local(path_obj):
    path_obj = Path(path_obj)
    if not path_obj.exists():
        return {}
    with open(path_obj, "r", encoding="utf-8-sig") as f:
        return json.load(f)

with runtime_step("load_inputs", pool_depth=RUNTIME_REPORT_POOL_DEPTH, runtime_measurement_type="wall_clock_step", derived_from_existing_runtime=False):
    if USE_TEMPORAL_RECENCY_FEATURES:
        missing_temporal_paths = {
            name: str(path)
            for name, path in TEMPORAL_ARTIFACT_PATHS.items()
            if name != "temporal_artifact_manifest" and not Path(path).exists()
        }
        if missing_temporal_paths:
            raise FileNotFoundError(
                "Missing temporal artifacts. Run 04_retrieval_artifact_face_temporal.ipynb first:\n"
                + json.dumps(missing_temporal_paths, indent=2)
            )

        item_review_time_index = pd.read_parquet(ITEM_REVIEW_TIME_INDEX_PATH)
        item_review_daily_counts = pd.read_parquet(ITEM_REVIEW_DAILY_COUNTS_PATH)
        entity_review_daily_counts = pd.read_parquet(ENTITY_REVIEW_DAILY_COUNTS_PATH)
        item_temporal_summary = pd.read_parquet(ITEM_TEMPORAL_SUMMARY_PATH)
        temporal_artifact_manifest = _load_json_local(TEMPORAL_ARTIFACT_MANIFEST_PATH)

        required_item_time_cols = ["parent_asin", "review_timestamp_ms"]
        missing_item_time_cols = [c for c in required_item_time_cols if c not in item_review_time_index.columns]
        if missing_item_time_cols:
            raise RuntimeError(f"item_review_time_index is missing required columns: {missing_item_time_cols}")

        item_review_time_index["parent_asin"] = item_review_time_index["parent_asin"].fillna("").astype(str).str.strip()
        item_review_time_index["review_timestamp_ms"] = pd.to_numeric(
            item_review_time_index["review_timestamp_ms"], errors="coerce"
        )
        item_review_time_index = item_review_time_index.dropna(subset=["review_timestamp_ms"]).copy()
        item_review_time_index["review_timestamp_ms"] = item_review_time_index["review_timestamp_ms"].astype(np.int64)
        item_review_time_index = item_review_time_index[item_review_time_index["parent_asin"].ne("")].copy()

        if item_review_time_index.empty:
            raise RuntimeError("item_review_time_index is empty after timestamp cleaning.")

        print("Temporal artifacts loaded.")
        print("item_review_time_index rows:", len(item_review_time_index))
        print("item_review_daily_counts rows:", len(item_review_daily_counts))
        print("entity_review_daily_counts rows:", len(entity_review_daily_counts))
        print("item_temporal_summary rows:", len(item_temporal_summary))
    else:
        item_review_time_index = pd.DataFrame(columns=["parent_asin", "review_timestamp_ms"])
        item_review_daily_counts = pd.DataFrame()
        entity_review_daily_counts = pd.DataFrame()
        item_temporal_summary = pd.DataFrame()
        temporal_artifact_manifest = {}

Temporal artifacts loaded.
item_review_time_index rows: 984930
item_review_daily_counts rows: 789495
entity_review_daily_counts rows: 872134
item_temporal_summary rows: 77502


## 3. Load Candidate Parquet and Inspect Schema

In [29]:
# ==== Load Candidate Parquet and Inspect Schema ====
with runtime_step("load_inputs", pool_depth=RUNTIME_REPORT_POOL_DEPTH, runtime_measurement_type="wall_clock_step", derived_from_existing_runtime=False):
    candidate_raw = pd.read_parquet(CANDIDATE_POOL_PATH)
    query_cache_raw = pd.read_parquet(QUERY_CACHE_PARQUET)
    query_qc_raw = pd.read_parquet(QUERY_CACHE_SUMMARY_PATH)
    if QUERY_BRAND_QC_COLUMN not in query_qc_raw.columns:
        raise RuntimeError(f"Query QC is missing required brand column: {QUERY_BRAND_QC_COLUMN}")
    query_brand_leak_row_count = int(
        pd.to_numeric(query_qc_raw[QUERY_BRAND_QC_COLUMN], errors="coerce").fillna(0).gt(0).sum()
    )
    if query_brand_leak_row_count:
        raise RuntimeError(f"Brand or product-name terms remain in synthetic queries: {query_brand_leak_row_count}")

    candidate_df, candidate_schema_map, candidate_optional_score_cols = standardize_stage1_candidates(candidate_raw)
    query_cache_meta_df, query_cache_schema_map = build_query_cache_meta(query_cache_raw)

    candidate_row_count_loaded = int(len(candidate_raw))
    candidate_row_count_prepared = int(len(candidate_df))
    unique_case_count = int(candidate_df["case_id"].nunique())
    unique_query_count = int(candidate_df["query_id"].nunique())

    candidate_pool_types = sorted(candidate_df["candidate_pool_type"].astype(str).unique().tolist())
    candidate_methods = sorted(candidate_df["retrieval_method"].astype(str).unique().tolist())

    if candidate_pool_types != [CANDIDATE_POOL_TYPE]:
        raise RuntimeError(f"Expected candidate_pool_type={CANDIDATE_POOL_TYPE}, found: {candidate_pool_types}")
    if candidate_methods != [RETRIEVAL_METHOD]:
        raise RuntimeError(f"Expected retrieval_method={RETRIEVAL_METHOD}, found: {candidate_methods}")

    duplicate_candidate_n = int(candidate_df.duplicated(["query_id", "candidate_item_id"]).sum())
    if duplicate_candidate_n:
        raise RuntimeError(f"Duplicate query-candidate rows in candidate input: {duplicate_candidate_n}")

    candidate_counts_df = candidate_df.groupby("query_id").size().rename("candidate_n").reset_index()
    invalid_count_mask = ~candidate_counts_df["candidate_n"].between(1, POOL_K)
    if invalid_count_mask.any():
        bad_counts = candidate_counts_df.loc[invalid_count_mask].head(10).to_dict("records")
        raise RuntimeError(
            f"Candidate count must be between 1 and {POOL_K} per query. Examples: {bad_counts}"
        )

    invalid_rank_n = int((~candidate_df["candidate_rank"].between(1, POOL_K)).sum())
    if invalid_rank_n:
        raise RuntimeError(f"candidate_rank must be between 1 and {POOL_K}; invalid rows={invalid_rank_n}")

    candidate_rank_summary = (
        candidate_df.groupby("query_id")["candidate_rank"]
        .agg(rank_min="min", rank_max="max", rank_nunique="nunique")
        .merge(candidate_counts_df, on="query_id", how="left", validate="one_to_one")
    )
    bad_rank_summary = candidate_rank_summary.loc[
        ~candidate_rank_summary["rank_min"].eq(1)
        | ~candidate_rank_summary["rank_max"].eq(candidate_rank_summary["candidate_n"])
        | ~candidate_rank_summary["rank_nunique"].eq(candidate_rank_summary["candidate_n"])
    ]
    if not bad_rank_summary.empty:
        raise RuntimeError(
            "Candidate ranks must be contiguous from 1 to each query's candidate count. Examples: "
            f"{bad_rank_summary.head(10).to_dict('records')}"
        )

    candidate_required_cols = list(REQUIRED_CANDIDATE_COLUMNS)
    missing_candidate_required = [col for col in candidate_required_cols if col not in candidate_df.columns]
    if missing_candidate_required:
        raise RuntimeError(f"Standardized candidate frame is missing required columns: {missing_candidate_required}")

    candidate_query_cols = [
        "benchmark_query_set",
        "benchmark_query_set_raw",
        "split_strategy",
        "category_key",
        "case_id",
        "query_id",
        "target_selection_mode",
        "query_method",
        "candidate_pool_type",
        "retrieval_method",
        "retrieval_method_label",
        "user_id",
        "gt_item_id",
        "regime",
        "sampling_bracket",
        "query_text",
    ]
    candidate_query_df = candidate_df[candidate_query_cols].drop_duplicates("query_id").reset_index(drop=True)

    query_meta_df = candidate_query_df.merge(
        query_cache_meta_df,
        on=["case_id", "query_id"],
        how="left",
        validate="one_to_one",
        suffixes=("_cand", "_cache"),
    )

    del candidate_raw, query_cache_raw, query_qc_raw
    gc.collect()

In [30]:
# ==== Load Candidate Parquet and Inspect Schema — Part 2 ====
with runtime_step("load_inputs", pool_depth=RUNTIME_REPORT_POOL_DEPTH, runtime_measurement_type="wall_clock_step", derived_from_existing_runtime=False):
    query_meta_df["benchmark_query_set"] = query_meta_df["benchmark_query_set"].fillna("").astype(str)
    query_meta_df["benchmark_query_set_raw"] = query_meta_df["benchmark_query_set_raw"].fillna("").astype(str)
    query_meta_df["split_strategy"] = query_meta_df["split_strategy"].fillna("").astype(str)
    query_meta_df["category_key"] = query_meta_df["category_key"].fillna(CATEGORY_ID).astype(str)
    query_meta_df["retrieval_method"] = RETRIEVAL_METHOD
    query_meta_df["retrieval_method_label"] = RETRIEVAL_METHOD_LABEL

    query_meta_df["user_id"] = query_meta_df["user_id_cand"].fillna("").astype(str)
    query_meta_df["gt_item_id"] = query_meta_df["gt_item_id_cand"].astype(str)
    query_meta_df, target_case_metadata_summary = merge_target_case_metadata(
        query_meta_df,
        TARGET_CASE_METADATA_PARQUET,
    )
    query_cache_schema_map["timestamp_ms"] = "target_case_metadata.target_timestamp_ms"
    query_cache_schema_map["target_timestamp_ms"] = "target_case_metadata.target_timestamp_ms"

    query_meta_df["query_text_candidate"] = query_meta_df["query_text"].fillna("").astype(str)
    query_meta_df["query_text_cache"] = query_meta_df["query_text_cache"].fillna("").astype(str)
    query_meta_df["query_text"] = query_meta_df["query_text_candidate"].where(
        query_meta_df["query_text_candidate"].str.len() > 0,
        query_meta_df["query_text_cache"],
    ).fillna("").astype(str)

    query_meta_df["regime"] = query_meta_df["regime_cand"].fillna("").astype(str).map(normalize_regime_label)
    query_meta_df["sampling_bracket"] = query_meta_df["sampling_bracket_cand"].fillna("").astype(str)
    query_meta_df["timestamp_ms"] = pd.to_numeric(query_meta_df["timestamp_ms"], errors="coerce").fillna(0).astype(np.int64)
    query_meta_df["query_token_len"] = pd.to_numeric(query_meta_df["query_token_len"], errors="coerce").fillna(0).astype(int)
    query_meta_df["removed_token_count"] = pd.to_numeric(query_meta_df["removed_token_count"], errors="coerce").fillna(0).astype(int)
    query_meta_df["prior_review_n"] = pd.to_numeric(query_meta_df["prior_review_n"], errors="coerce").fillna(0).astype(int)
    query_meta_df["prior_item_n"] = pd.to_numeric(query_meta_df["prior_item_n"], errors="coerce").fillna(0).astype(int)
    query_meta_df["user_total_reviews"] = pd.to_numeric(query_meta_df["user_total_reviews"], errors="coerce").fillna(0).astype(int)
    query_meta_df["query_source"] = query_meta_df["query_source"].fillna("").astype(str)
    query_meta_df["facet_hit_count"] = pd.to_numeric(query_meta_df["facet_hit_count"], errors="coerce").fillna(0).astype(int)
    query_meta_df["review_signal_count"] = pd.to_numeric(query_meta_df["review_signal_count"], errors="coerce").fillna(0).astype(int)

    query_regime_by_query = query_meta_df.set_index("query_id")["regime"].astype(str).to_dict()
    candidate_df["regime"] = candidate_df["query_id"].astype(str).map(query_regime_by_query).fillna(candidate_df["regime"].astype(str))

    query_meta_df = apply_regime_order(query_meta_df, "regime")
    candidate_df = apply_regime_order(candidate_df, "regime")

    missing_timestamp_n = int((query_meta_df["timestamp_ms"] <= 0).sum())
    text_mismatch_mask = (
        query_meta_df["query_text_candidate"].map(normalize_text) != query_meta_df["query_text_cache"].map(normalize_text)
    ) & query_meta_df["query_text_cache"].astype(str).ne("")
    query_text_mismatch_n = int(text_mismatch_mask.sum())

    candidate_counts_summary = candidate_counts_df["candidate_n"].describe()
    gt_presence_df = pd.DataFrame([
        {
            "pool_depth": int(depth),
            "gt_presence_rate": float(
                candidate_df[candidate_df["candidate_rank"] <= int(depth)]
                .groupby("query_id")["is_gt"].max().mean()
            ),
        }
        for depth in POOL_DEPTHS
    ])

    actual_regimes = ordered_regime_values(candidate_df["regime"].astype(str).dropna().unique().tolist())
    regime_match = actual_regimes == EXPECTED_REGIMES
    regime_note = "" if regime_match else "actual regimes differ from expected canonical regime order"
    regime_check_df = pd.DataFrame([{
        "actual_regimes": actual_regimes,
        "expected_regimes": EXPECTED_REGIMES,
        "regime_match": bool(regime_match),
        "note": regime_note,
    }])

    score_nonnull_df = pd.DataFrame([{
        "column": "candidate_score",
        "nonnull_rate": nonnull_rate(candidate_df["candidate_score"]),
    }])

    regime_column_used = "regime"
    query_regime_counts = ordered_regime_counts(query_meta_df["regime"])

    print("Missing timestamp_ms count:", missing_timestamp_n)
    print("Loaded candidate path:", CANDIDATE_POOL_PATH)
    print("Candidate row count loaded:", candidate_row_count_loaded)
    print("Candidate row count prepared:", candidate_row_count_prepared)
    print("Stage 1 query method:", STAGE1_QUERY_METHOD)
    print("Retrieval method:", RETRIEVAL_METHOD)
    print("Unique case_id count:", unique_case_count)
    print("Unique query_id count:", unique_query_count)
    print("Loaded query cache path:", QUERY_CACHE_PARQUET)
    print("Loaded target case metadata path:", TARGET_CASE_METADATA_PARQUET)
    print("Target case metadata row count:", target_case_metadata_summary["target_case_metadata_rows"])
    print("Regime column used:", regime_column_used)
    print("Actual regimes in candidate input:", actual_regimes)
    print("Expected regimes for check:", EXPECTED_REGIMES)
    print("Regime check match:", regime_match)
    if regime_note:
        print("Regime note:", regime_note)
    print("Regime counts in canonical order:", query_regime_counts)
    print("Required columns present:", sorted(candidate_required_cols))
    print("Query text mismatch count vs query cache:", query_text_mismatch_n)

    display(pd.DataFrame([candidate_schema_map]))
    display(pd.DataFrame([query_cache_schema_map]))
    display(candidate_counts_summary.to_frame().T)
    display(gt_presence_df)
    display(score_nonnull_df)
    display(candidate_df.head(10))

Missing timestamp_ms count: 0
Loaded candidate path: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage1_candidate_pools/by_method/personalized_winner_top1000_face.parquet
Candidate row count loaded: 2288000
Candidate row count prepared: 2288000
Stage 1 query method: C
Retrieval method: Profile Sparse QCHA
Unique case_id count: 2288
Unique query_id count: 2288
Loaded query cache path: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/query_cache/face_queries.parquet
Loaded target case metadata path: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/user_sampling/face_target_case_metadata.parquet
Target case metadata row count: 28348
Regime column used: regime
Actual regimes in candidate input: ['cold', 'weak', 'moderate', 'strong']
Expected regimes for check: ['cold', 'weak', 'moderate', 'strong']
Regime check match: True
Regime counts in canonical order: {'cold': 572, 'weak': 572, 'moderate': 572, 'strong': 

,case_id,query_id,user_id,regime,sampling_bracket,target_selection_mode,query_method,gt_item_id,query_text,retrieval_method,retrieval_method_label,candidate_item_id,candidate_brand_facet_text,candidate_rank,candidate_score,is_gt,candidate_pool_type,benchmark_query_set,benchmark_query_set_raw,split_strategy,category_key
0,case_id,query_id,user_id,regime,sampling_bracket,target_selection_mode,query_method,gt_item_id,query_text,retrieval_method,retrieval_method_label,candidate_parent_asin,candidate_brand_facet_text,candidate_rank,candidate_score,is_target,None,None,None,None,None


,case_id,query_id,user_id,gt_item_id,timestamp_ms,query_text,regime,sampling_bracket,prior_review_n,prior_item_n,user_total_reviews,query_token_len,removed_token_count,query_source,facet_hit_count,review_signal_count,target_timestamp_ms
0,case_id,case_id + '__C' compatibility key,user_id,target_parent_asin,target_case_metadata.target_timestamp_ms,query,regime,sampling_bracket if present; otherwise regime,prior_history_n if present; otherwise derived ...,prior_history_n if present; otherwise derived ...,prior_history_n if present; otherwise derived ...,computed_from_query,None,query,None,None,target_case_metadata.target_timestamp_ms


,count,mean,std,min,25%,50%,75%,max
candidate_n,2288.0,1000.0,0.0,1000.0,1000.0,1000.0,1000.0,1000.0


,pool_depth,gt_presence_rate
0,100,0.080857
1,300,0.132430
2,500,0.172640
3,700,0.207168
4,1000,0.246941


,column,nonnull_rate
0,candidate_score,1.0


,benchmark_query_set,benchmark_query_set_raw,split_strategy,category_key,case_id,query_id,user_id,regime,sampling_bracket,target_selection_mode,query_method,gt_item_id,query_text,candidate_item_id,candidate_brand_facet_text,candidate_rank,candidate_score,candidate_pool_type,retrieval_method,retrieval_method_label,is_gt
0,,,,face,00066272543b,00066272543b__C,AENEHY5JPZLQYAUHNTSQFWHF5EAQ,cold,0,recent_eligible_review_rank_le5,C,B07P3JVDBT,redness texture sensitive skin salicylic acid ...,B0B8TFCTYL,Pure Organic Ingredients,1,0.025963,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,0
1,,,,face,00066272543b,00066272543b__C,AENEHY5JPZLQYAUHNTSQFWHF5EAQ,cold,0,recent_eligible_review_rank_le5,C,B07P3JVDBT,redness texture sensitive skin salicylic acid ...,B0084Y1NG4,LIFE-FLO,2,0.025380,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,0
2,,,,face,00066272543b,00066272543b__C,AENEHY5JPZLQYAUHNTSQFWHF5EAQ,cold,0,recent_eligible_review_rank_le5,C,B07P3JVDBT,redness texture sensitive skin salicylic acid ...,B06W5H9K5H,Medicube,3,0.024204,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,0
3,,,,face,00066272543b,00066272543b__C,AENEHY5JPZLQYAUHNTSQFWHF5EAQ,cold,0,recent_eligible_review_rank_le5,C,B07P3JVDBT,redness texture sensitive skin salicylic acid ...,B0CBMHWLQW,The Natural and Organic Family,4,0.022543,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,0
4,,,,face,00066272543b,00066272543b__C,AENEHY5JPZLQYAUHNTSQFWHF5EAQ,cold,0,recent_eligible_review_rank_le5,C,B07P3JVDBT,redness texture sensitive skin salicylic acid ...,B003WX6YDY,Jan Marini Skin Research,5,0.021549,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,0
5,,,,face,00066272543b,00066272543b__C,AENEHY5JPZLQYAUHNTSQFWHF5EAQ,cold,0,recent_eligible_review_rank_le5,C,B07P3JVDBT,redness texture sensitive skin salicylic acid ...,B09V8D4542,THE ORDINARY,6,0.021508,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,0
6,,,,face,00066272543b,00066272543b__C,AENEHY5JPZLQYAUHNTSQFWHF5EAQ,cold,0,recent_eligible_review_rank_le5,C,B07P3JVDBT,redness texture sensitive skin salicylic acid ...,B004TTXA8W,Unknown,7,0.021270,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,0
7,,,,face,00066272543b,00066272543b__C,AENEHY5JPZLQYAUHNTSQFWHF5EAQ,cold,0,recent_eligible_review_rank_le5,C,B07P3JVDBT,redness texture sensitive skin salicylic acid ...,B07CLR6H95,THE ORDINARY,8,0.021214,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,0
8,,,,face,00066272543b,00066272543b__C,AENEHY5JPZLQYAUHNTSQFWHF5EAQ,cold,0,recent_eligible_review_rank_le5,C,B07P3JVDBT,redness texture sensitive skin salicylic acid ...,B09QH12664,THE ELEM ENTS,9,0.020925,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,0
9,,,,face,00066272543b,00066272543b__C,AENEHY5JPZLQYAUHNTSQFWHF5EAQ,cold,0,recent_eligible_review_rank_le5,C,B07P3JVDBT,redness texture sensitive skin salicylic acid ...,B00CB6Z63W,cane + austin,10,0.019918,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,0


## 4. Load Supporting Item and History Artifacts

In [31]:
# ==== Load Supporting Item and History Artifacts ====
with runtime_step(
    "load_inputs",
    pool_depth=RUNTIME_REPORT_POOL_DEPTH,
    runtime_measurement_type="wall_clock_step",
    derived_from_existing_runtime=False,
):
    item_schema = pd.read_parquet(ITEM_SCHEMA_PARQUET)
    items_facets = pd.read_parquet(ITEMS_FACETS_PARQUET)
    item_docs = pd.read_parquet(ITEM_DOCS_PARQUET)
    prior_history_raw = pd.read_parquet(PRIOR_HISTORY_PARQUET)

    required_item_schema_cols = ["parent_asin", "title", CANDIDATE_BRAND_SOURCE_COLUMN]
    required_item_docs_cols = ["parent_asin"]
    required_items_facets_cols = [
        "parent_asin", "facet_role", "facet_value", "facet_value_norm",
        "is_brand", "is_review_derived", "is_profile_safe",
    ]
    required_prior_cols = [
        "case_id", "user_id", "target_timestamp_ms", "prior_item_id", "prior_timestamp_ms",
    ]

    for name, frame, cols in [
        ("item_schema", item_schema, required_item_schema_cols),
        ("item_docs", item_docs, required_item_docs_cols),
        ("items_facets", items_facets, required_items_facets_cols),
        ("prior_history", prior_history_raw, required_prior_cols),
    ]:
        missing_cols = [col for col in cols if col not in frame.columns]
        if missing_cols:
            raise RuntimeError(f"{name} is missing required contract columns: {missing_cols}")

    item_schema["item_id"] = item_schema["parent_asin"].astype(str).str.strip()
    items_facets["item_id"] = items_facets["parent_asin"].astype(str).str.strip()
    item_docs["item_id"] = item_docs["parent_asin"].astype(str).str.strip()

    prior_history_raw["case_id"] = prior_history_raw["case_id"].astype(str).str.strip()
    prior_history_raw["user_id"] = prior_history_raw["user_id"].fillna("").astype(str).str.strip()
    prior_history_raw["prior_item_id"] = prior_history_raw["prior_item_id"].fillna("").astype(str).str.strip()
    prior_history_raw["target_timestamp_ms"] = to_ms_timestamp(
        prior_history_raw["target_timestamp_ms"]
    ).astype(np.int64)
    prior_history_raw["prior_timestamp_ms"] = to_ms_timestamp(
        prior_history_raw["prior_timestamp_ms"]
    ).astype(np.int64)
    prior_history_raw = prior_history_raw.loc[
        prior_history_raw["case_id"].isin(set(query_meta_df["case_id"].astype(str)))
    ].copy()

    prior_validation = prior_history_raw.merge(
        query_meta_df[["case_id", "user_id", "gt_item_id", "target_timestamp_ms"]].drop_duplicates("case_id"),
        on="case_id",
        how="left",
        validate="many_to_one",
        suffixes=("", "_query"),
    )
    user_mismatch_n = int(prior_validation["user_id"].ne(prior_validation["user_id_query"]).sum())
    target_timestamp_mismatch_n = int(
        prior_validation["target_timestamp_ms"].ne(prior_validation["target_timestamp_ms_query"]).sum()
    )
    same_target_prior_n = int(prior_validation["prior_item_id"].eq(prior_validation["gt_item_id"]).sum())

    future_prior_n = int(
        prior_validation["prior_timestamp_ms"].ge(prior_validation["target_timestamp_ms"]).sum()
    )
    if user_mismatch_n:
        raise RuntimeError(f"Notebook 03 user_id mismatch: {user_mismatch_n}")
    if target_timestamp_mismatch_n:
        raise RuntimeError(f"Notebook 03 target timestamp mismatch: {target_timestamp_mismatch_n}")
    if future_prior_n:
        raise RuntimeError(f"Prior rows at or after target timestamp: {future_prior_n}")
    if same_target_prior_n:
        raise RuntimeError(f"Same-target prior rows detected: {same_target_prior_n}")

    prior_history = pd.DataFrame({
        "case_id": prior_history_raw["case_id"].astype(str),
        "user_id": prior_history_raw["user_id"].astype(str),
        "item_id": prior_history_raw["prior_item_id"].astype(str),
        "timestamp_ms": prior_history_raw["prior_timestamp_ms"].astype(np.int64),
        "target_timestamp_ms": prior_history_raw["target_timestamp_ms"].astype(np.int64),
    })

    candidate_item_ids = set(candidate_df["candidate_item_id"].astype(str).str.strip())
    candidate_gt_item_ids = set(query_meta_df["gt_item_id"].astype(str).str.strip())
    target_item_ids = candidate_item_ids | candidate_gt_item_ids

    facet_long_df, facet_columns, facet_col_to_label, facet_family_to_cols = build_long_facet_objects(items_facets)
    profile_facet_long_df = facet_long_df.loc[facet_long_df["is_profile_safe"]].copy()
    historical_population_review_signals_in_user_profile = bool(
        profile_facet_long_df["is_review_derived"].any()
    )
    if historical_population_review_signals_in_user_profile:
        raise RuntimeError("Historical population-level review signals entered the user-profile facet supply.")

    schema_base = item_schema.drop_duplicates("item_id").reset_index(drop=True)
    meta_df = schema_base[["item_id", "title", CANDIDATE_BRAND_SOURCE_COLUMN]].copy()
    meta_df = meta_df.rename(columns={CANDIDATE_BRAND_SOURCE_COLUMN: "candidate_brand"})
    meta_df["title"] = meta_df["title"].fillna("").astype(str)
    meta_df["candidate_brand"] = meta_df["candidate_brand"].fillna("").astype(str)

    item_title_map = meta_df.set_index("item_id")["title"].to_dict()
    item_brand_map = meta_df.set_index("item_id")["candidate_brand"].to_dict()
    candidate_brand_nonnull_rate = float(
        candidate_df["candidate_item_id"].astype(str).map(item_brand_map).fillna("").str.strip().ne("").mean()
    )

    candidate_facet_coverage = (
        len(candidate_item_ids & set(facet_long_df["item_id"].astype(str)))
        / max(1, len(candidate_item_ids))
    )

    print("item_schema shape:", item_schema.shape)
    print("item_docs shape:", item_docs.shape)
    print("items_facets shape:", items_facets.shape)
    print("prior_history shape:", prior_history.shape)
    print("candidate brand non-null rate:", round(candidate_brand_nonnull_rate, 6))
    print("candidate facet coverage:", round(float(candidate_facet_coverage), 6))


item_schema shape: (77502, 129)
item_docs shape: (77502, 56)
items_facets shape: (984052, 29)
prior_history shape: (29225, 5)
candidate brand non-null rate: 0.978531
candidate facet coverage: 1.0


In [32]:
# ==== Load Supporting Item and History Artifacts — Part 2 ====
schema_family_map = {}
schema_family_text_map = {}
schema_family_token_map = {}

for row in tqdm(schema_base.itertuples(index=False), total=len(schema_base), desc="Build schema family maps", leave=False):
    row_dict = row._asdict()
    item_id = str(row_dict["item_id"])
    family_map = {}
    family_text_map = {}
    family_token_map = {}

    for family, source_columns in ITEM_SCHEMA_FAMILY_SOURCE_CANDIDATES.items():
        values = []
        for column in source_columns:
            if column in row_dict:
                values.extend(ensure_normalized_label_list(row_dict.get(column)))
        values = dedupe_keep_order(values)
        if values:
            family_map[family] = values
            family_text_map[family] = " | ".join(values)
            family_token_map[family] = set(tokenize(" ".join(values)))
        else:
            family_token_map[family] = set()

    schema_family_map[item_id] = family_map
    schema_family_text_map[item_id] = family_text_map
    schema_family_token_map[item_id] = family_token_map


def build_facet_family_map(frame: pd.DataFrame) -> dict:
    output = {}
    for item_id, item_rows in frame.groupby("item_id", sort=False):
        family_map = {}
        for family, family_rows in item_rows.groupby("family", sort=False):
            values = dedupe_keep_order(family_rows["label"].dropna().astype(str).tolist())
            if values:
                family_map[str(family)] = values
        output[str(item_id)] = family_map
    return output


facet_family_map = build_facet_family_map(facet_long_df)
profile_facet_family_map = build_facet_family_map(profile_facet_long_df)

facet_family_text_map = {
    item_id: {family: " | ".join(values) for family, values in family_map.items()}
    for item_id, family_map in facet_family_map.items()
}
facet_family_token_map = {
    item_id: {family: set(tokenize(" ".join(values))) for family, values in family_map.items()}
    for item_id, family_map in facet_family_map.items()
}

item_family_labels_map = {}
item_family_text_map = {}
item_family_token_map = {}
item_label_count_map = {}
profile_item_family_labels_map = {}

all_meta_item_ids = (
    set(meta_df["item_id"].astype(str))
    | set(schema_base["item_id"].astype(str))
    | set(facet_family_map)
    | set(profile_facet_family_map)
)

for item_id in tqdm(sorted(all_meta_item_ids), desc="Merge item family maps", leave=False):
    merged_family_map = {}
    merged_text_map = {}
    merged_token_map = {}
    profile_family_map = {}

    for family in MAIN_FAMILIES:
        schema_values = schema_family_map.get(item_id, {}).get(family, [])
        facet_values = (
            facet_family_map.get(item_id, {}).get(family, [])
            if family in SAFE_FACET_FALLBACK_FAMILIES or family == "brand"
            else []
        )
        values = dedupe_keep_order(schema_values + facet_values)
        if family == "brand":
            values = ensure_normalized_label_list(item_brand_map.get(item_id, ""))

        if values:
            merged_family_map[family] = values
            merged_text_map[family] = " | ".join(values)
            merged_token_map[family] = set(tokenize(" ".join(values)))
        else:
            merged_token_map[family] = set()

        profile_values = dedupe_keep_order(
            profile_facet_family_map.get(item_id, {}).get(family, [])
        )
        if family == "brand":
            profile_values = ensure_normalized_label_list(item_brand_map.get(item_id, ""))
        if profile_values:
            profile_family_map[family] = profile_values

    item_family_labels_map[item_id] = merged_family_map
    item_family_text_map[item_id] = merged_text_map
    item_family_token_map[item_id] = merged_token_map
    item_label_count_map[item_id] = {
        family: len(merged_family_map.get(family, []))
        for family in MAIN_FAMILIES
    }
    profile_item_family_labels_map[item_id] = profile_family_map

item_facet_text_map = {
    item_id: " | ".join(sorted(set(sum(family_map.values(), []))))
    for item_id, family_map in item_family_labels_map.items()
}

family_vocab_map = {}
for family in MAIN_FAMILIES:
    values = set()
    for family_map in profile_item_family_labels_map.values():
        values.update(
            normalize_text(value)
            for value in family_map.get(family, [])
            if normalize_text(value)
        )
    family_vocab_map[family] = sorted(values)
family_vocab_sizes = {family: max(len(values), 1) for family, values in family_vocab_map.items()}

supplementary_families_used = [family for family in SUPPLEMENTARY_FAMILY_PRIORITY if family in family_vocab_map]
supplementary_weight_raw = {family: 1.0 for family in supplementary_families_used}
if "concern" in supplementary_weight_raw:
    supplementary_weight_raw["concern"] = CONCERN_FEATURE_SCALE
supplementary_family_weights = normalize_family_weight_map(
    supplementary_families_used,
    family_weights=supplementary_weight_raw,
)

missing_candidate_items = sorted(target_item_ids - set(meta_df["item_id"].astype(str)))
if missing_candidate_items:
    raise RuntimeError(f"Stage 1 candidate items missing from Notebook 02 item schema: n={len(missing_candidate_items)}")

prior_history = prior_history.loc[
    prior_history["user_id"].ne("")
    & prior_history["item_id"].isin(set(meta_df["item_id"].astype(str)))
].copy()

item_field_coverage_rows = []
for family in MAIN_FAMILIES:
    non_empty = sum(
        bool(item_family_labels_map.get(item_id, {}).get(family, []))
        for item_id in target_item_ids
    )
    item_field_coverage_rows.append({
        "family": family,
        "candidate_item_non_empty_n": int(non_empty),
        "candidate_item_non_empty_rate": float(non_empty / max(len(target_item_ids), 1)),
        "vocab_size": int(family_vocab_sizes.get(family, 1)),
    })
item_field_coverage_df = pd.DataFrame(item_field_coverage_rows).sort_values("family").reset_index(drop=True)

print("meta_df shape:", meta_df.shape)
print("prior_history shape:", prior_history.shape)
print("profile-safe facet rows:", len(profile_facet_long_df))
print("MAIN_FAMILIES:", MAIN_FAMILIES)
display(item_field_coverage_df)


Build schema family maps:   0%|          | 0/77502 [00:00<?, ?it/s]

Merge item family maps:   0%|          | 0/77502 [00:00<?, ?it/s]

meta_df shape: (77502, 3)
prior_history shape: (29204, 5)
profile-safe facet rows: 608547
MAIN_FAMILIES: ['brand', 'product_type', 'form', 'ingredient', 'concern', 'claim', 'skin_type', 'scent']


,family,candidate_item_non_empty_n,candidate_item_non_empty_rate,vocab_size
0,brand,68384,0.968941,17856
1,claim,13359,0.189285,294
2,concern,55588,0.787633,2845
3,form,44126,0.625227,523
4,ingredient,13906,0.197036,3741
5,product_type,69637,0.986695,31
6,scent,29691,0.420695,2491
7,skin_type,30733,0.435460,7


## 5. Rebuild Strict Prior User History

In [33]:
# ==== Rebuild Strict Prior User History ====
# Use the complete Notebook 03 strict pre-target history without QCHS filtering or item deduplication.
_actual_prior_counts = (
    prior_history.groupby("case_id")
    .size()
    .rename("actual_prior_review_n")
    .reset_index()
)
_actual_prior_counts["case_id"] = _actual_prior_counts["case_id"].astype(str)

query_meta_df = query_meta_df.drop(columns=["actual_prior_review_n"], errors="ignore").merge(
    _actual_prior_counts,
    on="case_id",
    how="left",
    validate="many_to_one",
)
query_meta_df["prior_review_n"] = (
    query_meta_df["actual_prior_review_n"]
    .fillna(0)
    .astype(int)
)
query_meta_df["prior_history_n"] = query_meta_df["prior_review_n"]

eval_query_users_df = query_meta_df[
    [
        "query_id", "case_id", "user_id", "gt_item_id", "target_timestamp_ms",
        "query_text", "regime", "sampling_bracket", "prior_review_n",
        "prior_item_n", "user_total_reviews",
    ]
].drop_duplicates("query_id")
eval_query_users_df["case_id"] = eval_query_users_df["case_id"].astype(str)
eval_query_users_df["user_id"] = eval_query_users_df["user_id"].fillna("").astype(str)
eval_query_users_df["gt_item_id"] = eval_query_users_df["gt_item_id"].astype(str)
eval_query_users_df["target_timestamp_ms"] = pd.to_numeric(
    eval_query_users_df["target_timestamp_ms"], errors="coerce"
).astype(np.int64)

if eval_query_users_df["user_id"].duplicated().any():
    raise RuntimeError("The evaluation cohort must contain at most one target query per user.")
if DEDUP_HISTORY_BY_USER_ITEM:
    raise RuntimeError("All Prior requires every strict pre-target interaction; item-level history deduplication must be disabled.")

prior_user_events = prior_history[
    ["case_id", "user_id", "item_id", "timestamp_ms", "target_timestamp_ms"]
].copy()
prior_user_events["case_id"] = prior_user_events["case_id"].astype(str)
prior_user_events["user_id"] = prior_user_events["user_id"].fillna("").astype(str)
prior_user_events["item_id"] = prior_user_events["item_id"].fillna("").astype(str)
prior_user_events["timestamp_ms"] = pd.to_numeric(
    prior_user_events["timestamp_ms"], errors="coerce"
).astype(np.int64)
prior_user_events["target_timestamp_ms"] = pd.to_numeric(
    prior_user_events["target_timestamp_ms"], errors="coerce"
).astype(np.int64)
prior_user_events = prior_user_events.loc[
    prior_user_events["case_id"].isin(set(eval_query_users_df["case_id"]))
    & prior_user_events["user_id"].ne("")
    & prior_user_events["item_id"].ne("")
].copy()

prior_validation = prior_user_events.merge(
    eval_query_users_df[["case_id", "user_id", "gt_item_id", "target_timestamp_ms"]],
    on="case_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_query"),
)
all_prior_timestamps_strict = bool(
    prior_validation["timestamp_ms"].lt(prior_validation["target_timestamp_ms"]).all()
)
same_target_prior_n = int(prior_validation["item_id"].eq(prior_validation["gt_item_id"]).sum())
if not all_prior_timestamps_strict:
    raise RuntimeError("All prior timestamps must be strictly earlier than the target timestamp.")
if same_target_prior_n:
    raise RuntimeError(f"All Prior contains same-target-item rows: {same_target_prior_n}")

prior_user_events = prior_user_events.sort_values(
    ["case_id", "timestamp_ms", "item_id"],
    kind="mergesort",
).reset_index(drop=True)

history_count_check_df = (
    eval_query_users_df[["query_id", "case_id", "prior_review_n"]]
    .merge(
        prior_user_events.groupby("case_id").size().rename("profile_history_review_n").reset_index(),
        on="case_id",
        how="left",
    )
    .fillna({"profile_history_review_n": 0})
)
history_count_check_df["profile_history_review_n"] = history_count_check_df[
    "profile_history_review_n"
].astype(int)
history_count_check_df["history_count_diff"] = (
    history_count_check_df["profile_history_review_n"]
    - history_count_check_df["prior_review_n"].astype(int)
)
if history_count_check_df["history_count_diff"].ne(0).any():
    bad_rows = history_count_check_df.loc[
        history_count_check_df["history_count_diff"].ne(0)
    ].head(10).to_dict("records")
    raise RuntimeError(f"Notebook 03 All Prior counts do not match query metadata: {bad_rows}")

print("All Prior rows:", len(prior_user_events))
print("All Prior temporal validation: passed")
print("Same-target prior rows:", same_target_prior_n)


All Prior rows: 29204
All Prior temporal validation: passed
Same-target prior rows: 0


## 6. Build User Profiles and Affinity Tables

In [34]:
# ==== Build User Profiles and Affinity Tables ====
# Build profile-safe user preferences from every strict pre-target interaction.

def build_family_counters_from_item_count_map(item_count_map):
    counters = {family: Counter() for family in MAIN_FAMILIES}
    for item_id, interaction_count in item_count_map.items():
        interaction_count = int(interaction_count)
        if interaction_count <= 0:
            continue
        family_map = profile_item_family_labels_map.get(str(item_id), {})
        for family in MAIN_FAMILIES:
            for label in family_map.get(family, []):
                normalized_label = normalize_text(label)
                if normalized_label:
                    counters[family][normalized_label] += interaction_count
    return counters


def build_brand_unique_item_count_map(item_ids):
    brand_items = defaultdict(set)
    for item_id in item_ids:
        for brand in profile_item_family_labels_map.get(str(item_id), {}).get("brand", []):
            normalized_brand = normalize_text(brand)
            if normalized_brand:
                brand_items[normalized_brand].add(str(item_id))
    return {brand: len(items) for brand, items in brand_items.items()}


def counter_to_prob_dict(counter_obj: Counter):
    total = float(sum(counter_obj.values()))
    if total <= 0.0:
        return {}
    return {str(key): float(value / total) for key, value in counter_obj.items()}


def counter_to_count_dict(counter_obj: Counter):
    return {str(key): int(value) for key, value in counter_obj.items()}


user_profile_rows = []
user_profile_map = {}
user_groups = prior_user_events.groupby("user_id", sort=False)

for user_id, user_history in tqdm(user_groups, total=user_groups.ngroups, desc="Build All Prior profiles"):
    item_count_map = user_history["item_id"].value_counts().to_dict()
    counters = build_family_counters_from_item_count_map(item_count_map)
    brand_unique_item_count = build_brand_unique_item_count_map(item_count_map)

    profile_history_review_n = int(len(user_history))
    profile_history_item_n = int(len(item_count_map))
    family_pref = {}
    family_count_map = {}
    family_entropy_norm = {}
    family_top_share = {}

    for family in MAIN_FAMILIES:
        counter_obj = counters.get(family, Counter())
        family_pref[family] = counter_to_prob_dict(counter_obj)
        family_count_map[family] = counter_to_count_dict(counter_obj)
        stats = normalized_entropy_from_counter(counter_obj, family_vocab_sizes.get(family, 1))
        family_entropy_norm[family] = stats["entropy_norm"]
        total_count = float(sum(counter_obj.values()))
        family_top_share[family] = (
            float(max(counter_obj.values()) / total_count) if total_count > 0 else np.nan
        )

    brand_counter = counters.get("brand", Counter())
    brand_total = int(sum(brand_counter.values()))
    dominant_brands = sorted([
        brand for brand, count in brand_counter.items()
        if brand_total > 0 and count == max(brand_counter.values())
    ])
    user_prior_unique_brand_count = int(len(brand_counter))
    user_prior_brand_entropy_norm = family_entropy_norm.get("brand")
    if pd.isna(user_prior_brand_entropy_norm):
        user_prior_brand_entropy_norm = 0.0

    profile_regime = classify_regime_from_prior_review_n(profile_history_review_n)
    user_entropy_norm_mean = aggregate_entropy_norm(
        {family: family_entropy_norm.get(family) for family in NON_BRAND_FAMILIES if family != "concern"}
    )
    user_top_share_mean = aggregate_entropy_norm(
        {family: family_top_share.get(family) for family in NON_BRAND_FAMILIES if family != "concern"},
        default=np.nan,
    )

    row = {
        "user_id": str(user_id),
        "profile_history_review_n": profile_history_review_n,
        "profile_history_item_n": profile_history_item_n,
        "profile_regime": profile_regime,
        "user_entropy_norm_mean": float(user_entropy_norm_mean) if pd.notna(user_entropy_norm_mean) else np.nan,
        "user_top_share_mean": float(user_top_share_mean) if pd.notna(user_top_share_mean) else np.nan,
        "user_prior_unique_brand_count": user_prior_unique_brand_count,
        "user_prior_brand_entropy_norm": float(user_prior_brand_entropy_norm),
        "family_pref_json": json.dumps(make_jsonable_dict(family_pref), ensure_ascii=False),
        "family_count_json": json.dumps(make_jsonable_dict(family_count_map), ensure_ascii=False),
        "family_entropy_norm_json": json.dumps(make_jsonable_dict(family_entropy_norm), ensure_ascii=False),
        "family_top_share_json": json.dumps(make_jsonable_dict(family_top_share), ensure_ascii=False),
    }
    for family in MAIN_FAMILIES:
        row[f"count__{family}"] = int(sum(counters.get(family, Counter()).values()))
        row[f"entropy__{family}"] = (
            float(family_entropy_norm.get(family)) if pd.notna(family_entropy_norm.get(family)) else np.nan
        )
        row[f"top_share__{family}"] = (
            float(family_top_share.get(family)) if pd.notna(family_top_share.get(family)) else np.nan
        )

    user_profile_rows.append(row)
    user_profile_map[str(user_id)] = {
        "profile_history_review_n": profile_history_review_n,
        "profile_history_item_n": profile_history_item_n,
        "profile_regime": profile_regime,
        "user_entropy_norm_mean": row["user_entropy_norm_mean"],
        "user_top_share_mean": row["user_top_share_mean"],
        "user_prior_unique_brand_count": user_prior_unique_brand_count,
        "user_prior_brand_entropy_norm": float(user_prior_brand_entropy_norm),
        "family_pref": family_pref,
        "family_count": family_count_map,
        "family_entropy_norm": family_entropy_norm,
        "family_top_share": family_top_share,
        "brand_unique_item_count": brand_unique_item_count,
        "dominant_brands": dominant_brands,
    }

user_profiles_df = pd.DataFrame(user_profile_rows)
user_profiles_df = apply_regime_order(user_profiles_df, "profile_regime")
print("All Prior user profiles:", len(user_profiles_df))
display(user_profiles_df.head(5))


Build All Prior profiles:   0%|          | 0/1716 [00:00<?, ?it/s]

All Prior user profiles: 1716


,user_id,profile_history_review_n,profile_history_item_n,profile_regime,user_entropy_norm_mean,user_top_share_mean,user_prior_unique_brand_count,user_prior_brand_entropy_norm,family_pref_json,family_count_json,family_entropy_norm_json,family_top_share_json,count__brand,entropy__brand,top_share__brand,count__product_type,entropy__product_type,top_share__product_type,count__form,entropy__form,top_share__form,count__ingredient,entropy__ingredient,top_share__ingredient,count__concern,entropy__concern,top_share__concern,count__claim,entropy__claim,top_share__claim,count__skin_type,entropy__skin_type,top_share__skin_type,count__scent,entropy__scent,top_share__scent
0,AHZRYOSRZZBTVSAKGMQ6ALKQDG7Q,1,1,weak,0.319923,0.333333,1,0.000000,"{""brand"": {""glam up"": 1.0}, ""product_type"": {""...","{""brand"": {""glam up"": 1}, ""product_type"": {""se...","{""brand"": 0.0, ""product_type"": 0.3199232330365...","{""brand"": 1.0, ""product_type"": 0.3333333333333...",1,0.000000,1.000000,3,0.319923,0.333333,0,NaN,NaN,0,NaN,NaN,5,0.202361,0.200000,0,NaN,NaN,0,NaN,NaN,0,NaN,NaN
1,AH4IBPYL72DEZLCDWQSUNLJR5ROA,5,5,moderate,0.295812,0.602922,3,0.097064,"{""brand"": {""l or al paris"": 0.6, ""l oreal pari...","{""brand"": {""l or al paris"": 3, ""l oreal paris""...","{""brand"": 0.09706448735342894, ""product_type"":...","{""brand"": 0.6, ""product_type"": 0.1818181818181...",5,0.097064,0.600000,11,0.624883,0.181818,4,0.089836,0.750000,5,0.161926,0.40,17,0.252342,0.176471,1,0.000000,1.000000,7,0.898227,0.285714,2,0.000000,1.00
2,AH5HVOBS6PRUUVO2EUNF7VRJT73A,3,3,weak,0.141306,0.708333,3,0.112217,"{""brand"": {""becca"": 0.3333333333333333, ""emine...","{""brand"": {""becca"": 1, ""eminence organic skin ...","{""brand"": 0.1122167153428083, ""product_type"": ...","{""brand"": 0.3333333333333333, ""product_type"": ...",3,0.112217,0.333333,6,0.454489,0.333333,2,0.110734,0.500000,1,0.000000,1.00,2,0.087152,0.500000,0,NaN,NaN,0,NaN,NaN,1,0.000000,1.00
3,AEMIK2NVC2UCNWSLJQ45ZLTTWLFQ,9,9,moderate,0.350292,0.300926,6,0.171294,"{""brand"": {""meta foret"": 0.2222222222222222, ""...","{""brand"": {""meta foret"": 2, ""urban skin rx"": 3...","{""brand"": 0.1712943334472384, ""product_type"": ...","{""brand"": 0.3333333333333333, ""product_type"": ...",9,0.171294,0.333333,20,0.715846,0.150000,9,0.277195,0.222222,4,0.168503,0.25,39,0.276482,0.205128,6,0.274600,0.333333,5,0.488342,0.600000,4,0.177266,0.25
4,AE7P72I6DQJA7DFRXQH4EUUL7ZZA,1,1,weak,0.140959,0.645833,1,0.000000,"{""brand"": {""sand sky"": 1.0}, ""product_type"": {...","{""brand"": {""sand sky"": 1}, ""product_type"": {""m...","{""brand"": 0.0, ""product_type"": 0.3199232330365...","{""brand"": 1.0, ""product_type"": 0.3333333333333...",1,0.000000,1.000000,3,0.319923,0.333333,1,0.000000,1.000000,0,NaN,NaN,7,0.244666,0.142857,4,0.243912,0.250000,0,NaN,NaN,1,0.000000,1.00


In [35]:
# ==== Configuration Consistency Guard ====
# Core configuration is defined once in Section 1. This cell validates the Notebook 09 candidate source.

if CATEGORY_ID != 'face':
    raise RuntimeError(f"Unexpected CATEGORY_ID: {CATEGORY_ID}")

expected_candidate_pool_dir = PROJECT_ROOT / "outputs" / "stage1_candidate_pools" / "by_method"
expected_candidate_path = expected_candidate_pool_dir / 'personalized_winner_top1000_face.parquet'

if CANDIDATE_POOL_DIR != expected_candidate_pool_dir:
    raise RuntimeError(f"Unexpected candidate pool directory: {CANDIDATE_POOL_DIR}")
if not CANDIDATE_POOL_DIR.is_dir():
    raise FileNotFoundError(f"Missing candidate pool directory: {CANDIDATE_POOL_DIR}")

manifest_personalized_winner_label = candidate_pool_manifest.get("selected_personalized_method_label")
manifest_personalized_winner_key = candidate_pool_manifest.get("selected_personalized_method_slug")
if (
    STAGE1_QUERY_METHOD != "C"
    or not RETRIEVAL_METHOD
    or not CANDIDATE_POOL_TYPE
):
    raise RuntimeError(
        f"Unexpected candidate pool setting: query={STAGE1_QUERY_METHOD}, "
        f"retrieval_method={RETRIEVAL_METHOD}, candidate_pool_type={CANDIDATE_POOL_TYPE}"
    )
if manifest_personalized_winner_label and (
    RETRIEVAL_METHOD != manifest_personalized_winner_label
    or CANDIDATE_POOL_TYPE != manifest_personalized_winner_label
):
    raise RuntimeError(
        f"Notebook 09 query-only winner mismatch: manifest={manifest_personalized_winner_label}, "
        f"retrieval_method={RETRIEVAL_METHOD}, candidate_pool_type={CANDIDATE_POOL_TYPE}"
    )
if manifest_personalized_winner_path and Path(manifest_personalized_winner_path) != expected_candidate_path:
    raise RuntimeError("Notebook 09 manifest query-only winner path mismatch.")

if Path(CANDIDATE_POOL_PATH) != expected_candidate_path:
    raise RuntimeError(f"Unexpected candidate pool path: {CANDIDATE_POOL_PATH}")
if candidate_pool_manifest.get("candidate_budget_policy") != "exact_k_all_methods":
    raise RuntimeError("Notebook 09 candidate pool manifest must use exact_k_all_methods.")
if candidate_pool_manifest.get("candidate_scores_preserved") is not True:
    raise RuntimeError("Notebook 09 candidate pool manifest must preserve candidate scores.")

required_paths = {
    "candidate_pool_path": CANDIDATE_POOL_PATH,
    "candidate_pool_manifest_path": CANDIDATE_POOL_MANIFEST_PATH,
    "candidate_pool_summary_path": CANDIDATE_POOL_SUMMARY_PATH,
    "query_cache_parquet": QUERY_CACHE_PARQUET,
    "target_case_metadata_parquet": TARGET_CASE_METADATA_PARQUET,
}
missing_required_paths = [
    name for name, path_obj in required_paths.items()
    if path_obj is None or not Path(path_obj).exists()
]
if missing_required_paths:
    raise FileNotFoundError(
        "Missing required files:\n"
        + "\n".join(f"- {name}: {required_paths[name]}" for name in missing_required_paths)
    )

if REGIME_ORDER != ['cold', 'weak', 'moderate', 'strong']:
    raise RuntimeError(f"Unexpected REGIME_ORDER: {REGIME_ORDER}")
if set(REGIME_PERSONALIZATION_WEIGHT) != set(REGIME_ORDER):
    raise RuntimeError(
        f"REGIME_PERSONALIZATION_WEIGHT keys do not match REGIME_ORDER: {REGIME_PERSONALIZATION_WEIGHT}"
    )
if len({float(value) for value in REGIME_PERSONALIZATION_WEIGHT.values()}) != 1:
    raise RuntimeError(
        "LightGBM reranker should use uniform manual regime weights; regime-specific effects are learned from features."
    )

print("Configuration guard passed.")
print("CANDIDATE_POOL_PATH:", CANDIDATE_POOL_PATH)
print("QUERY_CACHE_PARQUET:", QUERY_CACHE_PARQUET)
print("TARGET_CASE_METADATA_PARQUET:", TARGET_CASE_METADATA_PARQUET)
print("OUT_DIR:", OUT_DIR)

Configuration guard passed.
CANDIDATE_POOL_PATH: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage1_candidate_pools/by_method/personalized_winner_top1000_face.parquet
QUERY_CACHE_PARQUET: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/query_cache/face_queries.parquet
TARGET_CASE_METADATA_PARQUET: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/user_sampling/face_target_case_metadata.parquet
OUT_DIR: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full


In [36]:
# ==== Build User Profiles and Affinity Tables — Part 3 ====
# Add explicit cold profiles without fabricating preference evidence.

eval_user_ids = set(eval_query_users_df["user_id"].fillna("").astype(str))
profile_user_ids = set(user_profiles_df["user_id"].fillna("").astype(str)) if len(user_profiles_df) else set()
missing_eval_users = sorted(user_id for user_id in eval_user_ids - profile_user_ids if user_id)

if missing_eval_users:
    extra_rows = []
    for user_id in missing_eval_users:
        extra_rows.append({
            "user_id": str(user_id),
            "profile_history_review_n": 0,
            "profile_history_item_n": 0,
            "profile_regime": "cold",
            "user_entropy_norm_mean": 0.0,
            "user_top_share_mean": 0.0,
            "user_prior_unique_brand_count": 0,
            "user_prior_brand_entropy_norm": 0.0,
            "family_pref_json": json.dumps({}, ensure_ascii=False),
            "family_count_json": json.dumps({}, ensure_ascii=False),
            "family_entropy_norm_json": json.dumps({}, ensure_ascii=False),
            "family_top_share_json": json.dumps({}, ensure_ascii=False),
            **{f"count__{family}": 0 for family in MAIN_FAMILIES},
            **{f"entropy__{family}": np.nan for family in MAIN_FAMILIES},
            **{f"top_share__{family}": np.nan for family in MAIN_FAMILIES},
        })
        user_profile_map[str(user_id)] = {
            "profile_history_review_n": 0,
            "profile_history_item_n": 0,
            "profile_regime": "cold",
            "user_entropy_norm_mean": 0.0,
            "user_top_share_mean": 0.0,
            "user_prior_unique_brand_count": 0,
            "user_prior_brand_entropy_norm": 0.0,
            "family_pref": {},
            "family_count": {},
            "family_entropy_norm": {},
            "family_top_share": {},
            "brand_unique_item_count": {},
            "dominant_brands": [],
        }
    user_profiles_df = pd.concat([user_profiles_df, pd.DataFrame(extra_rows)], ignore_index=True)
    user_profiles_df = apply_regime_order(user_profiles_df, "profile_regime")

remaining_missing_users = set(eval_query_users_df["user_id"].astype(str)) - set(user_profiles_df["user_id"].astype(str))
if remaining_missing_users:
    raise RuntimeError(f"Missing evaluation user profiles: {len(remaining_missing_users)}")
print("Cold profiles added:", len(missing_eval_users))


Cold profiles added: 572


## 7. Build Candidate-Level LightGBM Feature Frame

In [37]:
# ==== Build Candidate-Level LightGBM Feature Frame ====
with runtime_step("feature_preparation", pool_depth=RUNTIME_REPORT_POOL_DEPTH, runtime_measurement_type="wall_clock_step", derived_from_existing_runtime=False):
    query_meta_merge_cols = [
        "query_id", "case_id", "timestamp_ms", "target_timestamp_ms",
        "prior_review_n", "prior_item_n", "user_total_reviews",
        "query_token_len", "removed_token_count",
    ]
    query_meta_merge_cols = [column for column in query_meta_merge_cols if column in query_meta_df.columns]
    query_meta_for_merge_df = query_meta_df[query_meta_merge_cols].drop_duplicates("query_id")
    query_meta_for_merge_df["query_id"] = query_meta_for_merge_df["query_id"].astype(str)

    baseline_keep_cols = [
        "benchmark_query_set", "benchmark_query_set_raw", "split_strategy",
        "case_id", "query_id", "user_id", "regime", "sampling_bracket",
        "gt_item_id", "query_text", "candidate_pool_type", "retrieval_method",
        "retrieval_method_label", "candidate_rank", "candidate_item_id",
        "is_gt", "candidate_score", "category_key",
    ]
    baseline_keep_cols = [column for column in baseline_keep_cols if column in candidate_df.columns]
    baseline_candidates_work = (
        candidate_df[baseline_keep_cols]
        .sort_values(["query_id", "candidate_rank", "candidate_item_id"], kind="mergesort")
        .reset_index(drop=True)
    )

    candidate_work = baseline_candidates_work.merge(
        query_meta_for_merge_df,
        on="query_id",
        how="left",
        validate="many_to_one",
        suffixes=("", "_meta"),
    )
    if "case_id_meta" in candidate_work.columns:
        candidate_work["case_id"] = candidate_work["case_id"].where(
            candidate_work["case_id"].astype(str).ne(""),
            candidate_work["case_id_meta"].fillna("").astype(str),
        )
        candidate_work = candidate_work.drop(columns=["case_id_meta"])

    profile_merge_cols = [
        "user_id", "profile_history_review_n", "profile_history_item_n",
        "profile_regime", "user_entropy_norm_mean", "user_top_share_mean",
        "user_prior_unique_brand_count", "user_prior_brand_entropy_norm",
    ] + [
        column for column in user_profiles_df.columns
        if column.startswith("count__") or column.startswith("entropy__") or column.startswith("top_share__")
    ]
    candidate_work = candidate_work.merge(
        user_profiles_df[profile_merge_cols],
        on="user_id",
        how="left",
        validate="many_to_one",
    )

    candidate_work["candidate_brand"] = (
        candidate_work["candidate_item_id"].astype(str).map(item_brand_map).fillna("").astype(str)
    )
    candidate_work["candidate_brand_present"] = (
        candidate_work["candidate_brand"].str.strip().ne("").astype(np.float32)
    )
    candidate_work["is_gt"] = (
        candidate_work["candidate_item_id"].astype(str).eq(candidate_work["gt_item_id"].astype(str))
    ).astype(np.int8)
    candidate_work["candidate_score"] = pd.to_numeric(
        candidate_work["candidate_score"], errors="coerce"
    ).fillna(0.0).astype(np.float32)
    for column in ["timestamp_ms", "target_timestamp_ms"]:
        candidate_work[column] = pd.to_numeric(
            candidate_work[column], errors="coerce"
        ).fillna(0).astype(np.int64)
    for column in ["prior_review_n", "prior_item_n", "user_total_reviews", "query_token_len", "removed_token_count"]:
        candidate_work[column] = pd.to_numeric(
            candidate_work[column], errors="coerce"
        ).fillna(0).astype(np.int32)

    candidate_identity_before_features = candidate_work[["query_id", "candidate_item_id"]].copy()


In [38]:
# ==== Build Candidate-Level LightGBM Feature Frame — Part 2 ====
with runtime_step("feature_preparation", pool_depth=RUNTIME_REPORT_POOL_DEPTH, runtime_measurement_type="wall_clock_step", derived_from_existing_runtime=False):
    for column in [
        "profile_history_review_n", "profile_history_item_n",
        "user_prior_unique_brand_count", "user_prior_brand_entropy_norm",
    ]:
        candidate_work[column] = pd.to_numeric(
            candidate_work[column], errors="coerce"
        ).fillna(0)

    candidate_work["profile_history_review_n"] = candidate_work["profile_history_review_n"].astype(np.int32)
    candidate_work["profile_history_item_n"] = candidate_work["profile_history_item_n"].astype(np.int32)
    candidate_work["user_prior_unique_brand_count"] = candidate_work["user_prior_unique_brand_count"].astype(np.float32)
    candidate_work["user_prior_brand_entropy_norm"] = candidate_work["user_prior_brand_entropy_norm"].astype(np.float32)
    candidate_work["user_entropy_norm_mean"] = pd.to_numeric(
        candidate_work["user_entropy_norm_mean"], errors="coerce"
    ).fillna(0.0).astype(np.float32)
    candidate_work["user_top_share_mean"] = pd.to_numeric(
        candidate_work["user_top_share_mean"], errors="coerce"
    ).fillna(0.0).astype(np.float32)

    required_candidate_cols = [
        "query_text", "user_id", "timestamp_ms", "target_timestamp_ms",
        "gt_item_id", "regime", "sampling_bracket", "prior_review_n",
        "profile_regime", "candidate_brand",
    ]
    missing_candidate_cols = [column for column in required_candidate_cols if column not in candidate_work.columns]
    if missing_candidate_cols:
        raise RuntimeError(f"candidate_work is missing required columns: {missing_candidate_cols}")
    if candidate_work["user_id"].isna().any() or candidate_work["user_id"].astype(str).eq("").any():
        raise RuntimeError("candidate_work contains missing user_id values.")
    if candidate_work["profile_regime"].isna().any():
        raise RuntimeError("candidate_work contains missing user profiles.")

    candidate_work["candidate_rank"] = pd.to_numeric(
        candidate_work["candidate_rank"], errors="coerce"
    ).fillna(0).astype(np.int32)
    candidate_work["candidate_rank_log1p"] = np.log1p(
        candidate_work["candidate_rank"].astype(np.float32)
    ).astype(np.float32)
    candidate_work["candidate_rank_inv"] = (
        1.0 / candidate_work["candidate_rank"].astype(np.float32).clip(lower=1.0)
    ).astype(np.float32)
    candidate_work["candidate_in_max_pool"] = np.float32(1.0)

    query_text_by_id = candidate_work[["query_id", "query_text"]].drop_duplicates("query_id")
    query_token_map = {
        str(query_id): set(tokenize(str(query_text) if pd.notna(query_text) else ""))
        for query_id, query_text in zip(query_text_by_id["query_id"], query_text_by_id["query_text"])
    }
    query_token_len_map = {query_id: len(tokens) for query_id, tokens in query_token_map.items()}
    unique_item_ids = candidate_work["candidate_item_id"].astype(str).drop_duplicates().tolist()
    item_title_token_map = {
        item_id: set(tokenize(item_title_map.get(item_id, "")))
        for item_id in unique_item_ids
    }

    feature_cols_loop = [
        "item_title_token_overlap", "query_item_structured_match",
        "user_item_affinity", "user_item_seen_strength",
    ]
    for family in NON_BRAND_FAMILIES:
        feature_cols_loop.extend([
            f"qmatch__{family}", f"uaff__{family}", f"useen__{family}",
            f"ucountlog__{family}", f"item_label_n__{family}",
        ])
    if USER_BRAND_AFFINITY_ENABLED:
        feature_cols_loop.extend([
            "candidate_brand_seen_in_prior",
            "candidate_brand_prior_interaction_count",
            "candidate_brand_prior_unique_item_count",
            "candidate_brand_prior_share",
            "candidate_brand_is_dominant_prior_brand",
        ])

    feature_data = {column: [] for column in feature_cols_loop}
    item_family_token_map_local = item_family_token_map
    item_family_labels_map_local = item_family_labels_map
    profile_item_family_labels_map_local = profile_item_family_labels_map
    item_label_count_map_local = item_label_count_map
    user_profile_map_local = user_profile_map
    family_weights_local = FAMILY_STRENGTH_WEIGHTS


In [39]:
# ==== Build Candidate-Level LightGBM Feature Frame — Part 3 ====
with runtime_step("feature_preparation", pool_depth=RUNTIME_REPORT_POOL_DEPTH, runtime_measurement_type="wall_clock_step", derived_from_existing_runtime=False):
    for row in tqdm(
        candidate_work[["candidate_item_id", "user_id", "query_id"]].itertuples(index=False),
        total=len(candidate_work),
        desc="Build candidate features",
    ):
        item_id = str(row.candidate_item_id)
        user_id = str(row.user_id)
        query_id = str(row.query_id)
        query_tokens = query_token_map.get(query_id, set())
        user_profile = user_profile_map_local.get(user_id, {}) if USER_BRAND_AFFINITY_ENABLED else {}

        feature_data["item_title_token_overlap"].append(float(jaccard_set_similarity(
            query_tokens,
            item_title_token_map.get(item_id, set()),
        )))
        query_match_by_family = {}
        user_affinity_by_family = {}
        user_count_by_family = {}
        item_family_tokens = item_family_token_map_local.get(item_id, {})
        item_family_labels = item_family_labels_map_local.get(item_id, {})
        item_label_counts = item_label_count_map_local.get(item_id, {})

        for family in NON_BRAND_FAMILIES:
            item_tokens = item_family_tokens.get(family, set())
            item_labels = item_family_labels.get(family, [])
            query_match = jaccard_set_similarity(query_tokens, item_tokens)
            if family == "concern":
                query_match *= CONCERN_FEATURE_SCALE
            query_match_by_family[family] = query_match

            preference_map = user_profile.get("family_pref", {}).get(family, {})
            count_map = user_profile.get("family_count", {}).get(family, {})
            item_label_norms = [normalize_text(value) for value in item_labels if normalize_text(value)]
            user_probability = max([float(preference_map.get(label, 0.0)) for label in item_label_norms] + [0.0])
            user_count = max([int(count_map.get(label, 0)) for label in item_label_norms] + [0])
            user_seen = float(user_count > 0)
            if family == "concern":
                user_probability *= CONCERN_FEATURE_SCALE
                user_seen *= CONCERN_FEATURE_SCALE

            user_affinity_by_family[family] = user_probability
            user_count_by_family[family] = user_count
            feature_data[f"qmatch__{family}"].append(float(query_match))
            feature_data[f"uaff__{family}"].append(float(user_probability))
            feature_data[f"useen__{family}"].append(float(user_seen))
            feature_data[f"ucountlog__{family}"].append(float(np.log1p(user_count)))
            feature_data[f"item_label_n__{family}"].append(float(item_label_counts.get(family, 0)))

        if USER_BRAND_AFFINITY_ENABLED:
            candidate_brands = [
                normalize_text(value)
                for value in profile_item_family_labels_map_local.get(item_id, {}).get("brand", [])
                if normalize_text(value)
            ]
            brand_preference_map = user_profile.get("family_pref", {}).get("brand", {})
            brand_count_map = user_profile.get("family_count", {}).get("brand", {})
            brand_unique_item_count_map = user_profile.get("brand_unique_item_count", {})
            dominant_brands = set(user_profile.get("dominant_brands", []))
            brand_count = max([int(brand_count_map.get(brand, 0)) for brand in candidate_brands] + [0])
            brand_unique_item_count = max([
                int(brand_unique_item_count_map.get(brand, 0)) for brand in candidate_brands
            ] + [0])
            brand_share = max([
                float(brand_preference_map.get(brand, 0.0)) for brand in candidate_brands
            ] + [0.0])
            brand_seen = float(brand_count > 0)
            dominant_brand_match = float(bool(set(candidate_brands) & dominant_brands))

            feature_data["candidate_brand_seen_in_prior"].append(brand_seen)
            feature_data["candidate_brand_prior_interaction_count"].append(float(brand_count))
            feature_data["candidate_brand_prior_unique_item_count"].append(float(brand_unique_item_count))
            feature_data["candidate_brand_prior_share"].append(float(brand_share))
            feature_data["candidate_brand_is_dominant_prior_brand"].append(dominant_brand_match)
            user_affinity_by_family["brand"] = brand_share * BRAND_FEATURE_SCALE
            user_count_by_family["brand"] = brand_count

        feature_data["query_item_structured_match"].append(float(
            weighted_mean_from_feature_map(query_match_by_family, family_weights_local)
        ))
        feature_data["user_item_affinity"].append(float(
            weighted_mean_from_feature_map(user_affinity_by_family, family_weights_local)
        ))
        feature_data["user_item_seen_strength"].append(float(
            weighted_mean_from_feature_map(
                {family: float(np.log1p(count)) for family, count in user_count_by_family.items()},
                family_weights_local,
            )
        ))

    item_feature_df = pd.DataFrame(feature_data, index=candidate_work.index).astype(np.float32)
    candidate_work = pd.concat(
        [candidate_work.reset_index(drop=True), item_feature_df.reset_index(drop=True)],
        axis=1,
    )

    duplicated_feature_columns = candidate_work.columns[candidate_work.columns.duplicated()].tolist()
    if duplicated_feature_columns:
        raise RuntimeError(f"Feature construction created duplicated columns: {duplicated_feature_columns}")

    if "query_token_len" not in candidate_work.columns:
        candidate_work["query_token_len"] = candidate_work["query_id"].astype(str).map(query_token_len_map).fillna(0).astype(np.int32)
    else:
        candidate_work["query_token_len"] = candidate_work["query_token_len"].fillna(
            candidate_work["query_id"].astype(str).map(query_token_len_map)
        ).fillna(0).astype(np.int32)

    candidate_work["query_token_len_log1p"] = np.log1p(candidate_work["query_token_len"].astype(np.float32)).astype(np.float32)
    candidate_work["removed_token_count_log1p"] = np.log1p(candidate_work["removed_token_count"].astype(np.float32)).astype(np.float32)
    candidate_work["prior_review_n_log1p"] = np.log1p(candidate_work["prior_review_n"].astype(np.float32)).astype(np.float32)
    candidate_work["prior_item_n_log1p"] = np.log1p(candidate_work["prior_item_n"].astype(np.float32)).astype(np.float32)
    candidate_work["user_total_reviews_log1p"] = np.log1p(candidate_work["user_total_reviews"].astype(np.float32)).astype(np.float32)
    candidate_work["profile_history_review_log1p"] = np.log1p(candidate_work["profile_history_review_n"].astype(np.float32)).astype(np.float32)
    candidate_work["profile_history_item_log1p"] = np.log1p(candidate_work["profile_history_item_n"].astype(np.float32)).astype(np.float32)
    candidate_work["profile_history_gap_vs_cache"] = (
        candidate_work["profile_history_review_n"].astype(np.float32)
        - candidate_work["prior_review_n"].astype(np.float32)
    ).astype(np.float32)
    for regime in REGIME_ORDER:
        candidate_work[f"profile_regime_{regime}"] = (
            candidate_work["profile_regime"].astype(str).eq(regime).astype(np.float32)
        )
        candidate_work[f"regime_{regime}"] = (
            candidate_work["regime"].astype(str).eq(regime).astype(np.float32)
        )
    candidate_work["regime_personalization_weight"] = (
        candidate_work["regime"].astype(str).map(REGIME_PERSONALIZATION_WEIGHT).fillna(1.0).astype(np.float32)
    )

    optional_feature_col_map = {}
    available_optional_features = []


Build candidate features:   0%|          | 0/2288000 [00:00<?, ?it/s]

In [40]:
# ==== Leakage-Safe Temporal Recency Features and Feature Contract ====
def _as_float32(values):
    return np.asarray(values, dtype=np.float32)


def _days_between_ms(later_ms, earlier_ms):
    return (np.asarray(later_ms, dtype=np.float64) - np.asarray(earlier_ms, dtype=np.float64)) / float(MS_PER_DAY)


def _first_existing_col(columns, candidates):
    colset = set(columns)
    for candidate in candidates:
        if candidate in colset:
            return candidate
    return None


def _safe_normalize_text_value(value):
    if "normalize_text" in globals():
        return normalize_text(value)
    return re.sub(r"\s+", " ", str(value).strip().lower())


def build_item_timestamp_map(item_time_df: pd.DataFrame) -> dict:
    if item_time_df.empty:
        return {}
    work = item_time_df[["parent_asin", "review_timestamp_ms"]].copy()
    work["parent_asin"] = work["parent_asin"].fillna("").astype(str).str.strip()
    work["review_timestamp_ms"] = pd.to_numeric(work["review_timestamp_ms"], errors="coerce")
    work = work.dropna(subset=["review_timestamp_ms"])
    work = work[work["parent_asin"].ne("")]
    work["review_timestamp_ms"] = work["review_timestamp_ms"].astype(np.int64)
    out = {}
    for item_id, group in work.groupby("parent_asin", sort=False):
        ts = np.sort(group["review_timestamp_ms"].to_numpy(dtype=np.int64))
        if len(ts):
            out[str(item_id)] = ts
    return out


def add_item_temporal_features(df: pd.DataFrame, item_ts_map: dict) -> pd.DataFrame:
    if not USE_TEMPORAL_RECENCY_FEATURES:
        for col in COMMON_TEMPORAL_ITEM_FEATURE_COLS:
            df[col] = np.float32(0.0)
        return df

    n = len(df)
    prequery_count = np.zeros(n, dtype=np.float32)
    last_gap_days = np.full(n, RECENCY_FEATURE_FILL_DAYS, dtype=np.float32)
    first_age_days = np.full(n, RECENCY_FEATURE_FILL_DAYS, dtype=np.float32)
    recent_counts = {days: np.zeros(n, dtype=np.float32) for days in TEMPORAL_WINDOWS_DAYS}

    item_values = df["candidate_item_id"].astype(str).to_numpy()
    query_ts_col = "target_timestamp_ms" if "target_timestamp_ms" in df.columns else "timestamp_ms"
    query_ts_values = pd.to_numeric(df[query_ts_col], errors="coerce").fillna(0).astype(np.int64).to_numpy()

    row_by_item = defaultdict(list)
    for idx, item_id in enumerate(item_values):
        row_by_item[str(item_id)].append(idx)

    for item_id, row_indices in row_by_item.items():
        ts = item_ts_map.get(str(item_id))
        if ts is None or len(ts) == 0:
            continue
        idx_arr = np.asarray(row_indices, dtype=np.int64)
        qts = query_ts_values[idx_arr]
        valid_q = qts > 0
        if not valid_q.any():
            continue
        idx_valid = idx_arr[valid_q]
        qts_valid = qts[valid_q]

        right = np.searchsorted(ts, qts_valid, side="left")
        prequery_count[idx_valid] = right.astype(np.float32)

        has_prior = right > 0
        if has_prior.any():
            idx_has = idx_valid[has_prior]
            qts_has = qts_valid[has_prior]
            right_has = right[has_prior]
            last_ts = ts[right_has - 1]
            first_ts = ts[0]
            last_gap_days[idx_has] = np.maximum(0.0, _days_between_ms(qts_has, last_ts)).astype(np.float32)
            first_age_days[idx_has] = np.maximum(0.0, _days_between_ms(qts_has, np.full_like(qts_has, first_ts))).astype(np.float32)

        for days in TEMPORAL_WINDOWS_DAYS:
            left = np.searchsorted(ts, qts_valid - int(days * MS_PER_DAY), side="left")
            recent_counts[days][idx_valid] = np.maximum(0, right - left).astype(np.float32)

    df["item_prequery_review_count"] = prequery_count
    df["item_last_review_gap_days"] = last_gap_days
    df["item_first_review_age_days"] = first_age_days

    denom = np.maximum(prequery_count, 1.0)
    for days in TEMPORAL_WINDOWS_DAYS:
        count_col = f"item_recent_review_count_{days}d"
        share_col = f"item_recent_review_share_{days}d"
        velocity_col = f"item_review_velocity_{days}d"
        df[count_col] = recent_counts[days].astype(np.float32)
        df[share_col] = (recent_counts[days] / denom).astype(np.float32)
        df[velocity_col] = (recent_counts[days] / np.float32(days)).astype(np.float32)

    return df


def build_item_family_key_frame() -> pd.DataFrame:
    rows = []
    source_map = globals().get("profile_item_family_labels_map", {})
    for item_id, family_map in source_map.items():
        for family in FAMILY_RECENCY_FEATURE_FAMILIES:
            labels = family_map.get(family, []) if isinstance(family_map, dict) else []
            for label in labels:
                label_norm = _safe_normalize_text_value(label)
                if label_norm:
                    rows.append({"item_id": str(item_id), "family": str(family), "label": str(label_norm)})
    if not rows:
        return pd.DataFrame(columns=["item_id", "family", "label"])
    return pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)


def standardize_prior_temporal_source(prior_source: pd.DataFrame, query_source: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    required_cols = ["case_id", "user_id", "item_id", "timestamp_ms", "target_timestamp_ms"]
    missing_cols = [column for column in required_cols if column not in prior_source.columns]
    if missing_cols:
        raise RuntimeError(f"Standardized All Prior source is missing columns: {missing_cols}")

    prior = prior_source[required_cols].copy()
    prior = prior.rename(columns={"item_id": "prior_item_id", "timestamp_ms": "prior_timestamp_ms"})
    prior["case_id"] = prior["case_id"].astype(str).str.strip()
    prior["user_id"] = prior["user_id"].fillna("").astype(str).str.strip()
    prior["prior_item_id"] = prior["prior_item_id"].fillna("").astype(str).str.strip()
    prior["prior_timestamp_ms"] = pd.to_numeric(prior["prior_timestamp_ms"], errors="coerce")
    prior["target_timestamp_ms"] = pd.to_numeric(prior["target_timestamp_ms"], errors="coerce")
    prior = prior.dropna(subset=["prior_timestamp_ms", "target_timestamp_ms"])
    prior["prior_timestamp_ms"] = prior["prior_timestamp_ms"].astype(np.int64)
    prior["target_timestamp_ms"] = prior["target_timestamp_ms"].astype(np.int64)
    prior = prior.loc[
        prior["case_id"].ne("") & prior["user_id"].ne("") & prior["prior_item_id"].ne("")
    ].copy()
    future_or_equal_n = int(prior["prior_timestamp_ms"].ge(prior["target_timestamp_ms"]).sum())
    if future_or_equal_n:
        raise RuntimeError(f"All Prior temporal contract failed: {future_or_equal_n} rows are not strictly pre-target.")
    return prior.reset_index(drop=True), 0

def build_user_temporal_maps(prior_source: pd.DataFrame, query_source: pd.DataFrame) -> dict:
    prior, future_or_equal_excluded_n = standardize_prior_temporal_source(prior_source, query_source)
    if prior.empty:
        return {
            "user_last_ts": {},
            "user_item_last_ts": {},
            "user_item_recent_count_180d": {},
            "user_facet_last_ts": {},
            "user_facet_recent_count_180d": {},
            "prior_future_or_equal_excluded_n": int(future_or_equal_excluded_n),
            "strict_prior_row_count": 0,
        }

    key_case_user = ["case_id", "user_id"]
    user_last_ts = prior.groupby(key_case_user, sort=False)["prior_timestamp_ms"].max().to_dict()
    user_item_last_ts = prior.groupby(["case_id", "user_id", "prior_item_id"], sort=False)["prior_timestamp_ms"].max().to_dict()

    prior["event_age_days"] = (
        (prior["target_timestamp_ms"].astype(np.float64) - prior["prior_timestamp_ms"].astype(np.float64))
        / float(MS_PER_DAY)
    )
    user_item_recent_count_180d = (
        prior.loc[prior["event_age_days"].le(180.0)]
        .groupby(["case_id", "user_id", "prior_item_id"], sort=False)
        .size()
        .astype(int)
        .to_dict()
    )

    facet_key = build_item_family_key_frame()
    if facet_key.empty:
        user_facet_last_ts = {}
        user_facet_recent_count_180d = {}
    else:
        prior_facet = prior.merge(facet_key, left_on="prior_item_id", right_on="item_id", how="inner")
        if prior_facet.empty:
            user_facet_last_ts = {}
            user_facet_recent_count_180d = {}
        else:
            user_facet_last_ts = (
                prior_facet.groupby(["case_id", "user_id", "family", "label"], sort=False)["prior_timestamp_ms"]
                .max()
                .to_dict()
            )
            user_facet_recent_count_180d = (
                prior_facet.loc[prior_facet["event_age_days"].le(180.0)]
                .groupby(["case_id", "user_id", "family", "label"], sort=False)
                .size()
                .astype(int)
                .to_dict()
            )

    return {
        "user_last_ts": user_last_ts,
        "user_item_last_ts": user_item_last_ts,
        "user_item_recent_count_180d": user_item_recent_count_180d,
        "user_facet_last_ts": user_facet_last_ts,
        "user_facet_recent_count_180d": user_facet_recent_count_180d,
        "prior_future_or_equal_excluded_n": int(future_or_equal_excluded_n),
        "strict_prior_row_count": int(len(prior)),
    }


def add_user_temporal_features(df: pd.DataFrame, temporal_maps: dict) -> pd.DataFrame:
    if not USE_TEMPORAL_RECENCY_FEATURES:
        for column in COMMON_TEMPORAL_USER_FEATURE_COLS + COMMON_TEMPORAL_FAMILY_FEATURE_COLS + BRAND_ALL_PRIOR_TEMPORAL_FEATURE_COLS:
            df[column] = np.float32(0.0)
        return df

    user_last_ts_map = temporal_maps.get("user_last_ts", {})
    user_item_last_ts_map = temporal_maps.get("user_item_last_ts", {})
    user_item_recent_count_map = temporal_maps.get("user_item_recent_count_180d", {})
    user_facet_last_ts_map = temporal_maps.get("user_facet_last_ts", {})
    user_facet_recent_count_map = temporal_maps.get("user_facet_recent_count_180d", {})

    row_count = len(df)
    user_last_gap = np.full(row_count, RECENCY_FEATURE_FILL_DAYS, dtype=np.float32)
    user_item_recency = np.full(row_count, RECENCY_FEATURE_FILL_DAYS, dtype=np.float32)
    user_item_recent_count = np.zeros(row_count, dtype=np.float32)
    family_recency = {
        family: np.full(row_count, RECENCY_FEATURE_FILL_DAYS, dtype=np.float32)
        for family in FAMILY_RECENCY_FEATURE_FAMILIES
    }
    family_recent_count = {
        family: np.zeros(row_count, dtype=np.float32)
        for family in FAMILY_RECENCY_FEATURE_FAMILIES
    }
    brand_recency = np.full(row_count, RECENCY_FEATURE_FILL_DAYS, dtype=np.float32)
    brand_recent_count = np.zeros(row_count, dtype=np.float32)

    iter_cols = ["case_id", "user_id", "candidate_item_id", "target_timestamp_ms"]
    for index, row in enumerate(df[iter_cols].itertuples(index=False, name=None)):
        case_id, user_id, item_id = str(row[0]), str(row[1]), str(row[2])
        query_timestamp = int(row[3]) if pd.notna(row[3]) else 0
        if query_timestamp <= 0:
            continue

        user_key = (case_id, user_id)
        last_user_timestamp = user_last_ts_map.get(user_key)
        if last_user_timestamp is not None:
            user_last_gap[index] = max(
                0.0,
                float(query_timestamp - int(last_user_timestamp)) / float(MS_PER_DAY),
            )

        item_key = (case_id, user_id, item_id)
        last_item_timestamp = user_item_last_ts_map.get(item_key)
        if last_item_timestamp is not None:
            user_item_recency[index] = max(
                0.0,
                float(query_timestamp - int(last_item_timestamp)) / float(MS_PER_DAY),
            )
        user_item_recent_count[index] = float(user_item_recent_count_map.get(item_key, 0))

        item_family_labels = profile_item_family_labels_map.get(item_id, {})
        for family in FAMILY_RECENCY_FEATURE_FAMILIES:
            labels = [normalize_text(value) for value in item_family_labels.get(family, []) if normalize_text(value)]
            last_timestamps = [
                int(timestamp)
                for label in labels
                for timestamp in [user_facet_last_ts_map.get((case_id, user_id, family, label))]
                if timestamp is not None
            ]
            if last_timestamps:
                family_recency[family][index] = max(
                    0.0,
                    float(query_timestamp - max(last_timestamps)) / float(MS_PER_DAY),
                )
            family_recent_count[family][index] = float(sum(
                int(user_facet_recent_count_map.get((case_id, user_id, family, label), 0))
                for label in labels
            ))

        if USER_BRAND_AFFINITY_ENABLED:
            brand_labels = [normalize_text(value) for value in item_family_labels.get("brand", []) if normalize_text(value)]
            brand_last_timestamps = [
                int(timestamp)
                for label in brand_labels
                for timestamp in [user_facet_last_ts_map.get((case_id, user_id, "brand", label))]
                if timestamp is not None
            ]
            if brand_last_timestamps:
                brand_recency[index] = max(
                    0.0,
                    float(query_timestamp - max(brand_last_timestamps)) / float(MS_PER_DAY),
                )
            brand_recent_count[index] = float(sum(
                int(user_facet_recent_count_map.get((case_id, user_id, "brand", label), 0))
                for label in brand_labels
            ))

    df["user_last_interaction_gap_days"] = user_last_gap
    df["user_item_recency_days"] = user_item_recency
    df["user_item_recent_count_180d"] = user_item_recent_count
    for family in FAMILY_RECENCY_FEATURE_FAMILIES:
        df[f"user_{family}_recency_days"] = family_recency[family]
        df[f"user_{family}_recent_count_180d"] = family_recent_count[family]
    if USER_BRAND_AFFINITY_ENABLED:
        df["candidate_brand_last_interaction_age_days"] = brand_recency
        df["candidate_brand_recent_interaction_count_180d"] = brand_recent_count
    return df

def add_item_doc_overlap_feature(df: pd.DataFrame) -> pd.DataFrame:
    if "item_doc_token_overlap" in df.columns:
        df["item_doc_token_overlap"] = pd.to_numeric(df["item_doc_token_overlap"], errors="coerce").fillna(0.0).astype(np.float32)
        return df
    item_doc_text_col = None
    for col in ["bm25_text_v2", "sparse_text", "dense_text_v2", "dense_text", "canonical_retrieval_text"]:
        if "item_docs" in globals() and col in item_docs.columns:
            item_doc_text_col = col
            break
    if item_doc_text_col is None or "query_token_map" not in globals():
        df["item_doc_token_overlap"] = np.float32(0.0)
        return df

    docs_local = item_docs[["item_id", item_doc_text_col]].drop_duplicates("item_id").copy()
    doc_token_map = {
        str(item_id): set(tokenize(str(text) if pd.notna(text) else ""))
        for item_id, text in zip(docs_local["item_id"], docs_local[item_doc_text_col])
    }
    df["item_doc_token_overlap"] = [
        float(jaccard_set_similarity(query_token_map.get(str(qid), set()), doc_token_map.get(str(item_id), set())))
        for qid, item_id in df[["query_id", "candidate_item_id"]].itertuples(index=False)
    ]
    df["item_doc_token_overlap"] = pd.to_numeric(df["item_doc_token_overlap"], errors="coerce").fillna(0.0).astype(np.float32)
    return df


def ensure_common_regime_features(df: pd.DataFrame) -> pd.DataFrame:
    regime_source_col = "profile_regime" if "profile_regime" in df.columns else "regime"
    if regime_source_col not in df.columns:
        df["profile_regime"] = ""
        regime_source_col = "profile_regime"
    for regime in REGIME_ORDER:
        profile_col = f"profile_regime_{regime}"
        regime_col = f"regime_{regime}"
        values = (df[regime_source_col].astype(str) == regime).astype(np.float32)
        if profile_col not in df.columns:
            df[profile_col] = values
        if regime_col not in df.columns:
            df[regime_col] = values
    if "regime_personalization_weight" not in df.columns:
        weight_map = globals().get("REGIME_PERSONALIZATION_WEIGHT", globals().get("REGIME_PERSONALIZATION_WEIGHTS", {}))
        df["regime_personalization_weight"] = df["regime"].astype(str).map(weight_map).fillna(1.0).astype(np.float32)
    return df


def build_feature_leakage_qc(feature_cols: list, temporal_cols: list, temporal_maps: dict) -> pd.DataFrame:
    feature_set = set(feature_cols)
    forbidden_exact = sorted(feature_set & FORBIDDEN_MODEL_FEATURE_COLS)
    forbidden_prefix = sorted([
        c for c in feature_cols
        if any(str(c).startswith(prefix) for prefix in FORBIDDEN_MODEL_FEATURE_PREFIXES)
    ])
    raw_text_cols = sorted([
        c for c in feature_cols
        if any(term in str(c).lower() for term in ["query_text", "review_text", "review_body", "raw_text"])
    ])
    negative_temporal_cols = []
    for col in [c for c in temporal_cols if c in candidate_work.columns and c.endswith("_days")]:
        if pd.to_numeric(candidate_work[col], errors="coerce").dropna().lt(0).any():
            negative_temporal_cols.append(col)

    checks = [
        {
            "check_name": "raw_timestamp_columns_excluded",
            "passed": not bool({"timestamp_ms", "target_timestamp_ms"} & feature_set),
            "detail": json.dumps(sorted({"timestamp_ms", "target_timestamp_ms"} & feature_set)),
        },
        {
            "check_name": "target_identifier_columns_excluded",
            "passed": not bool(set(["case_id", "query_id", "user_id", "target_parent_asin", "candidate_parent_asin", "target_item_id", "candidate_item_id", "gt_item_id", "is_gt"]) & feature_set),
            "detail": json.dumps(sorted(set(["case_id", "query_id", "user_id", "target_parent_asin", "candidate_parent_asin", "target_item_id", "candidate_item_id", "gt_item_id", "is_gt"]) & feature_set)),
        },
        {
            "check_name": "label_columns_excluded",
            "passed": "label" not in feature_set,
            "detail": json.dumps(["label"] if "label" in feature_set else []),
        },
        {
            "check_name": "raw_text_columns_excluded",
            "passed": not bool(raw_text_cols),
            "detail": json.dumps(raw_text_cols),
        },
        {
            "check_name": "forbidden_prefix_columns_excluded",
            "passed": not bool(forbidden_prefix),
            "detail": json.dumps(forbidden_prefix),
        },
        {
            "check_name": "temporal_artifacts_loaded",
            "passed": bool(USE_TEMPORAL_RECENCY_FEATURES and len(item_review_time_index) > 0),
            "detail": json.dumps({"item_review_time_index_rows": int(len(item_review_time_index))}),
        },
        {
            "check_name": "recency_features_use_strict_prequery_rule",
            "passed": True,
            "detail": TEMPORAL_LEAKAGE_RULE,
        },
        {
            "check_name": "future_or_equal_prior_rows_excluded",
            "passed": True,
            "detail": json.dumps({"excluded_rows": int(temporal_maps.get("prior_future_or_equal_excluded_n", 0))}),
        },
        {
            "check_name": "negative_recency_days_absent",
            "passed": not bool(negative_temporal_cols),
            "detail": json.dumps(negative_temporal_cols),
        },
        {
            "check_name": "model_feature_contract_clean",
            "passed": not bool(forbidden_exact or forbidden_prefix or raw_text_cols or negative_temporal_cols),
            "detail": json.dumps({"forbidden_exact": forbidden_exact, "forbidden_prefix": forbidden_prefix, "raw_text": raw_text_cols}),
        },
    ]
    return pd.DataFrame(checks)


with runtime_step("feature_preparation", pool_depth=RUNTIME_REPORT_POOL_DEPTH, runtime_measurement_type="wall_clock_step", derived_from_existing_runtime=False):
    if "target_timestamp_ms" not in candidate_work.columns and "timestamp_ms" in candidate_work.columns:
        candidate_work["target_timestamp_ms"] = pd.to_numeric(candidate_work["timestamp_ms"], errors="coerce").fillna(0).astype(np.int64)
    if "target_timestamp_ms" in candidate_work.columns:
        candidate_work["target_timestamp_ms"] = pd.to_numeric(candidate_work["target_timestamp_ms"], errors="coerce").fillna(0).astype(np.int64)
        candidate_work["timestamp_ms"] = candidate_work["target_timestamp_ms"].astype(np.int64)

    item_timestamp_map = build_item_timestamp_map(item_review_time_index)
    prior_temporal_source = globals().get("prior_history", globals().get("prior_history_raw", pd.DataFrame()))
    user_temporal_maps = build_user_temporal_maps(prior_temporal_source, query_meta_df)

    candidate_work = add_item_doc_overlap_feature(candidate_work)
    candidate_work = add_item_temporal_features(candidate_work, item_timestamp_map)
    candidate_work = add_user_temporal_features(candidate_work, user_temporal_maps)
    candidate_work["user_item_seen_strength"] = (
        pd.to_numeric(candidate_work["user_item_recency_days"], errors="coerce")
        .lt(float(RECENCY_FEATURE_FILL_DAYS))
        .astype(np.float32)
    )

    exact_item_positive = candidate_work["is_gt"].astype(int).eq(1)
    target_seen_strength_n = int(
        pd.to_numeric(candidate_work.loc[exact_item_positive, "user_item_seen_strength"], errors="coerce")
        .fillna(0).gt(0).sum()
    )
    target_recent_count_n = int(
        pd.to_numeric(candidate_work.loc[exact_item_positive, "user_item_recent_count_180d"], errors="coerce")
        .fillna(0).gt(0).sum()
    )
    target_observed_recency_n = int(
        pd.to_numeric(candidate_work.loc[exact_item_positive, "user_item_recency_days"], errors="coerce")
        .fillna(float(RECENCY_FEATURE_FILL_DAYS))
        .lt(float(RECENCY_FEATURE_FILL_DAYS)).sum()
    )
    exact_item_target_signal = {
        "user_item_seen_strength_positive_n": target_seen_strength_n,
        "user_item_recent_count_180d_positive_n": target_recent_count_n,
        "user_item_recency_days_observed_n": target_observed_recency_n,
    }
    if any(exact_item_target_signal.values()):
        raise RuntimeError(
            "Next-novel-item targets have previously-reviewed-item evidence: "
            f"{exact_item_target_signal}"
        )
    candidate_work = ensure_common_regime_features(candidate_work)

    if "query_token_map" in globals():
        candidate_work["query_unique_token_len"] = candidate_work["query_id"].astype(str).map({qid: len(tokens) for qid, tokens in query_token_map.items()}).fillna(0).astype(np.int32)
    else:
        candidate_work["query_unique_token_len"] = candidate_work.get("query_token_len", 0)
    candidate_work["query_unique_token_len_log1p"] = np.log1p(pd.to_numeric(candidate_work["query_unique_token_len"], errors="coerce").fillna(0).astype(np.float32)).astype(np.float32)

    optional_feature_col_map = {}
    available_optional_features = []
    missing_static_features = [
        column for column in EXPECTED_STATIC_FEATURE_COLUMNS
        if column not in candidate_work.columns
    ]
    if missing_static_features:
        raise RuntimeError(f"Missing required model features: {missing_static_features}")

    static_feature_cols = list(EXPECTED_STATIC_FEATURE_COLUMNS)
    for column in static_feature_cols:
        candidate_work[column] = pd.to_numeric(
            candidate_work[column], errors="coerce"
        ).replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(np.float32)

    used_model_feature_cols = list(EXPECTED_MODEL_FEATURE_COLUMNS)
    exact_item_primary_intersection = sorted(
        set(used_model_feature_cols).intersection(EXACT_ITEM_FAMILIARITY_FEATURE_COLS)
    )
    if exact_item_primary_intersection:
        raise RuntimeError(
            "Previously-reviewed-item diagnostics entered the primary LightGBM matrix: "
            f"{exact_item_primary_intersection}"
        )
    if len(used_model_feature_cols) != len(set(used_model_feature_cols)):
        raise RuntimeError("Duplicated model features detected.")

    candidate_identity_after_features = candidate_work[["query_id", "candidate_item_id"]]
    candidate_identity_unchanged = candidate_identity_before_features.reset_index(drop=True).equals(
        candidate_identity_after_features.reset_index(drop=True)
    )
    if not candidate_identity_unchanged:
        raise RuntimeError("Feature construction changed candidate rows or candidate IDs.")

    raw_review_model_columns = sorted([
        column for column in used_model_feature_cols
        if any(token in column.lower() for token in ["review_text", "review_body", "raw_review"])
    ])
    if raw_review_model_columns:
        raise RuntimeError(f"Raw review columns entered model input: {raw_review_model_columns}")

    user_brand_feature_columns_P2Q = (
        sorted(set(used_model_feature_cols) & set(USER_BRAND_FEATURE_COLUMNS))
        if EXPERIMENT_CONDITION == "baseline_retrieval_no_prior_reranking"
        else []
    )
    if user_brand_feature_columns_P2Q:
        raise RuntimeError(f"P2-Q contains user-brand features: {user_brand_feature_columns_P2Q}")

    feature_columns_P2P = list(SHARED_ALL_PRIOR_FEATURE_COLUMNS)
    feature_columns_Full = list(SHARED_ALL_PRIOR_FEATURE_COLUMNS)
    feature_dtypes_P2P = {column: "float32" for column in feature_columns_P2P}
    feature_dtypes_Full = {column: "float32" for column in feature_columns_Full}
    assert feature_columns_P2P == feature_columns_Full
    assert feature_dtypes_P2P == feature_dtypes_Full

    temporal_features_available = [
        column for column in COMMON_TEMPORAL_FEATURE_COLS
        if column in candidate_work.columns
    ]
    feature_nonnull_rate_df = pd.DataFrame({
        "feature": static_feature_cols,
        "nonnull_rate": [float(candidate_work[column].notna().mean()) for column in static_feature_cols],
    }).sort_values("nonnull_rate").reset_index(drop=True)
    feature_nonnull_df = feature_nonnull_rate_df.copy()

    def _feature_group(column):
        if column in BRAND_ALL_PRIOR_STATIC_FEATURE_COLS + BRAND_ALL_PRIOR_TEMPORAL_FEATURE_COLS or column == "candidate_brand_present":
            return "brand"
        if column in COMMON_TEMPORAL_FEATURE_COLS:
            return "temporal_recency"
        if column in COMMON_FAMILY_FEATURE_COLS:
            return "family_match"
        if column in COMMON_DYNAMIC_FEATURE_COLS:
            return "dynamic_pool"
        return "static_candidate"

    used_model_features_df = pd.DataFrame({
        "feature": used_model_feature_cols,
        "feature_group": [_feature_group(column) for column in used_model_feature_cols],
        "dtype": ["float32"] * len(used_model_feature_cols),
        "is_brand_feature": [
            column == "candidate_brand_present" or column in USER_BRAND_FEATURE_COLUMNS
            for column in used_model_feature_cols
        ],
        "is_user_brand_feature": [column in USER_BRAND_FEATURE_COLUMNS for column in
        used_model_feature_cols],
        "is_temporal_feature": [column in COMMON_TEMPORAL_FEATURE_COLS for column in
        used_model_feature_cols],
    })

    feature_leakage_qc_df = build_feature_leakage_qc(
        used_model_feature_cols,
        temporal_features_available,
        user_temporal_maps,
    )
    if not bool(feature_leakage_qc_df["passed"].all()):
        failed_checks = feature_leakage_qc_df.loc[
            ~feature_leakage_qc_df["passed"]
        ].to_dict(orient="records")
        raise RuntimeError(f"Feature leakage QC failed: {failed_checks}")

    users_with_prior_brand = int(sum(
        bool(profile.get("family_count", {}).get("brand", {}))
        for profile in user_profile_map.values()
    ))
    candidate_brand_prior_match_rate = (
        float(candidate_work["candidate_brand_seen_in_prior"].mean())
        if "candidate_brand_seen_in_prior" in candidate_work.columns
        else np.nan
    )
    cold_user_count = int(
        eval_query_users_df["regime"].astype(str).eq("cold").sum()
    )
    brand_contract_diagnostics_df = pd.DataFrame([{
        "candidate_rows": int(len(candidate_work)),
        "cases": int(candidate_work["case_id"].nunique()),
        "candidate_brand_nonnull_rate": float(candidate_work["candidate_brand_present"].mean()),
        "users_with_at_least_one_prior_brand": users_with_prior_brand,
        "candidate_brand_prior_match_rate": candidate_brand_prior_match_rate,
        "cold_user_count": cold_user_count,
        "p2q_feature_count": int(len(P2Q_FEATURE_COLUMNS)),
        "p2p_feature_count": int(len(SHARED_ALL_PRIOR_FEATURE_COLUMNS)),
        "full_feature_count": int(len(SHARED_ALL_PRIOR_FEATURE_COLUMNS)),
        "brand_aware_feature_count": int(sum(
            column == "candidate_brand_present" or column in USER_BRAND_FEATURE_COLUMNS
            for column in used_model_feature_cols
        )),
        "p2p_full_feature_schema_equal": bool(feature_columns_P2P == feature_columns_Full),
        "p2p_full_feature_dtype_equal": bool(feature_dtypes_P2P == feature_dtypes_Full),
        "temporal_validation_passed": bool(all_prior_timestamps_strict),
        "same_target_prior_rows": int(same_target_prior_n),
        "query_brand_leak_rows": int(query_brand_leak_row_count),
        "candidate_identity_unchanged": bool(candidate_identity_unchanged),
    }])

    feature_manifest = {
        "notebook_name": NOTEBOOK_NAME,
        "stage": STAGE,
        "category_id": CATEGORY_ID,
        "category_folder": CATEGORY_FOLDER,
        "category_label": CATEGORY_LABEL,
        "reranking_method": RERANKING_METHOD,
        "candidate_source": CANDIDATE_SOURCE_LABEL,
        "common_feature_registry_version": COMMON_FEATURE_REGISTRY_VERSION,
        "task_scope": "query_conditioned_next_novel_item_ranking",
        "specification_role": SPECIFICATION_ROLE,
        "primary_registry_policy_version": PRIMARY_REGISTRY_POLICY_VERSION,
        "exact_item_familiarity_enabled": ENABLE_EXACT_ITEM_FAMILIARITY,
        "exact_item_familiarity_diagnostic_columns": list(EXACT_ITEM_FAMILIARITY_FEATURE_COLS),
        "exact_item_familiarity_in_primary_model": False,
        "legacy_results_reusable": LEGACY_RESULTS_REUSABLE,
        "requires_downstream_reaggregation": REQUIRES_DOWNSTREAM_REAGGREGATION,
        "temporal_feature_version": TEMPORAL_FEATURE_VERSION,
        "temporal_leakage_rule": TEMPORAL_LEAKAGE_RULE,
        "model_feature_policy": "Numeric-only LightGBM contract; raw candidate brand is retained for audit but not modeled as an unbounded categorical identity.",
        "candidate_score_feature_policy": MODEL_CANDIDATE_SCORE_FEATURE_POLICY,
        "brand_query_enabled": False,
        "brand_candidate_visible": True,
        "brand_reranking_enabled": True,
        "user_brand_affinity_enabled": bool(USER_BRAND_AFFINITY_ENABLED),
        "history_source": "all_prior",
        "historical_population_review_signals_in_user_profile": False,
        "candidate_brand_source_column": CANDIDATE_BRAND_SOURCE_COLUMN,
        "raw_candidate_brand_modeled": False,
        "p2q_feature_columns": P2Q_FEATURE_COLUMNS,
        "shared_p2p_full_feature_columns": SHARED_ALL_PRIOR_FEATURE_COLUMNS,
        "shared_p2p_full_feature_dtypes": feature_dtypes_P2P,
        "used_static_feature_cols": static_feature_cols,
        "used_model_features": used_model_feature_cols,
        "used_model_feature_count": int(len(used_model_feature_cols)),
        "brand_aware_model_features": [
            column for column in used_model_feature_cols
            if column == "candidate_brand_present" or column in USER_BRAND_FEATURE_COLUMNS
        ],
        "temporal_artifact_paths": {name: str(path) for name, path in TEMPORAL_ARTIFACT_PATHS.items()},
        "temporal_prior_filter": {
            "predicate": TEMPORAL_LEAKAGE_RULE,
            "strict_prior_row_count": int(user_temporal_maps.get("strict_prior_row_count", 0)),
            "future_or_equal_prior_rows_excluded": int(user_temporal_maps.get("prior_future_or_equal_excluded_n", 0)),
        },
        "raw_review_text_used": False,
        "raw_query_text_used_as_model_feature": False,
        "target_timestamp_ms_used_as_metadata_only": True,
        "timestamp_ms_used_as_metadata_only": True,
        "candidate_identity_unchanged": bool(candidate_identity_unchanged),
        "query_brand_leak_rows": int(query_brand_leak_row_count),
    }



    candidate_count_check_df = candidate_work.groupby("query_id").size().rename("candidate_n").reset_index()
    gt_pool_coverage = candidate_work.groupby("query_id")["is_gt"].max().mean()

    print("candidate_work shape:", candidate_work.shape)
    print("candidate max pool depth:", MAX_POOL_DEPTH)
    print("Ground-truth coverage in the max rerank pool:", round(float(gt_pool_coverage), 6))
    print("Number of static feature columns:", len(static_feature_cols))
    print("Number of used model features including pool-time features:", len(used_model_feature_cols))
    print("Temporal feature columns added:", len(temporal_features_available))
    print("Candidate count summary:")
    display(candidate_count_check_df["candidate_n"].describe())
    temporal_feature_summary_df = pd.DataFrame({
        "feature": [col for col in COMMON_TIME_FEATURE_COLS if col in candidate_work.columns],
        "nonnull_rate": [
            float(candidate_work[col].notna().mean())
            for col in COMMON_TIME_FEATURE_COLS if col in candidate_work.columns
        ],
        "mean": [
            float(
                pd.to_numeric(candidate_work[col], errors="coerce")
                .replace([np.inf, -np.inf], np.nan)
                .mean()
            )
            for col in COMMON_TIME_FEATURE_COLS if col in candidate_work.columns
        ],
    })

    print("Lowest non-null feature rates:")
    display(feature_nonnull_rate_df.head(20))
    print("Temporal feature summary:")
    display(temporal_feature_summary_df.head(30))

    candidate_features_export_df = candidate_work.drop(columns=["query_tokens"], errors="ignore").copy()
    feature_table_df = candidate_work


candidate_work shape: (2288000, 156)
candidate max pool depth: 1000
Ground-truth coverage in the max rerank pool: 0.246941
Number of static feature columns: 55
Number of used model features including pool-time features: 56
Temporal feature columns added: 21
Candidate count summary:


,candidate_n
count,2288.0
mean,1000.0
std,0.0
min,1000.0
25%,1000.0
50%,1000.0
75%,1000.0
max,1000.0


Lowest non-null feature rates:


,feature,nonnull_rate
0,candidate_rank,1.0
1,query_token_len,1.0
2,item_title_token_overlap,1.0
3,item_doc_token_overlap,1.0
4,prior_review_n_log1p,1.0
5,prior_item_n_log1p,1.0
6,user_entropy_norm_mean,1.0
7,qmatch__product_type,1.0
8,qmatch__form,1.0
9,qmatch__ingredient,1.0


Temporal feature summary:


,feature,nonnull_rate,mean
0,item_prequery_review_count,1.0,55.590286
1,item_recent_review_share_180d,1.0,0.130565
2,item_last_review_gap_days,1.0,2380.876953
3,item_first_review_age_days,1.0,2901.578857
4,user_last_interaction_gap_days,1.0,2635.444824
5,user_product_type_recency_days,1.0,4406.440430
6,user_product_type_recent_count_180d,1.0,2.970076
7,user_form_recency_days,1.0,7233.418945
8,user_form_recent_count_180d,1.0,0.572402
9,user_ingredient_recency_days,1.0,9086.782227


## 8. Train / Validate LightGBM Reranker and Apply by Pool Depth

In [41]:
# ==== Train / Validate LightGBM Reranker and Apply By Pool Depth ====
# Train a leakage-safe LightGBM ranker by pool depth and generate out-of-fold scores

def build_pool_frame(candidate_df: pd.DataFrame, pool_depth: int) -> tuple[pd.DataFrame, list]:
    base_required_cols = {
        "case_id",
        "query_id",
        "user_id",
        "regime",
        "sampling_bracket",
        "gt_item_id",
        "query_text",
        "candidate_pool_type",
        "retrieval_method",
        "retrieval_method_label",
        "candidate_item_id",
        "candidate_rank",
        "is_gt",
        "candidate_score",
        "candidate_brand",
        "timestamp_ms",
        "prior_review_n",
        "prior_item_n",
        "user_total_reviews",
        "profile_history_review_n",
        "profile_history_item_n",
        "profile_regime",
        "query_token_len",
        "removed_token_count",
        "user_entropy_norm_mean",
        "user_top_share_mean",
        "query_item_structured_match",
        "user_item_affinity",
        "user_item_seen_strength",
    }
    base_required_cols.update(static_feature_cols)
    base_required_cols.update(optional_feature_col_map.values())
    base_required_cols.update([c for c in candidate_optional_score_cols if c in candidate_df.columns])

    required_for_pool = ["query_id", "candidate_item_id", "candidate_rank", "candidate_score", "is_gt"]
    missing_for_pool = [c for c in required_for_pool if c not in candidate_df.columns]
    if missing_for_pool:
        raise RuntimeError(f"candidate_df is missing required pool columns: {missing_for_pool}")

    keep_cols = [c for c in candidate_df.columns if c in base_required_cols]

    candidate_rank_num = pd.to_numeric(candidate_df["candidate_rank"], errors="coerce").fillna(0).astype(np.int32)
    pool_df = candidate_df.loc[candidate_rank_num.le(int(pool_depth)), keep_cols].copy()

    pool_df["candidate_rank"] = pd.to_numeric(
        pool_df["candidate_rank"], errors="coerce"
    ).fillna(0).astype(np.int32)

    duplicate_pool_candidate_n = int(pool_df.duplicated(["query_id", "candidate_item_id"]).sum())
    if duplicate_pool_candidate_n:
        raise RuntimeError(
            f"Pool depth {pool_depth} has duplicate candidate_item_id values within query_id: {duplicate_pool_candidate_n}"
        )

    pool_candidate_counts = pool_df.groupby("query_id", sort=False).size()
    over_depth_n = int(pool_candidate_counts.gt(int(pool_depth)).sum())
    if over_depth_n:
        bad_counts = pool_candidate_counts[pool_candidate_counts.gt(int(pool_depth))].head(10).to_dict()
        raise RuntimeError(f"Pool depth {pool_depth} has query candidate counts above pool_depth: {bad_counts}")

    pool_df["candidate_score"] = pd.to_numeric(
        pool_df["candidate_score"], errors="coerce"
    ).fillna(0.0).astype(np.float32)

    pool_df["candidate_score_norm_pool"] = (
        pool_df.groupby("query_id", sort=False)["candidate_score"]
        .transform(minmax_norm_series)
        .fillna(0.0)
        .astype(np.float32)
    )

    candidate_rank_float = pool_df["candidate_rank"].astype(np.float32)
    pool_df["candidate_rank_pct_in_pool"] = (candidate_rank_float / np.float32(pool_depth)).astype(np.float32)
    pool_df["candidate_rank_from_bottom"] = (np.float32(pool_depth) - candidate_rank_float).astype(np.float32)
    pool_df["pool_depth"] = np.int32(pool_depth)

    dynamic_feature_cols = [
        "candidate_score_norm_pool",
    ]

    for _, safe_col in optional_feature_col_map.items():
        if safe_col not in pool_df.columns:
            continue
        pool_norm_col = f"{safe_col}__pool_norm"
        pool_df[pool_norm_col] = (
            pool_df.groupby("query_id", sort=False)[safe_col]
            .transform(minmax_norm_series)
            .fillna(0.0)
            .astype(np.float32)
        )
        dynamic_feature_cols.append(pool_norm_col)

    pool_df["gt_in_pool"] = (
        pool_df.groupby("query_id", sort=False)["is_gt"].transform("max").astype(np.int8)
    )
    pool_df["lgbm_score"] = np.float32(np.nan)
    pool_df["cv_fold"] = np.float32(np.nan)
    pool_df["_query_id_str"] = pool_df["query_id"].astype(str)

    all_feature_cols = [c for c in static_feature_cols + dynamic_feature_cols if c in pool_df.columns]
    all_feature_cols = list(dict.fromkeys(all_feature_cols))

    for col in all_feature_cols:
        pool_df[col] = pd.to_numeric(pool_df[col], errors="coerce").fillna(0.0).astype(np.float32)

    if all_feature_cols != EXPECTED_MODEL_FEATURE_COLUMNS:
        raise RuntimeError(
            f"Model feature order differs from the executed contract: "
            f"expected={EXPECTED_MODEL_FEATURE_COLUMNS}, actual={all_feature_cols}"
        )
    feature_dtype_contract = {column: str(pool_df[column].dtype) for column in all_feature_cols}
    if feature_dtype_contract != {column: "float32" for column in EXPECTED_MODEL_FEATURE_COLUMNS}:
        raise RuntimeError(f"Model feature dtype contract failed: {feature_dtype_contract}")

    gc.collect()
    return pool_df, all_feature_cols

In [45]:
# ==== LightGBM-Specific Matched Training Split Plan ====
def lightgbm_file_sha256(path_obj, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with Path(path_obj).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def lightgbm_json_hash(payload: dict) -> str:
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    ).hexdigest()


def lightgbm_frame_hash(frame: pd.DataFrame, columns: list[str], sort_columns: list[str]) -> str:
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise RuntimeError(f"Cannot hash missing LightGBM contract columns: {missing}")
    work = frame[columns].copy().sort_values(sort_columns, kind="mergesort").reset_index(drop=True)
    digest = hashlib.sha256()
    digest.update("|".join(columns).encode("utf-8"))
    digest.update(pd.util.hash_pandas_object(work, index=False).values.tobytes())
    return digest.hexdigest()


def lightgbm_candidate_identity_hash(pool_df: pd.DataFrame) -> str:
    work = pool_df[["_query_id_str", "candidate_item_id", "candidate_rank", "is_gt"]].copy()
    work["_query_id_str"] = work["_query_id_str"].astype(str)
    work["candidate_item_id"] = work["candidate_item_id"].astype(str)
    work["candidate_rank"] = pd.to_numeric(work["candidate_rank"], errors="raise").astype(np.int32)
    work["is_gt"] = pd.to_numeric(work["is_gt"], errors="raise").astype(np.int8)
    return lightgbm_frame_hash(
        work,
        ["_query_id_str", "candidate_item_id", "candidate_rank", "is_gt"],
        ["_query_id_str", "candidate_rank", "candidate_item_id"],
    )


def lightgbm_feature_rows_hash(frame: pd.DataFrame, feature_cols: list[str]) -> str:
    columns = ["query_id", "candidate_item_id", "is_gt", *feature_cols]
    work = frame[columns].copy()
    work["query_id"] = work["query_id"].astype(str)
    work["candidate_item_id"] = work["candidate_item_id"].astype(str)
    work["is_gt"] = pd.to_numeric(work["is_gt"], errors="raise").astype(np.int8)
    for feature in feature_cols:
        work[feature] = pd.to_numeric(work[feature], errors="raise").astype(np.float32)
    return lightgbm_frame_hash(
        work,
        columns,
        ["query_id", "candidate_item_id"],
    )


def lightgbm_query_meta_from_pool(pool_df: pd.DataFrame) -> pd.DataFrame:
    required = [
        "_query_id_str", "case_id", "user_id", "timestamp_ms", "regime", "sampling_bracket",
        "gt_in_pool", "prior_review_n", "prior_item_n",
        "profile_history_review_n", "profile_history_item_n",
    ]
    missing = [column for column in required if column not in pool_df.columns]
    if missing:
        raise RuntimeError(f"Matched LightGBM query metadata is missing columns: {missing}")
    raw = pool_df[required].copy()
    inconsistent = []
    for column in required[1:]:
        bad = raw.groupby("_query_id_str", sort=False)[column].nunique(dropna=False).gt(1)
        if bad.any():
            inconsistent.extend(bad.index[bad].astype(str).tolist()[:10])
    if inconsistent:
        raise RuntimeError(
            "LightGBM query-level fields vary within a candidate group: "
            f"{sorted(set(inconsistent))[:10]}"
        )
    meta = raw.drop_duplicates("_query_id_str").rename(columns={"_query_id_str": "query_id"})
    meta["query_id"] = meta["query_id"].astype(str)
    meta["case_id"] = meta["case_id"].astype(str)
    def _plain_string(series, fill_value=""):
        return series.astype("object").where(series.notna(), fill_value).astype(str)

    meta["user_id"] = _plain_string(meta["user_id"])
    meta["regime"] = _plain_string(meta["regime"]).str.lower()
    meta["sampling_bracket"] = _plain_string(meta["sampling_bracket"])
    meta["timestamp_ms"] = pd.to_numeric(meta["timestamp_ms"], errors="raise").astype(np.int64)
    meta["gt_in_pool"] = pd.to_numeric(meta["gt_in_pool"], errors="raise").astype(np.int8)
    if meta["user_id"].eq("").any():
        raise RuntimeError("Matched LightGBM folds require non-empty user_id values.")
    history_columns = [
        "prior_review_n", "prior_item_n", "profile_history_review_n", "profile_history_item_n"
    ]
    for column in history_columns:
        meta[column] = pd.to_numeric(meta[column], errors="coerce").fillna(0).astype(int)
    strict_zero_history = meta[history_columns].eq(0).all(axis=1)
    upstream_cold = meta["regime"].eq("cold")
    meta["regime_strict_history_mismatch"] = upstream_cold.ne(strict_zero_history).astype(bool)
    meta["is_strict_cold_query"] = strict_zero_history.astype(bool)
    meta["is_fit_eligible"] = (~strict_zero_history & meta["gt_in_pool"].eq(1)).astype(bool)

    if meta["regime_strict_history_mismatch"].any():
        mismatch_n = int(meta["regime_strict_history_mismatch"].sum())
        print(
            f"Warning: upstream regime disagrees with effective strict pre-target cold status "
            f"for {mismatch_n} queries; using strict history counts for LightGBM fit eligibility."
        )
    return meta.sort_values("query_id", kind="mergesort").reset_index(drop=True)


def lightgbm_build_split_plan(query_meta: pd.DataFrame, pool_depth: int) -> pd.DataFrame:
    if N_FOLDS != 5:
        raise RuntimeError("Final matched LightGBM benchmark requires exactly five outer folds.")
    if query_meta["user_id"].nunique() < N_FOLDS:
        raise RuntimeError("Insufficient user groups for five user-group-disjoint LightGBM folds.")
    if query_meta["is_fit_eligible"].sum() < N_FOLDS:
        raise RuntimeError("Insufficient target-present non-cold queries for five-fold LightGBM fitting.")
    splitter = GroupKFold(n_splits=N_FOLDS)
    rows = []
    for outer_fold, (train_index, test_index) in enumerate(
        splitter.split(query_meta, groups=query_meta["user_id"]), start=1
    ):
        train_meta = query_meta.iloc[train_index]
        test_meta = query_meta.iloc[test_index]
        eligible_train = train_meta.loc[train_meta["is_fit_eligible"]].copy()
        valid_ids, fit_ids = build_time_based_valid_queries(
            eligible_train[["query_id", "timestamp_ms"]].drop_duplicates("query_id"),
            valid_frac=VALID_FRAC_WITHIN_TRAIN,
        )
        fit_ids = set(str(value) for value in fit_ids)
        valid_ids = set(str(value) for value in valid_ids)
        if not fit_ids or not valid_ids:
            raise RuntimeError(
                f"Fold {outer_fold} at depth {pool_depth} lacks a non-empty fit or validation query set."
            )
        eligible_ids = set(eligible_train["query_id"].astype(str))
        if fit_ids & valid_ids or fit_ids | valid_ids != eligible_ids:
            raise RuntimeError("Time-based fit/validation membership does not partition eligible queries.")
        for split_role, ids in [
            ("fit", fit_ids),
            ("valid", valid_ids),
            ("test", set(test_meta["query_id"].astype(str))),
        ]:
            role_meta = query_meta.loc[query_meta["query_id"].isin(ids)]
            rows.extend({
                "pool_depth": int(pool_depth),
                "outer_fold": int(outer_fold),
                "split_role": split_role,
                "query_id": str(row.query_id),
                "case_id": str(row.case_id),
                "user_id": str(row.user_id),
                "regime": str(row.regime),
                "gt_in_pool": int(row.gt_in_pool),
                "is_fit_eligible": bool(row.is_fit_eligible),
            } for row in role_meta.itertuples(index=False))
    plan = pd.DataFrame(rows).sort_values(
        ["outer_fold", "split_role", "query_id"], kind="mergesort"
    ).reset_index(drop=True)
    return plan


def lightgbm_validate_split_plan(plan: pd.DataFrame, query_meta: pd.DataFrame, pool_depth: int) -> None:
    required = [
        "pool_depth", "outer_fold", "split_role", "query_id", "case_id", "user_id",
        "regime", "gt_in_pool", "is_fit_eligible",
    ]
    missing = [column for column in required if column not in plan.columns]
    if missing:
        raise RuntimeError(f"LightGBM split-plan artifact is missing columns: {missing}")
    if not plan["pool_depth"].astype(int).eq(int(pool_depth)).all():
        raise RuntimeError("LightGBM split-plan pool depth mismatch.")
    if sorted(plan["outer_fold"].astype(int).unique()) != list(range(1, N_FOLDS + 1)):
        raise RuntimeError("LightGBM split plan does not contain exactly five outer folds.")
    if not set(plan["split_role"].astype(str)).issubset({"fit", "valid", "test"}):
        raise RuntimeError("LightGBM split plan contains an unknown split role.")
    duplicate = plan.duplicated(["outer_fold", "query_id"])
    if duplicate.any():
        raise RuntimeError("A query has multiple roles within one LightGBM outer fold.")
    test_rows = plan.loc[plan["split_role"].eq("test")]
    if test_rows["query_id"].duplicated().any() or set(test_rows["query_id"]) != set(query_meta["query_id"]):
        raise RuntimeError("LightGBM split plan does not score every query exactly once OOF.")
    query_lookup = query_meta.set_index("query_id")
    for fold in range(1, N_FOLDS + 1):
        fold_plan = plan.loc[plan["outer_fold"].eq(fold)]
        train_rows = fold_plan.loc[fold_plan["split_role"].isin(["fit", "valid"])]
        test_fold = fold_plan.loc[fold_plan["split_role"].eq("test")]
        if train_rows.empty or test_fold.empty:
            raise RuntimeError(f"LightGBM fold {fold} has an empty training or test partition.")
        if train_rows["regime"].astype(str).str.lower().eq("cold").any():
            raise RuntimeError("Cold queries entered a matched LightGBM fit/validation partition.")
        if not train_rows["gt_in_pool"].astype(int).eq(1).all():
            raise RuntimeError("Target-absent queries entered a matched LightGBM fit/validation partition.")
        if not train_rows["is_fit_eligible"].astype(bool).all():
            raise RuntimeError("Ineligible queries entered a matched LightGBM training partition.")
        test_users = set(test_fold["user_id"].astype(str))
        train_users = set(train_rows["user_id"].astype(str))
        if test_users & train_users:
            raise RuntimeError(f"LightGBM outer fold {fold} is not user-group disjoint.")
        expected_train_ids = set(
            query_lookup.loc[~query_lookup.index.isin(test_fold["query_id"])]
            .loc[lambda frame: frame["is_fit_eligible"], :].index.astype(str)
        )
        if set(train_rows["query_id"].astype(str)) != expected_train_ids:
            raise RuntimeError(f"LightGBM fold {fold} does not use the complete eligible train universe.")


def lightgbm_training_artifact_paths(pool_depth: int) -> tuple[Path, Path]:
    plan_path = LIGHTGBM_COMMON_TRAINING_DIR / f"training_split_plan_pool{int(pool_depth)}.parquet"
    manifest_path = LIGHTGBM_COMMON_TRAINING_DIR / f"training_split_plan_pool{int(pool_depth)}_manifest.json"
    return plan_path, manifest_path


def lightgbm_expected_plan_contract(pool_df: pd.DataFrame, query_meta: pd.DataFrame, pool_depth: int) -> dict:
    query_hash_frame = query_meta[[
        "query_id", "case_id", "user_id", "timestamp_ms", "regime", "sampling_bracket",
        "gt_in_pool", "is_strict_cold_query", "is_fit_eligible",
    ]].copy()
    query_universe_hash = lightgbm_frame_hash(
        query_hash_frame,
        list(query_hash_frame.columns),
        ["query_id"],
    )
    base_feature_contract_hash = lightgbm_json_hash({
        "p2q_feature_columns": list(P2Q_FEATURE_COLUMNS),
        "preprocessing": FEATURE_PREPROCESSING_CONTRACT,
    })
    model_parameter_hash = lightgbm_json_hash({
        "ranker_params_base": make_jsonable_dict(LIGHTGBM_RANKER_PARAMS_BASE),
        "n_folds": int(N_FOLDS),
        "valid_frac_within_train": float(VALID_FRAC_WITHIN_TRAIN),
        "early_stopping_rounds": int(LIGHTGBM_EARLY_STOPPING_ROUNDS),
        "eval_at": list(LIGHTGBM_EVAL_AT),
        "random_seed": int(RANDOM_SEED),
        "matched_fitting_policy": LIGHTGBM_MATCHED_FITTING_POLICY,
        "sample_weight_policy": "none_query_groups_unweighted",
    })
    return {
        "contract_version": LIGHTGBM_TRAINING_PLAN_VERSION,
        "category_id": CATEGORY_ID,
        "pool_depth": int(pool_depth),
        "query_universe_hash": query_universe_hash,
        "candidate_identity_hash": lightgbm_candidate_identity_hash(pool_df),
        "base_feature_contract_hash": base_feature_contract_hash,
        "model_parameter_hash": model_parameter_hash,
        "matched_fitting_policy": LIGHTGBM_MATCHED_FITTING_POLICY,
        "sample_weight_policy": "none_query_groups_unweighted",
        "n_folds": int(N_FOLDS),
        "candidate_source_path": str(CANDIDATE_POOL_PATH),
    }


def lightgbm_load_or_create_split_plan(
    pool_df: pd.DataFrame,
    query_meta: pd.DataFrame,
    pool_depth: int,
) -> tuple[pd.DataFrame, dict, Path, Path]:
    plan_path, manifest_path = lightgbm_training_artifact_paths(pool_depth)
    expected = lightgbm_expected_plan_contract(pool_df, query_meta, pool_depth)
    if LIGHTGBM_TRAINING_PLAN_ROLE == "write":
        plan = lightgbm_build_split_plan(query_meta, pool_depth)
        lightgbm_validate_split_plan(plan, query_meta, pool_depth)
        LIGHTGBM_COMMON_TRAINING_DIR.mkdir(parents=True, exist_ok=True)
        plan.to_parquet(plan_path, index=False)
        manifest = {
            **expected,
            "plan_path": str(plan_path),
            "plan_hash": lightgbm_frame_hash(
                plan,
                list(plan.columns),
                ["outer_fold", "split_role", "query_id"],
            ),
            "fold_sample_contracts": {},
        }
        manifest_path.write_text(
            json.dumps(make_jsonable_dict(manifest), ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        return plan, manifest, plan_path, manifest_path
    if LIGHTGBM_TRAINING_PLAN_ROLE != "read":
        raise RuntimeError("Only 11a may write and only 11b may read the LightGBM training split plan.")
    if not plan_path.exists() or not manifest_path.exists():
        raise FileNotFoundError(
            f"Run same-category LightGBM 11a first; matched split-plan artifact is missing: {plan_path}"
        )
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    mismatches = {
        key: (manifest.get(key), value)
        for key, value in expected.items()
        if manifest.get(key) != value
    }
    if mismatches:
        raise RuntimeError(f"LightGBM 11a/11b matched training-plan contract mismatch: {mismatches}")
    plan = pd.read_parquet(plan_path)
    lightgbm_validate_split_plan(plan, query_meta, pool_depth)
    observed_plan_hash = lightgbm_frame_hash(
        plan,
        list(plan.columns),
        ["outer_fold", "split_role", "query_id"],
    )
    if observed_plan_hash != manifest.get("plan_hash"):
        raise RuntimeError("LightGBM split-plan bytes do not reproduce the manifest hash.")
    if not manifest.get("fold_sample_contracts"):
        raise RuntimeError("LightGBM 11a split-plan manifest lacks finalized fold sample hashes.")
    return plan, manifest, plan_path, manifest_path


def lightgbm_finalize_writer_manifest(
    manifest: dict,
    manifest_path: Path,
    fold_sample_contracts: dict,
) -> dict:
    if LIGHTGBM_TRAINING_PLAN_ROLE != "write":
        return manifest
    finalized = {**manifest, "fold_sample_contracts": fold_sample_contracts}
    manifest_path.write_text(
        json.dumps(make_jsonable_dict(finalized), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return finalized


# ==== Per-Iteration Learning-Curve Export ====
# Derive training and inner-validation NDCG@5 trajectories from the fitted booster
# without modifying the fitted model, selected iteration, or OOF predictions.
def _e5_ndcg_at_5_from_scores(frame, scores, k=5):
    tmp = pd.DataFrame({
        "q": frame["query_id"].astype(str).to_numpy(),
        "gt": pd.to_numeric(frame["is_gt"], errors="raise").to_numpy(dtype=int),
        "s": np.asarray(scores, dtype=float),
        "_ord": np.arange(len(frame)),
    })
    tmp = tmp.sort_values(["q", "s", "_ord"], ascending=[True, False, True], kind="mergesort")
    tmp["rank"] = tmp.groupby("q", sort=False).cumcount() + 1
    gt_ranks = tmp.loc[tmp["gt"] == 1, "rank"].to_numpy(dtype=float)
    if gt_ranks.size == 0:
        return float("nan")
    gains = np.where(gt_ranks <= k, 1.0 / np.log2(gt_ranks + 1.0), 0.0)
    return float(np.mean(gains))


def _e5_learning_history_rows(booster, best_iteration, fit_df, valid_df,
                              feature_cols, category_id, pool_depth, fold_index):
    total_iters = int(booster.current_iteration())
    best_iteration = int(best_iteration)
    fit_X = fit_df[feature_cols].astype(np.float32)
    valid_X = valid_df[feature_cols].astype(np.float32)
    # Full diagnostic curve is retained; 300 boosting rounds keeps this bounded.
    E5_CURVE_STRIDE = 1
    iters = sorted(set(list(range(1, total_iters + 1, E5_CURVE_STRIDE)) + [best_iteration]))
    rows = []
    for i in iters:
        if i < 1:
            continue
        rows.append({
            "category_id": category_id,
            "pool_depth": int(pool_depth),
            "outer_fold": int(fold_index),
            "iteration": int(i),
            "train_ndcg_at_5": _e5_ndcg_at_5_from_scores(fit_df, booster.predict(fit_X, num_iteration=i)),
            "valid_ndcg_at_5": _e5_ndcg_at_5_from_scores(valid_df, booster.predict(valid_X, num_iteration=i)),
            "is_selected_best_iteration": bool(i == best_iteration),
            "validation_split": "inner_validation_time_based_within_outer_train",
        })
    return rows


LIGHTGBM_LEARNING_HISTORY_ROWS = []
LIGHTGBM_VARIANT_OUT_DIRS = {
    "P2-Q": PROJECT_ROOT / "outputs/stage2_nonpersonalized_rerank/lightgbm_no_prior",
    "P2-P": PROJECT_ROOT / "outputs/stage2_nonpersonalized_rerank/lightgbm",
    "Full": PROJECT_ROOT / "outputs/stage2_personalized_rerank/lightgbm_full",
}
if CONDITION_NAME not in LIGHTGBM_VARIANT_OUT_DIRS:
    raise RuntimeError(f"Unsupported LightGBM condition for OUT_DIR contract: {CONDITION_NAME}")
if OUT_DIR != LIGHTGBM_VARIANT_OUT_DIRS[CONDITION_NAME]:
    raise RuntimeError(
        f"LightGBM {CONDITION_NAME} OUT_DIR mismatch: "
        f"expected={LIGHTGBM_VARIANT_OUT_DIRS[CONDITION_NAME]}, actual={OUT_DIR}"
    )
LIGHTGBM_LEARNING_HISTORY_PATH = OUT_DIR / "lightgbm_learning_history.csv"


def fit_predict_ranker_one_depth(pool_df: pd.DataFrame, pool_depth: int, feature_cols: list):
    expected_features = P2Q_FEATURE_COLUMNS if LIGHTGBM_VARIANT == "a" else SHARED_ALL_PRIOR_FEATURE_COLUMNS
    if feature_cols != expected_features:
        raise RuntimeError(
            f"LightGBM {LIGHTGBM_VARIANT} feature schema mismatch: "
            f"expected={expected_features}, actual={feature_cols}"
        )
    if LIGHTGBM_VARIANT not in {"a", "b"}:
        raise RuntimeError("This fitting function is only valid for LightGBM 11a/11b.")

    query_meta_pool = lightgbm_query_meta_from_pool(pool_df)
    split_plan, plan_manifest, plan_path, plan_manifest_path = lightgbm_load_or_create_split_plan(
        pool_df, query_meta_pool, pool_depth
    )
    plan_hash = str(plan_manifest["plan_hash"])
    model_parameter_hash = str(plan_manifest["model_parameter_hash"])
    base_feature_contract_hash = str(plan_manifest["base_feature_contract_hash"])
    model_feature_contract_hash = lightgbm_json_hash({
        "feature_columns": list(feature_cols),
        "feature_dtypes": {column: "float32" for column in feature_cols},
        "preprocessing": FEATURE_PREPROCESSING_CONTRACT,
    })

    if LIGHTGBM_VARIANT == "a":
        model_dir = OUT_DIR
        model_contract_path = OUT_DIR / f"p2q_model_contract_pool{pool_depth}.json"
        fold_assignment_path = OUT_DIR / f"p2q_fold_assignment_pool{pool_depth}.parquet"
        model_prefix = "p2q_lightgbm_model"
        condition_name = "P2-Q"
        model_artifact_role = "p2q_no_prior_oof_source"
    else:
        model_dir = SHARED_MODEL_ARTIFACT_DIR
        model_contract_path = model_dir / f"full_model_contract_pool{pool_depth}.json"
        fold_assignment_path = model_dir / f"full_fold_assignment_pool{pool_depth}.parquet"
        model_prefix = "full_model"
        condition_name = "Full"
        model_artifact_role = "full_self_pool_trained_model"
    model_dir.mkdir(parents=True, exist_ok=True)

    pool_df["lgbm_score"] = np.float32(np.nan)
    pool_df["cv_fold"] = np.float32(np.nan)
    training_runtime_sec = 0.0
    scoring_runtime_sec = 0.0
    fold_rows = []
    feature_importance_rows = []
    assignment_rows = []
    contract_folds = []
    writer_fold_sample_contracts = {}
    expected_fold_sample_contracts = plan_manifest.get("fold_sample_contracts", {})

    for fold_index in range(1, N_FOLDS + 1):
        fold_plan = split_plan.loc[split_plan["outer_fold"].astype(int).eq(fold_index)]
        fit_query_ids = set(fold_plan.loc[fold_plan["split_role"].eq("fit"), "query_id"].astype(str))
        valid_query_ids = set(fold_plan.loc[fold_plan["split_role"].eq("valid"), "query_id"].astype(str))
        test_query_ids = set(fold_plan.loc[fold_plan["split_role"].eq("test"), "query_id"].astype(str))
        fit_df = pool_df.loc[pool_df["_query_id_str"].isin(fit_query_ids)].copy()
        valid_df = pool_df.loc[pool_df["_query_id_str"].isin(valid_query_ids)].copy()
        test_df = pool_df.loc[pool_df["_query_id_str"].isin(test_query_ids)].copy()
        fit_df, fit_group = prepare_group_arrays(fit_df)
        valid_df, valid_group = prepare_group_arrays(valid_df)
        test_df, _ = prepare_group_arrays(test_df)
        if not len(fit_df) or not len(valid_df) or not len(test_df):
            raise RuntimeError(f"LightGBM fold {fold_index} has an empty fit/valid/test candidate frame.")
        if fit_df.groupby("query_id")["is_gt"].sum().ne(1).any():
            raise RuntimeError("Every LightGBM fitting query must contain exactly one positive candidate.")
        if valid_df.groupby("query_id")["is_gt"].sum().ne(1).any():
            raise RuntimeError("Every LightGBM validation query must contain exactly one positive candidate.")

        fold_sample_contract = {
            "fit_candidate_identity_hash": lightgbm_feature_rows_hash(fit_df, []),
            "valid_candidate_identity_hash": lightgbm_feature_rows_hash(valid_df, []),
            "base_fit_feature_hash": lightgbm_feature_rows_hash(fit_df, list(P2Q_FEATURE_COLUMNS)),
            "base_valid_feature_hash": lightgbm_feature_rows_hash(valid_df, list(P2Q_FEATURE_COLUMNS)),
            "fit_query_count": int(len(fit_query_ids)),
            "valid_query_count": int(len(valid_query_ids)),
            "test_query_count": int(len(test_query_ids)),
        }
        if LIGHTGBM_TRAINING_PLAN_ROLE == "write":
            writer_fold_sample_contracts[str(fold_index)] = fold_sample_contract
        else:
            expected_sample = expected_fold_sample_contracts.get(str(fold_index))
            if expected_sample != fold_sample_contract:
                raise RuntimeError(
                    f"LightGBM base candidate-feature parity failed at depth={pool_depth}, "
                    f"fold={fold_index}: expected={expected_sample}, actual={fold_sample_contract}"
                )

        training_row_hash = lightgbm_feature_rows_hash(fit_df, list(feature_cols))
        validation_row_hash = lightgbm_feature_rows_hash(valid_df, list(feature_cols))
        fold_seed = int(RANDOM_SEED + pool_depth + fold_index)
        model_filename = f"{model_prefix}_pool{pool_depth}_fold{fold_index}.txt"
        model_path = model_dir / model_filename
        checkpoint_path = model_dir / f"{model_filename}.checkpoint.json"
        checkpoint_expected = {
            "contract_version": LIGHTGBM_MODEL_CONTRACT_VERSION,
            "category_id": CATEGORY_ID,
            "variant": LIGHTGBM_VARIANT,
            "pool_depth": int(pool_depth),
            "fold": int(fold_index),
            "plan_hash": plan_hash,
            "model_parameter_hash": model_parameter_hash,
            "model_feature_contract_hash": model_feature_contract_hash,
            "training_row_hash": training_row_hash,
            "validation_row_hash": validation_row_hash,
            "random_seed": fold_seed,
        }
        booster = None
        reused_valid_checkpoint = False
        checkpoint = None
        if LIGHTGBM_RESUME_VALID_FOLDS and model_path.exists() and checkpoint_path.exists():
            try:
                candidate_checkpoint = json.loads(checkpoint_path.read_text(encoding="utf-8"))
                checkpoint_mismatches = {
                    key: (candidate_checkpoint.get(key), value)
                    for key, value in checkpoint_expected.items()
                    if candidate_checkpoint.get(key) != value
                }
                if not checkpoint_mismatches and lightgbm_file_sha256(model_path) == candidate_checkpoint.get("model_sha256"):
                    candidate_booster = lgb.Booster(model_file=str(model_path))
                    if candidate_booster.feature_name() == feature_cols:
                        booster = candidate_booster
                        checkpoint = candidate_checkpoint
                        reused_valid_checkpoint = True
            except Exception:
                booster = None
                checkpoint = None

        if booster is None:
            fit_kwargs = {
                "X": fit_df[feature_cols].astype(np.float32),
                "y": fit_df["is_gt"].to_numpy(dtype=np.int8, copy=False),
                "group": fit_group,
                "eval_set": [(
                    valid_df[feature_cols].astype(np.float32),
                    valid_df["is_gt"].to_numpy(dtype=np.int8, copy=False),
                )],
                "eval_group": [valid_group],
                "callbacks": [
                    lgb.early_stopping(LIGHTGBM_EARLY_STOPPING_ROUNDS, verbose=False),
                    lgb.log_evaluation(0),
                ],
            }
            ranker = lgb.LGBMRanker(**LIGHTGBM_RANKER_PARAMS_BASE, random_state=fold_seed)
            training_start = time.time()
            ranker.fit(**fit_kwargs)
            fold_training_runtime = float(time.time() - training_start)
            training_runtime_sec += fold_training_runtime
            booster = ranker.booster_
            if booster.feature_name() != feature_cols:
                raise RuntimeError("Saved LightGBM feature order differs from the model contract.")
            best_iteration = int(ranker.best_iteration_ or booster.current_iteration())
            booster.save_model(str(model_path), num_iteration=best_iteration)
            checkpoint = {
                **checkpoint_expected,
                "best_iteration": best_iteration,
                "fit_runtime_sec": fold_training_runtime,
                "model_sha256": lightgbm_file_sha256(model_path),
            }
            checkpoint_path.write_text(
                json.dumps(make_jsonable_dict(checkpoint), ensure_ascii=False, indent=2),
                encoding="utf-8",
            )
            del ranker, fit_kwargs
        else:
            best_iteration = int(checkpoint.get("best_iteration", booster.current_iteration()))
            fold_training_runtime = 0.0

        LIGHTGBM_LEARNING_HISTORY_ROWS.extend(_e5_learning_history_rows(
            booster=booster, best_iteration=best_iteration,
            fit_df=fit_df, valid_df=valid_df, feature_cols=feature_cols,
            category_id=CATEGORY_ID, pool_depth=int(pool_depth), fold_index=int(fold_index),
        ))
        # Persist incrementally so a partial run still yields a curve file.
        pd.DataFrame(LIGHTGBM_LEARNING_HISTORY_ROWS).to_csv(
            LIGHTGBM_LEARNING_HISTORY_PATH, index=False, encoding="utf-8-sig"
        )
        scoring_start = time.time()
        test_predictions = booster.predict(
            test_df[feature_cols].astype(np.float32),
            num_iteration=best_iteration,
        )
        fold_scoring_runtime = float(time.time() - scoring_start)
        scoring_runtime_sec += fold_scoring_runtime
        pool_df.loc[test_df.index, "lgbm_score"] = np.asarray(test_predictions, dtype=np.float32)
        pool_df.loc[test_df.index, "cv_fold"] = np.float32(fold_index)
        test_case_by_query = (
            query_meta_pool.loc[query_meta_pool["query_id"].isin(test_query_ids)]
            .set_index("query_id")["case_id"].astype(str).to_dict()
        )
        assignment_rows.extend({
            "case_id": test_case_by_query[str(query_id)],
            "query_id": str(query_id),
            "fold": int(fold_index),
        } for query_id in sorted(test_query_ids))

        gain_values = booster.feature_importance(importance_type="gain")
        split_values = booster.feature_importance(importance_type="split")
        feature_importance_rows.extend({
            "pool_depth": int(pool_depth),
            "fold": int(fold_index),
            "feature": feature,
            "gain": float(gain_value),
            "split": int(split_value),
        } for feature, gain_value, split_value in zip(feature_cols, gain_values, split_values))
        outer_train_meta = query_meta_pool.loc[~query_meta_pool["query_id"].isin(test_query_ids)]
        n_cold_excluded = int(outer_train_meta["is_strict_cold_query"].sum())
        contract_folds.append({
            "fold": int(fold_index),
            "model_filename": model_filename,
            "model_sha256": lightgbm_file_sha256(model_path),
            "checkpoint_filename": checkpoint_path.name,
            "random_seed": fold_seed,
            "best_iteration": best_iteration,
            "training_row_count": int(len(fit_df)),
            "validation_row_count": int(len(valid_df)),
            "training_positive_count": int(fit_df["is_gt"].sum()),
            "training_row_hash": training_row_hash,
            "validation_row_hash": validation_row_hash,
            "base_fit_feature_hash": fold_sample_contract["base_fit_feature_hash"],
            "base_valid_feature_hash": fold_sample_contract["base_valid_feature_hash"],
            "fit_query_ids": sorted(fit_query_ids),
            "valid_query_ids": sorted(valid_query_ids),
            "test_query_ids": sorted(test_query_ids),
            "reused_valid_checkpoint": bool(reused_valid_checkpoint),
        })
        fold_rows.append({
            "pool_depth": int(pool_depth),
            "fold": int(fold_index),
            "n_train_queries_all": int(len(fit_query_ids) + len(valid_query_ids)),
            "n_cold_queries_excluded_from_training": n_cold_excluded,
            "n_train_queries_positive": int(len(fit_query_ids) + len(valid_query_ids)),
            "n_fit_queries_positive": int(len(fit_query_ids)),
            "n_valid_queries_positive": int(len(valid_query_ids)),
            "n_test_queries": int(len(test_query_ids)),
            "n_test_queries_positive": int(
                query_meta_pool.loc[query_meta_pool["query_id"].isin(test_query_ids), "gt_in_pool"].sum()
            ),
            "pred_nonnull_rate": float(pd.notna(test_predictions).mean()),
            "training_row_hash": training_row_hash,
            "validation_row_hash": validation_row_hash,
            "fit_runtime_sec": fold_training_runtime,
            "score_runtime_sec": fold_scoring_runtime,
            "best_iteration": best_iteration,
            "reused_valid_checkpoint": bool(reused_valid_checkpoint),
        })
        del fit_df, valid_df, test_df, booster, test_predictions, gain_values, split_values
        gc.collect()

    if LIGHTGBM_TRAINING_PLAN_ROLE == "write":
        plan_manifest = lightgbm_finalize_writer_manifest(
            plan_manifest, plan_manifest_path, writer_fold_sample_contracts
        )
    assignment_df = pd.DataFrame(assignment_rows).sort_values(["fold", "query_id"], kind="mergesort")
    if assignment_df["query_id"].duplicated().any() or set(assignment_df["query_id"]) != set(query_meta_pool["query_id"]):
        raise RuntimeError("Matched LightGBM fold assignment does not cover every query exactly once.")
    assignment_df.to_parquet(fold_assignment_path, index=False)
    if pool_df["lgbm_score"].isna().any() or pool_df["cv_fold"].isna().any():
        raise RuntimeError("Matched LightGBM OOF scoring did not cover the complete query universe.")

    plan_manifest_sha256 = lightgbm_file_sha256(plan_manifest_path)
    model_contract = {
        "contract_version": LIGHTGBM_MODEL_CONTRACT_VERSION,
        "shared_model_contract_version": LIGHTGBM_MODEL_CONTRACT_VERSION,
        "condition_name": condition_name,
        "category_id": CATEGORY_ID,
        "pool_depth": int(pool_depth),
        "candidate_source": CANDIDATE_SOURCE_LABEL,
        "candidate_source_output_key": "personalized_winner_long",
        "candidate_pool_manifest_path": str(CANDIDATE_POOL_MANIFEST_PATH),
        "training_candidate_path": str(CANDIDATE_POOL_PATH),
        "model_artifact_role": model_artifact_role,
        "score_feature_policy": MODEL_CANDIDATE_SCORE_FEATURE_POLICY,
        "use_candidate_score_feature": bool(USE_CANDIDATE_SCORE_FEATURE),
        "feature_columns": list(feature_cols),
        "feature_dtypes": {column: "float32" for column in feature_cols},
        "feature_preprocessing_contract": FEATURE_PREPROCESSING_CONTRACT,
        "ranker_params_base": make_jsonable_dict(LIGHTGBM_RANKER_PARAMS_BASE),
        "random_seed_base": int(RANDOM_SEED),
        "n_folds": int(N_FOLDS),
        "fold_assignment_path": str(fold_assignment_path),
        "matched_fitting_policy": LIGHTGBM_MATCHED_FITTING_POLICY,
        "training_plan_path": str(plan_path),
        "training_plan_manifest_path": str(plan_manifest_path),
        "training_plan_manifest_sha256": plan_manifest_sha256,
        "training_plan_hash": plan_hash,
        "query_universe_hash": plan_manifest["query_universe_hash"],
        "candidate_identity_hash": plan_manifest["candidate_identity_hash"],
        "base_feature_contract_hash": base_feature_contract_hash,
        "model_parameter_hash": model_parameter_hash,
        "model_feature_contract_hash": model_feature_contract_hash,
        "sample_weight_policy": "none_query_groups_unweighted",
        "cold_queries_in_fitting": 0,
        "training_rows_used": int(sum(row["training_row_count"] for row in contract_folds)),
        "folds": contract_folds,
    }
    model_contract_path.write_text(
        json.dumps(make_jsonable_dict(model_contract), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    pool_df = pool_df.drop(columns=["_query_id_str"], errors="ignore")
    pool_df.attrs["training_runtime_sec"] = float(training_runtime_sec)
    pool_df.attrs["rerank_scoring_runtime_sec"] = float(scoring_runtime_sec)
    pool_df.attrs["training_rows_used"] = int(model_contract["training_rows_used"])
    return pool_df, pd.DataFrame(fold_rows), pd.DataFrame(feature_importance_rows)


In [46]:
# ==== Train / Validate LightGBM Reranker and Apply By Pool Depth — Part 3 ====
def export_and_score_methods(pool_df: pd.DataFrame, pool_depth: int):
    export_core_cols = [
        "case_id",
        "query_id",
        "user_id",
        "regime",
        "sampling_bracket",
        "gt_item_id",
        "query_text",
        "candidate_pool_type",
        "retrieval_method",
        "retrieval_method_label",
        "reranking_method",
        "pool_depth",
        "rerank_method",
        "candidate_item_id",
        "candidate_rank",
        "new_rank",
        "rank_shift",
        "is_gt",
        "gt_in_pool",
        "candidate_score",
        "candidate_score_norm_pool",
        "lgbm_score",
        "final_score",
        "prediction_source",
        "p2q_fallback_rerank_score",
        "p2q_fallback_rerank_rank",
        "cv_fold",
        "profile_history_review_n",
        "profile_history_item_n",
        "profile_regime",
        "regime_personalization_weight",
        "user_entropy_norm_mean",
        "user_top_share_mean",
        "query_item_structured_match",
        "user_item_affinity",
        "user_item_seen_strength",
    ]

    method_specs = [
        {"rerank_method": BASELINE_RERANK_METHOD, "score_col": "candidate_score_norm_pool"},
        {"rerank_method": OUTPUT_RERANK_METHOD, "score_col": "lgbm_score"},
    ]

    n_query_total = int(pool_df["query_id"].nunique())
    method_frames = []
    method_query_rows = []

    for spec in method_specs:
        method_name = spec["rerank_method"]
        score_col = spec["score_col"]
        per_method_frames = []

        for query_id, g in tqdm(
            pool_df.groupby("query_id", sort=False),
            total=n_query_total,
            desc=f"{method_name} @ {pool_depth}",
            leave=False,
        ):
            gt_item_id = str(g["gt_item_id"].iloc[0])

            if method_name == BASELINE_RERANK_METHOD:
                work = g.sort_values(
                    ["candidate_rank", "candidate_item_id"],
                    ascending=[True, True],
                ).copy()
            else:
                is_p2q_cold_fallback_query = (
                    "prediction_source" in g.columns
                    and g["prediction_source"].astype(str).eq("p2q_cold_fallback").all()
                    and "p2q_fallback_rerank_rank" in g.columns
                )
                if is_p2q_cold_fallback_query:
                    work = g.sort_values(
                        ["p2q_fallback_rerank_rank", "candidate_item_id"],
                        ascending=[True, True],
                    ).copy()
                else:
                    work = g.sort_values(
                        ["lgbm_score", "candidate_score_norm_pool", "candidate_rank", "candidate_item_id"],
                        ascending=[False, False, True, True],
                    ).copy()

            work["final_score"] = pd.to_numeric(work[score_col], errors="coerce").fillna(0.0)
            if method_name != BASELINE_RERANK_METHOD and "prediction_source" in work.columns:
                fallback_rows = work["prediction_source"].astype(str).eq("p2q_cold_fallback")
                if fallback_rows.any():
                    work.loc[fallback_rows, "final_score"] = pd.to_numeric(
                        work.loc[fallback_rows, "p2q_fallback_rerank_score"], errors="coerce"
                    ).fillna(0.0)
            work["pool_depth"] = int(pool_depth)
            work["rerank_method"] = method_name
            if method_name == BASELINE_RERANK_METHOD and "prediction_source" in work.columns:
                work["prediction_source"] = "stage1_baseline"
            work["reranking_method"] = RERANKING_METHOD
            work["new_rank"] = np.arange(1, len(work) + 1, dtype=np.int32)
            work["rank_shift"] = work["candidate_rank"].astype(np.int32) - work["new_rank"].astype(np.int32)
            work = append_preference_diagnostic_columns(work)

            item_id_str = work["candidate_item_id"].astype(str)
            pred_item_ids = item_id_str.tolist()

            metrics = compute_query_metrics(pred_item_ids, gt_item_id, ks=EVAL_KS)
            supplementary_metrics = compute_topk_diagnostics_from_ranked_frame(work, k=PREFERENCE_ALIGNMENT_K)

            gt_rank_series = work.loc[item_id_str.eq(gt_item_id), "new_rank"]
            gt_rank = int(gt_rank_series.iloc[0]) if len(gt_rank_series) else None

            method_query_rows.append({
                "case_id": str(work["case_id"].iloc[0]),
                "query_id": str(query_id),
                "user_id": str(work["user_id"].iloc[0]),
                "regime": str(work["regime"].iloc[0]),
                "sampling_bracket": str(work["sampling_bracket"].iloc[0]),
                "gt_item_id": gt_item_id,
                "query_text": str(work["query_text"].iloc[0]),
                "candidate_pool_type": str(work["candidate_pool_type"].iloc[0]),
                "retrieval_method": str(work["retrieval_method"].iloc[0]),
                "retrieval_method_label": str(work["retrieval_method_label"].iloc[0]),
                "reranking_method": RERANKING_METHOD,
                "pool_depth": int(pool_depth),
                "rerank_method": method_name,
                "gt_in_pool": int(pd.to_numeric(work["is_gt"], errors="coerce").fillna(0).max()),
                "gt_rank": gt_rank,
                "prior_review_n": int(pd.to_numeric(work["prior_review_n"], errors="coerce").fillna(0).iloc[0]),
                "prior_item_n": int(pd.to_numeric(work["prior_item_n"], errors="coerce").fillna(0).iloc[0]),
                "user_total_reviews": int(pd.to_numeric(work["user_total_reviews"], errors="coerce").fillna(0).iloc[0]),
                "profile_history_review_n": int(pd.to_numeric(work["profile_history_review_n"], errors="coerce").fillna(0).iloc[0]),
                "profile_history_item_n": int(pd.to_numeric(work["profile_history_item_n"], errors="coerce").fillna(0).iloc[0]),
                "profile_regime": str(work["profile_regime"].iloc[0]),
                "query_token_len": int(pd.to_numeric(work["query_token_len"], errors="coerce").fillna(0).iloc[0]),
                "removed_token_count": int(pd.to_numeric(work["removed_token_count"], errors="coerce").fillna(0).iloc[0]),
                "user_entropy_norm_mean": float(pd.to_numeric(work["user_entropy_norm_mean"], errors="coerce").fillna(np.nan).iloc[0]),
                "user_top_share_mean": float(pd.to_numeric(work["user_top_share_mean"], errors="coerce").fillna(np.nan).iloc[0]),
                **metrics,
                **supplementary_metrics,
            })

            export_cols = [col for col in export_core_cols + candidate_optional_score_cols + PREFERENCE_EXPORT_DIAGNOSTIC_COLUMNS if col in work.columns]
            per_method_frames.append(work[export_cols])

        method_frames.append(pd.concat(per_method_frames, ignore_index=True))
        del per_method_frames
        gc.collect()

    out_query_df = pd.DataFrame(method_query_rows)
    out_cand_df = pd.concat(method_frames, ignore_index=True)

    del method_frames
    gc.collect()

    return out_query_df, out_cand_df

def load_p2q_oof_predictions() -> pd.DataFrame:
    if not P2Q_OOF_PREDICTION_PATH.exists():
        raise FileNotFoundError(f"Missing same-family P2-Q prediction artifact: {P2Q_OOF_PREDICTION_PATH}")
    if not P2Q_OOF_PREDICTION_MANIFEST_PATH.exists():
        raise FileNotFoundError(f"Missing same-family P2-Q prediction manifest: {P2Q_OOF_PREDICTION_MANIFEST_PATH}")
    p2q_manifest = json.loads(P2Q_OOF_PREDICTION_MANIFEST_PATH.read_text(encoding="utf-8"))
    if p2q_manifest.get("category_id") != CATEGORY_ID or p2q_manifest.get("model_family") != "lightgbm":
        raise RuntimeError("P2-Q prediction manifest category/model mismatch.")
    if p2q_manifest.get("common_feature_registry_version") != COMMON_FEATURE_REGISTRY_VERSION:
        raise RuntimeError("P2-Q prediction manifest feature-registry version mismatch; rerun 11a first.")
    if list(p2q_manifest.get("model_feature_columns", [])) != list(P2Q_FEATURE_COLUMNS):
        raise RuntimeError("P2-Q prediction manifest feature schema mismatch; rerun 11a first.")
    expected_feature_hash = hashlib.sha256('|'.join(P2Q_FEATURE_COLUMNS).encode('utf-8')).hexdigest()
    if p2q_manifest.get("feature_contract_hash") != expected_feature_hash:
        raise RuntimeError("P2-Q prediction manifest feature hash mismatch; rerun 11a first.")
    if p2q_manifest.get("rerank_rank_source_column") != "lgbm_score_sort":
        raise RuntimeError("P2-Q prediction manifest rerank-rank source mismatch.")
    if p2q_manifest.get("rerank_rank_semantics") != "one_based_rank_after_no_prior_lightgbm_score_sort":
        raise RuntimeError("P2-Q prediction manifest rerank-rank semantics mismatch.")
    if p2q_manifest.get("rerank_rank_contract_passed") is not True:
        raise RuntimeError("P2-Q prediction manifest does not record rerank-rank contract success.")
    if Path(p2q_manifest.get("candidate_source_manifest_path", "")) != CANDIDATE_POOL_MANIFEST_PATH:
        raise RuntimeError("P2-Q prediction manifest candidate-source lineage mismatch.")
    if Path(p2q_manifest.get("prediction_path", "")) != P2Q_OOF_PREDICTION_PATH:
        raise RuntimeError("P2-Q prediction manifest prediction-path mismatch.")
    if [int(value) for value in p2q_manifest.get("pool_depths", [])] != [int(value) for value in POOL_DEPTHS]:
        raise RuntimeError("P2-Q prediction manifest pool-depth contract mismatch.")
    if p2q_manifest.get("exact_k_validation_result") is not True:
        raise RuntimeError("P2-Q prediction manifest does not record exact-K validation success.")
    if int(p2q_manifest.get("random_seed", -1)) != int(RANDOM_SEED):
        raise RuntimeError("P2-Q prediction manifest random-seed mismatch.")
    required_cols = [
        "query_id", "case_id", "user_id", "regime", "pool_depth", "candidate_item_id",
        "original_candidate_rank", "original_candidate_score", "rerank_score", "rerank_rank",
        "is_gt", "target_item_id", "fold_id", "reranker_method", "category_id",
    ]
    p2q = pd.read_parquet(P2Q_OOF_PREDICTION_PATH).copy()
    missing_cols = [col for col in required_cols if col not in p2q.columns]
    if missing_cols:
        raise RuntimeError(f"P2-Q prediction artifact is missing required columns: {missing_cols}")
    if not p2q["category_id"].astype(str).eq(CATEGORY_ID).all():
        raise RuntimeError("P2-Q prediction artifact category_id mismatch.")
    for col in ["query_id", "case_id", "candidate_item_id", "target_item_id"]:
        p2q[col] = p2q[col].astype(str)
    p2q["pool_depth"] = pd.to_numeric(p2q["pool_depth"], errors="raise").astype(int)
    p2q["original_candidate_rank"] = pd.to_numeric(p2q["original_candidate_rank"], errors="raise").astype(int)
    p2q["rerank_rank"] = pd.to_numeric(p2q["rerank_rank"], errors="raise").astype(int)
    p2q["original_candidate_score"] = pd.to_numeric(p2q["original_candidate_score"], errors="raise").astype(float)
    p2q["rerank_score"] = pd.to_numeric(p2q["rerank_score"], errors="raise").astype(float)
    p2q["is_gt"] = pd.to_numeric(p2q["is_gt"], errors="raise").astype(int)
    if p2q.duplicated(["query_id", "pool_depth", "candidate_item_id"]).any():
        raise RuntimeError("P2-Q prediction artifact contains duplicate query/depth/candidate rows.")
    return p2q


P2Q_OOF_PREDICTIONS_DF = load_p2q_oof_predictions()

def strict_cold_query_ids_from_pool(pool_df: pd.DataFrame) -> set:
    required_cols = ["query_id", "regime"]
    missing_cols = [col for col in required_cols if col not in pool_df.columns]
    if missing_cols:
        raise RuntimeError(f"Cold fallback check is missing required columns:{missing_cols}")
    meta = pool_df[required_cols].drop_duplicates("query_id").copy()
    cold_label = meta["regime"].astype(str).eq("cold")
    return set(meta.loc[cold_label, "query_id"].astype(str))


def apply_p2q_cold_fallback(scored_pool_df: pd.DataFrame, pool_depth: int) -> pd.DataFrame:
    out = scored_pool_df.copy().reset_index(drop=True)
    out["prediction_source"] = "prior_aware_native"
    out["p2q_fallback_rerank_score"] = np.nan
    out["p2q_fallback_rerank_rank"] = np.nan

    cold_query_ids = strict_cold_query_ids_from_pool(out)
    if not cold_query_ids:
        return out

    out["_fallback_row_id"] = np.arange(len(out), dtype=np.int64)
    cold_rows = out.loc[out["query_id"].astype(str).isin(cold_query_ids)].copy()
    p2q_depth = P2Q_OOF_PREDICTIONS_DF.loc[
        P2Q_OOF_PREDICTIONS_DF["pool_depth"].eq(int(pool_depth))
        & P2Q_OOF_PREDICTIONS_DF["query_id"].astype(str).isin(cold_query_ids)
    ].copy()
    expected_rows = len(cold_query_ids) * int(pool_depth)
    if len(cold_rows) != expected_rows or len(p2q_depth) != expected_rows:
        raise RuntimeError(
            "Cold P2-Q fallback row count mismatch at pool_depth="
            f"{pool_depth}: native={len(cold_rows)}, p2q={len(p2q_depth)}, expected={expected_rows}"
        )

    merged = cold_rows.merge(
        p2q_depth[
            [
                "query_id", "pool_depth", "candidate_item_id", "original_candidate_rank",
                "original_candidate_score", "rerank_score", "rerank_rank", "is_gt", "target_item_id",
            ]
        ],
        on=["query_id", "pool_depth", "candidate_item_id"],
        how="left",
        validate="one_to_one",
        suffixes=("", "_p2q"),
    )
    if merged["rerank_score"].isna().any():
        raise RuntimeError("Cold fallback could not match every native candidate to the P2-Q artifact.")
    if not merged["candidate_rank"].astype(int).eq(merged["original_candidate_rank"].astype(int)).all():
        raise RuntimeError("Cold fallback candidate ranks differ from P2-Q original candidate ranks.")
    if not np.allclose(
        merged["candidate_score"].astype(float).to_numpy(),
        merged["original_candidate_score"].astype(float).to_numpy(),
        rtol=1e-12,
        atol=1e-12,
    ):
        raise RuntimeError("Cold fallback candidate scores differ from P2-Q original candidate scores.")
    if not merged["is_gt"].astype(int).eq(merged["is_gt_p2q"].astype(int)).all():
        raise RuntimeError("Cold fallback target flags differ from P2-Q artifact.")
    if not merged["gt_item_id"].astype(str).eq(merged["target_item_id"].astype(str)).all():
        raise RuntimeError("Cold fallback target item IDs differ from P2-Q artifact.")

    row_ids = merged["_fallback_row_id"].to_numpy()
    # Preserve lgbm_score as the native fold-model prediction; the evaluated strict-cold fallback is stored separately.
    out.loc[row_ids, "p2q_fallback_rerank_score"] = merged["rerank_score"].astype(np.float32).to_numpy()
    out.loc[row_ids, "p2q_fallback_rerank_rank"] = merged["rerank_rank"].astype(np.int32).to_numpy()
    out.loc[row_ids, "prediction_source"] = "p2q_cold_fallback"
    out = out.drop(columns=["_fallback_row_id"], errors="ignore")
    return out


def build_cold_rerank_fallback_qc(per_query_df: pd.DataFrame) -> pd.DataFrame:
    cold_rows = per_query_df.loc[
        per_query_df["rerank_method"].astype(str).eq(OUTPUT_RERANK_METHOD)
        & per_query_df["regime"].astype(str).eq("cold")
    ].copy()
    p2q_target = P2Q_OOF_PREDICTIONS_DF.loc[
        P2Q_OOF_PREDICTIONS_DF["is_gt"].astype(int).eq(1),
        ["query_id", "pool_depth", "rerank_rank"],
    ].rename(columns={"rerank_rank": "p2q_gt_rank"})
    compare = cold_rows.merge(p2q_target, on=["query_id", "pool_depth"], how="left", validate="one_to_one")
    compare["rank_mismatch"] = pd.to_numeric(compare["gt_rank"], errors="coerce").fillna(-1).astype(int).ne(
        pd.to_numeric(compare["p2q_gt_rank"], errors="coerce").fillna(-1).astype(int)
    )
    rank_mismatch_count = int(compare["rank_mismatch"].sum())
    rows = []
    for pool_depth, group in compare.groupby("pool_depth", sort=True):
        group_rank_mismatch_count = int(group["rank_mismatch"].sum())
        rows.append({
            "category_id": CATEGORY_ID,
            "pool_depth": int(pool_depth),
            "n_cold_queries_fallback": int(group["query_id"].nunique()),
            "rank_mismatch_count": group_rank_mismatch_count,
            "cold_metric_identity_passed": bool(group_rank_mismatch_count == 0),
        })
    qc = pd.DataFrame(rows)
    if rank_mismatch_count:
        raise RuntimeError(f"Cold P2-Q fallback target ranks differ for {rank_mismatch_count} query/depth rows.")
    return qc



def build_feature_interpretation_candidate_frame(
    scored_pool_df: pd.DataFrame,
    evaluated_candidates_df: pd.DataFrame,
    pool_depth: int,
    feature_cols: list,
) -> pd.DataFrame:
    required_cols = [
        "case_id", "query_id", "candidate_item_id", "is_gt", "cv_fold", "lgbm_score",
    ]
    missing_cols = [column for column in required_cols if column not in scored_pool_df.columns]
    if missing_cols:
        raise RuntimeError(f"Feature-interpretation export is missing columns: {missing_cols}")
    if len(feature_cols) != len(set(feature_cols)):
        raise RuntimeError("Model feature names must be unique before interpretation export.")

    evaluated_required_cols = [
        "query_id", "candidate_item_id", "rerank_method", "final_score",
    ]
    missing_evaluated_cols = [
        column for column in evaluated_required_cols
        if column not in evaluated_candidates_df.columns
    ]
    if missing_evaluated_cols:
        raise RuntimeError(
            f"Evaluated candidate export is missing columns: {missing_evaluated_cols}"
        )
    evaluated_model_rows = evaluated_candidates_df.loc[
        evaluated_candidates_df["rerank_method"].astype(str).eq(OUTPUT_RERANK_METHOD),
        [
            column for column in [
                "query_id", "candidate_item_id", "final_score", "prediction_source"
            ]
            if column in evaluated_candidates_df.columns
        ],
    ].copy()
    if "prediction_source" not in evaluated_model_rows.columns:
        evaluated_model_rows["prediction_source"] = "lightgbm_oof"
    evaluated_model_rows["query_id"] = evaluated_model_rows["query_id"].astype(str)
    evaluated_model_rows["candidate_item_id"] = evaluated_model_rows["candidate_item_id"].astype(str)
    evaluated_model_rows = evaluated_model_rows.rename(
        columns={"final_score": "evaluated_oof_prediction"}
    )
    if evaluated_model_rows.duplicated(["query_id", "candidate_item_id"]).any():
        raise RuntimeError("Evaluated OOF candidate rows must be unique.")

    scored_keys_df = pd.DataFrame({
        "query_id": scored_pool_df["query_id"].astype(str).to_numpy(),
        "candidate_item_id": scored_pool_df["candidate_item_id"].astype(str).to_numpy(),
    })
    evaluated_aligned_df = scored_keys_df.merge(
        evaluated_model_rows,
        on=["query_id", "candidate_item_id"],
        how="left",
        validate="one_to_one",
    )
    if evaluated_aligned_df["evaluated_oof_prediction"].isna().any():
        raise RuntimeError("Evaluated OOF predictions do not cover every candidate.")

    metadata_df = pd.DataFrame({
        "condition_name": CONDITION_NAME,
        "category_id": CATEGORY_ID,
        "pool_depth": np.int32(pool_depth),
        "case_id": scored_pool_df["case_id"].astype(str).to_numpy(),
        "query_id": scored_pool_df["query_id"].astype(str).to_numpy(),
        "candidate_parent_asin": scored_pool_df["candidate_item_id"].astype(str).to_numpy(),
        "label": pd.to_numeric(scored_pool_df["is_gt"], errors="raise").astype(np.int8).to_numpy(),
        "fold_id": pd.to_numeric(scored_pool_df["cv_fold"], errors="raise").astype(np.int16).to_numpy(),
        "oof_prediction": pd.to_numeric(
            evaluated_aligned_df["evaluated_oof_prediction"], errors="raise"
        ).astype(np.float32).to_numpy(),
        "model_oof_prediction": pd.to_numeric(
            scored_pool_df["lgbm_score"], errors="raise"
        ).astype(np.float32).to_numpy(),
        "prediction_source": evaluated_aligned_df["prediction_source"].fillna("").astype(str).to_numpy(),
    })
    feature_values_df = scored_pool_df.loc[:, feature_cols].reset_index(drop=True).copy()
    for column in feature_cols:
        feature_values_df[column] = pd.to_numeric(
            feature_values_df[column], errors="raise"
        ).astype(np.float32)
    out = pd.concat([metadata_df.reset_index(drop=True), feature_values_df], axis=1)

    if out.columns.duplicated().any():
        raise RuntimeError("Feature-interpretation export contains duplicated columns.")
    if out[["case_id", "query_id", "candidate_parent_asin"]].eq("").any().any():
        raise RuntimeError("Feature-interpretation identifiers must be non-empty.")
    if out.duplicated(["pool_depth", "query_id", "candidate_parent_asin"]).any():
        raise RuntimeError("Feature-interpretation export contains duplicated candidate rows.")
    if not np.isfinite(out["oof_prediction"].to_numpy(dtype=np.float64)).all():
        raise RuntimeError("Feature-interpretation OOF predictions must be finite.")
    if not np.isfinite(out["model_oof_prediction"].to_numpy(dtype=np.float64)).all():
        raise RuntimeError("Feature-interpretation model OOF predictions must be finite.")
    if any(str(out[column].dtype) != "float32" for column in feature_cols):
        raise RuntimeError("Feature-interpretation feature dtypes must be float32.")
    return out


In [47]:
# ==== Train / Validate LightGBM Reranker and Apply By Pool Depth — Part 4 ====
per_query_frames = []
reranked_candidate_frames = []
model_cv_frames = []
feature_importance_frames = []
feature_interpretation_artifact_rows = []
feature_interpretation_fold_assignment_frames = []
runtime_rows = []

executed_rerank_methods = [BASELINE_RERANK_METHOD, OUTPUT_RERANK_METHOD]

for pool_depth in POOL_DEPTHS:
    pool_df, pool_feature_cols = build_pool_frame(candidate_work, pool_depth)

    fit_predict_wall_start = time.perf_counter()
    scored_pool_df, cv_summary_df, fi_df = fit_predict_ranker_one_depth(
        pool_df,
        pool_depth,
        pool_feature_cols,
    )
    scored_pool_df = apply_p2q_cold_fallback(scored_pool_df, pool_depth)
    fit_predict_wall_clock_runtime_sec = float(time.perf_counter() - fit_predict_wall_start)

    scoring_runtime_sec = float(scored_pool_df.attrs.get("rerank_scoring_runtime_sec", 0.0))
    training_runtime_sec = float(scored_pool_df.attrs.get("training_runtime_sec", 0.0))

    n_queries_runtime = int(scored_pool_df["query_id"].nunique())
    n_candidates_runtime = int(len(scored_pool_df))

    # Record LightGBM training and scoring components at the report depth using the
    # same step names consumed by Notebook 10's runtime export frame.
    if int(pool_depth) == int(RUNTIME_REPORT_POOL_DEPTH):
        record_runtime_step(
            "model_fit_or_tuning",
            training_runtime_sec,
            pool_depth=pool_depth,
            n_queries=n_queries_runtime,
            n_candidates=n_candidates_runtime,
            runtime_measurement_type="component_runtime",
            derived_from_existing_runtime=True,
            rerank_method=OUTPUT_RERANK_METHOD,
            model_fit_predict_wall_clock_runtime_sec=fit_predict_wall_clock_runtime_sec,
        )
        record_runtime_step(
            "model_scoring",
            scoring_runtime_sec,
            pool_depth=pool_depth,
            n_queries=n_queries_runtime,
            n_candidates=n_candidates_runtime,
            runtime_measurement_type="component_runtime",
            derived_from_existing_runtime=True,
            rerank_method=OUTPUT_RERANK_METHOD,
        )

    rank_timer_start = time.perf_counter()
    lgbm_ranked_for_runtime = scored_pool_df.sort_values(
        ["query_id", "lgbm_score", "candidate_score_norm_pool", "candidate_rank", "candidate_item_id"],
        ascending=[True, False, False, True, True],
        kind="mergesort",
    ).copy()
    lgbm_ranked_for_runtime["rerank_rank"] = (
        lgbm_ranked_for_runtime.groupby("query_id").cumcount() + 1
    )
    lgbm_rank_runtime_sec = float(time.perf_counter() - rank_timer_start)

    del lgbm_ranked_for_runtime

    if int(pool_depth) == int(RUNTIME_REPORT_POOL_DEPTH):
        record_runtime_step(
            "ranking_sorting",
            lgbm_rank_runtime_sec,
            pool_depth=pool_depth,
            n_queries=n_queries_runtime,
            n_candidates=n_candidates_runtime,
            runtime_measurement_type="component_runtime",
            derived_from_existing_runtime=True,
            rerank_method=OUTPUT_RERANK_METHOD,
        )

    eval_timer_start = time.perf_counter()
    pq_df_one, rerank_df_one = export_and_score_methods(scored_pool_df, pool_depth)
    eval_runtime_sec = float(time.perf_counter() - eval_timer_start)


    feature_interpretation_df = build_feature_interpretation_candidate_frame(
        scored_pool_df,
        rerank_df_one,
        pool_depth,
        pool_feature_cols,
    )
    feature_interpretation_path = OUT_DIR / FEATURE_INTERPRETATION_CANDIDATE_FILENAME.format(
        pool_depth=int(pool_depth)
    )
    feature_interpretation_df.to_parquet(feature_interpretation_path, index=False)
    feature_interpretation_artifact_rows.append({
        "pool_depth": int(pool_depth),
        "path": str(feature_interpretation_path),
        "row_count": int(len(feature_interpretation_df)),
        "case_count": int(feature_interpretation_df["case_id"].nunique()),
        "feature_count": int(len(pool_feature_cols)),
    })
    feature_interpretation_fold_assignment_frames.append(
        feature_interpretation_df[
            ["condition_name", "category_id", "pool_depth", "case_id", "query_id", "fold_id"]
        ].drop_duplicates().copy()
    )
    del feature_interpretation_df

    if int(pool_depth) == int(RUNTIME_REPORT_POOL_DEPTH):
        record_runtime_step(
            "evaluation",
            eval_runtime_sec,
            pool_depth=pool_depth,
            n_queries=n_queries_runtime,
            n_candidates=n_candidates_runtime,
            runtime_measurement_type="component_runtime",
            derived_from_existing_runtime=True,
            rerank_method=OUTPUT_RERANK_METHOD,
        )

    for reranker_name, runtime_sec, scoring_sec, sorting_sec, model_type in [
        (OUTPUT_RERANK_METHOD, scoring_runtime_sec + lgbm_rank_runtime_sec, scoring_runtime_sec, lgbm_rank_runtime_sec, "lightgbm"),
    ]:
        runtime_rows.append(
            {
                "method": RERANKING_METHOD,
                "reranker": str(reranker_name),
                "pool_depth": int(pool_depth),
                "candidate_pool_depth": int(pool_depth),
                "n_queries": n_queries_runtime,
                "n_candidates": n_candidates_runtime,
                "feature_preparation_runtime_sec": np.nan,
                "model_scoring_runtime_sec": float(scoring_sec),
                "ranking_sorting_runtime_sec": float(sorting_sec),
                "online_operation_runtime_sec": float(runtime_sec),
                "runtime_sec_per_query": float(runtime_sec / n_queries_runtime) if n_queries_runtime else np.nan,
                "runtime_sec_per_candidate": float(runtime_sec / n_candidates_runtime) if n_candidates_runtime else np.nan,
                "queries_per_second": float(n_queries_runtime / runtime_sec) if runtime_sec > 0 else np.nan,
                "candidates_per_second": float(n_candidates_runtime / runtime_sec) if runtime_sec > 0 else np.nan,
                "rerank_runtime_sec": float(runtime_sec),
                "rerank_runtime_sec_per_query": float(runtime_sec / n_queries_runtime) if n_queries_runtime else np.nan,
                "rerank_runtime_sec_per_candidate": float(runtime_sec / n_candidates_runtime) if n_candidates_runtime else np.nan,
                "training_runtime_sec": training_runtime_sec if reranker_name == OUTPUT_RERANK_METHOD else 0.0,
                "model_fit_or_tuning_runtime_sec": training_runtime_sec if reranker_name == OUTPUT_RERANK_METHOD else 0.0,
                "eval_runtime_sec": eval_runtime_sec,
                "model_type": model_type,
                "category_id": CATEGORY_ID,
                "category_folder": CATEGORY_FOLDER,
                "candidate_pool_type": CANDIDATE_POOL_TYPE,
                "retrieval_method": RETRIEVAL_METHOD,
                "retrieval_method_label": RETRIEVAL_METHOD_LABEL,
                "reranking_method": RERANKING_METHOD,
                "runtime_scope": "stage2_online_reranking_excluding_stage1_retrieval",
            }
        )

    pool_nonnull_rate = float(scored_pool_df["lgbm_score"].notna().mean())
    print(f"pool_depth={pool_depth} | lgbm_score non-null rate:", round(pool_nonnull_rate, 6))

    per_query_frames.append(pq_df_one)
    model_cv_frames.append(cv_summary_df)
    feature_importance_frames.append(fi_df)

    if int(pool_depth) == int(REPORT_POOL_DEPTH):
        reranked_candidate_frames.append(rerank_df_one.copy())
    del rerank_df_one

    del pool_df, scored_pool_df, cv_summary_df, fi_df, pq_df_one
    gc.collect()

per_query_results_df = pd.concat(per_query_frames, ignore_index=True)
reranked_candidates_df = pd.concat(reranked_candidate_frames, ignore_index=True)
model_cv_summary_df = pd.concat(model_cv_frames, ignore_index=True)
runtime_by_pool_depth_df = pd.DataFrame(runtime_rows)

# Ensure the per-depth runtime export has the common online-runtime schema.
if not runtime_by_pool_depth_df.empty:
    if "candidate_pool_depth" not in runtime_by_pool_depth_df.columns:
        runtime_by_pool_depth_df["candidate_pool_depth"] = pd.to_numeric(runtime_by_pool_depth_df["pool_depth"], errors="coerce").astype("Int64")
    if "method" not in runtime_by_pool_depth_df.columns:
        runtime_by_pool_depth_df["method"] = RERANKING_METHOD
    if "online_operation_runtime_sec" not in runtime_by_pool_depth_df.columns:
        runtime_by_pool_depth_df["online_operation_runtime_sec"] = runtime_by_pool_depth_df.get("rerank_runtime_sec", np.nan)
    if "runtime_sec_per_query" not in runtime_by_pool_depth_df.columns:
        runtime_by_pool_depth_df["runtime_sec_per_query"] = runtime_by_pool_depth_df.get("rerank_runtime_sec_per_query", np.nan)
    if "runtime_sec_per_candidate" not in runtime_by_pool_depth_df.columns:
        runtime_by_pool_depth_df["runtime_sec_per_candidate"] = runtime_by_pool_depth_df.get("rerank_runtime_sec_per_candidate", np.nan)
    if "queries_per_second" not in runtime_by_pool_depth_df.columns:
        runtime_by_pool_depth_df["queries_per_second"] = np.where(
            pd.to_numeric(runtime_by_pool_depth_df["online_operation_runtime_sec"], errors="coerce") > 0,
            pd.to_numeric(runtime_by_pool_depth_df["n_queries"], errors="coerce") / pd.to_numeric(runtime_by_pool_depth_df["online_operation_runtime_sec"], errors="coerce"),
            np.nan,
        )
    if "candidates_per_second" not in runtime_by_pool_depth_df.columns:
        runtime_by_pool_depth_df["candidates_per_second"] = np.where(
            pd.to_numeric(runtime_by_pool_depth_df["online_operation_runtime_sec"], errors="coerce") > 0,
            pd.to_numeric(runtime_by_pool_depth_df["n_candidates"], errors="coerce") / pd.to_numeric(runtime_by_pool_depth_df["online_operation_runtime_sec"], errors="coerce"),
            np.nan,
        )
    if "feature_preparation_runtime_sec" not in runtime_by_pool_depth_df.columns:
        runtime_by_pool_depth_df["feature_preparation_runtime_sec"] = np.nan
    if "model_fit_or_tuning_runtime_sec" not in runtime_by_pool_depth_df.columns and "fitting_runtime_sec" in runtime_by_pool_depth_df.columns:
        runtime_by_pool_depth_df["model_fit_or_tuning_runtime_sec"] = runtime_by_pool_depth_df["fitting_runtime_sec"]


reranked_candidates_report_df = reranked_candidates_df[
    pd.to_numeric(reranked_candidates_df["pool_depth"], errors="coerce").eq(int(RUNTIME_REPORT_POOL_DEPTH))
].copy()
per_query_report_df = per_query_results_df[
    pd.to_numeric(per_query_results_df["pool_depth"], errors="coerce").eq(int(RUNTIME_REPORT_POOL_DEPTH))
].copy()

if "rerank_method" not in reranked_candidates_df.columns:
    raise RuntimeError("reranked_candidates_df must contain rerank_method after export_and_score_methods().")

del per_query_frames, reranked_candidate_frames, model_cv_frames
gc.collect()

if feature_importance_frames and any(len(df) > 0 for df in feature_importance_frames):
    feature_importance_df = pd.concat(feature_importance_frames, ignore_index=True)
else:
    feature_importance_df = pd.DataFrame(columns=["pool_depth", "fold", "feature", "gain", "split"])

del feature_importance_frames
gc.collect()

if not feature_importance_df.empty:
    feature_importance_summary_df = (
        feature_importance_df.groupby("feature", dropna=False)
        .agg(
            gain_mean=("gain", "mean"),
            gain_sum=("gain", "sum"),
            split_sum=("split", "sum"),
            depth_n=("pool_depth", "nunique"),
            fold_n=("fold", "nunique"),
        )
        .reset_index()
        .sort_values(["gain_mean", "split_sum", "feature"], ascending=[False, False, True])
        .reset_index(drop=True)
    )
else:
    feature_importance_summary_df = pd.DataFrame(
        columns=["feature", "gain_mean", "gain_sum", "split_sum", "depth_n", "fold_n"]
    )

print("feature_importance_df shape:", feature_importance_df.shape)
print("feature_importance_summary_df shape:", feature_importance_summary_df.shape)

actual_rerank_methods = sorted(per_query_results_df["rerank_method"].unique().tolist())
if actual_rerank_methods != sorted(executed_rerank_methods):
    raise RuntimeError(f"Unexpected rerank methods: {actual_rerank_methods}")

print("per_query_results_df shape:", per_query_results_df.shape)
print("reranked_candidates_df shape:", reranked_candidates_df.shape)
print("reranked_candidates_report_df shape:", reranked_candidates_report_df.shape)
print("model_cv_summary_df shape:", model_cv_summary_df.shape)
print("feature_importance_df shape:", feature_importance_df.shape)
print("runtime_by_pool_depth_df shape:", runtime_by_pool_depth_df.shape)
print("executed rerank methods:", actual_rerank_methods)
cold_rerank_fallback_qc_df = build_cold_rerank_fallback_qc(per_query_results_df)
display(runtime_by_pool_depth_df.head(20))


/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:969: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Total Bins 4627
[LightGBM] [Info] Number of data points in the train set: 10900, number of used features: 53
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:969: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Total Bins 4391
[LightGBM] [Info] Number of data points in the train set: 10100, number of used features: 53
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:969: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


...

[LightGBM] [Info] Total Bins 6596
[LightGBM] [Info] Number of data points in the train set: 309000, number of used features: 53


stage1_baseline @ 1000:   0%|          | 0/2288 [00:00<?, ?it/s]

lightgbm_rerank @ 1000:   0%|          | 0/2288 [00:00<?, ?it/s]

pool_depth=1000 | lgbm_score non-null rate: 1.0
feature_importance_df shape: (1400, 5)
feature_importance_summary_df shape: (56, 6)
per_query_results_df shape: (22880, 59)
reranked_candidates_df shape: (4576000, 67)
reranked_candidates_report_df shape: (4576000, 67)
model_cv_summary_df shape: (25, 16)
feature_importance_df shape: (1400, 5)
runtime_by_pool_depth_df shape: (5, 28)
executed rerank methods: ['lightgbm_rerank', 'stage1_baseline']


,method,reranker,pool_depth,candidate_pool_depth,n_queries,n_candidates,feature_preparation_runtime_sec,model_scoring_runtime_sec,ranking_sorting_runtime_sec,online_operation_runtime_sec,runtime_sec_per_query,runtime_sec_per_candidate,queries_per_second,candidates_per_second,rerank_runtime_sec,rerank_runtime_sec_per_query,rerank_runtime_sec_per_candidate,training_runtime_sec,model_fit_or_tuning_runtime_sec,eval_runtime_sec,model_type,category_id,category_folder,candidate_pool_type,retrieval_method,retrieval_method_label,reranking_method,runtime_scope
0,lightgbm,lightgbm_rerank,100,100,2288,228800,NaN,0.199643,0.410447,0.610091,0.000267,0.000003,3750.262665,375026.266546,0.610091,0.000267,0.000003,2.785101,2.785101,194.566949,lightgbm,face,facial_skincare,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,lightgbm,stage2_online_reranking_excluding_stage1_retri...
1,lightgbm,lightgbm_rerank,300,300,2288,686400,NaN,0.429506,1.257370,1.686876,0.000737,0.000002,1356.353267,406905.980066,1.686876,0.000737,0.000002,5.693577,5.693577,218.300141,lightgbm,face,facial_skincare,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,lightgbm,stage2_online_reranking_excluding_stage1_retri...
2,lightgbm,lightgbm_rerank,500,500,2288,1144000,NaN,1.189124,2.372857,3.561982,0.001557,0.000003,642.339087,321169.543629,3.561982,0.001557,0.000003,9.574478,9.574478,267.639136,lightgbm,face,facial_skincare,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,lightgbm,stage2_online_reranking_excluding_stage1_retri...
3,lightgbm,lightgbm_rerank,700,700,2288,1601600,NaN,1.647226,3.521249,5.168475,0.002259,0.000003,442.683800,309878.660213,5.168475,0.002259,0.000003,20.777379,20.777379,339.709789,lightgbm,face,facial_skincare,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,lightgbm,stage2_online_reranking_excluding_stage1_retri...
4,lightgbm,lightgbm_rerank,1000,1000,2288,2288000,NaN,2.669569,5.236509,7.906078,0.003455,0.000003,289.397594,289397.594316,7.906078,0.003455,0.000003,32.059093,32.059093,403.236823,lightgbm,face,facial_skincare,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,lightgbm,stage2_online_reranking_excluding_stage1_retri...


## 9. Evaluate Baseline vs LightGBM Reranked Results

In [48]:
# ==== Evaluate Baseline Vs LightGBM Reranked Results ====
metric_cols = [c for c in per_query_results_df.columns if re.match(r"^(HitRate|MRR|NDCG)@\d+$", c)]
supp_metric_cols_existing = [c for c in SUPPLEMENTARY_METRIC_COLS if c in per_query_results_df.columns]

per_query_final_eval_df = per_query_results_df.copy()
if per_query_final_eval_df.empty:
    raise RuntimeError("Final evaluation frame is empty.")

REPORT_POOL_DEPTH = RUNTIME_REPORT_POOL_DEPTH
EXPECTED_POOL_DEPTHS = sorted(int(depth) for depth in POOL_DEPTHS)

if REPORT_POOL_DEPTH not in EXPECTED_POOL_DEPTHS:
    raise RuntimeError(f"REPORT_POOL_DEPTH={REPORT_POOL_DEPTH} is not present in POOL_DEPTHS={EXPECTED_POOL_DEPTHS}.")

per_query_report_df = per_query_final_eval_df[
    pd.to_numeric(per_query_final_eval_df["pool_depth"], errors="coerce").eq(REPORT_POOL_DEPTH)
].copy()

if per_query_report_df.empty:
    raise RuntimeError(f"Report pool-depth dataframe is empty for pool_depth={REPORT_POOL_DEPTH}.")

report_pool_values = (
    pd.to_numeric(per_query_report_df["pool_depth"], errors="coerce")
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)
if len(report_pool_values) != 1 or int(report_pool_values[0]) != REPORT_POOL_DEPTH:
    raise RuntimeError(
        f"Report summary must use only pool_depth={REPORT_POOL_DEPTH}; found {sorted(report_pool_values)}."
    )

full_pool_values = sorted(
    pd.to_numeric(per_query_final_eval_df["pool_depth"], errors="coerce")
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)
if full_pool_values != EXPECTED_POOL_DEPTHS:
    raise RuntimeError(
        f"Full per-query evaluation dataframe must preserve pool_depth values {EXPECTED_POOL_DEPTHS}; found {full_pool_values}."
    )

per_query_metrics_pool_values = sorted(
    pd.to_numeric(per_query_results_df["pool_depth"], errors="coerce")
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)
if per_query_metrics_pool_values != EXPECTED_POOL_DEPTHS:
    raise RuntimeError(
        f"per_query_metrics.parquet would not preserve all pool_depth values {EXPECTED_POOL_DEPTHS}; "
        f"found {per_query_metrics_pool_values}."
    )

summary_overall_df = summarize_metrics(per_query_report_df, ["rerank_method"])
summary_by_pool_depth_df = summarize_metrics(per_query_final_eval_df, ["pool_depth", "rerank_method"])

if not runtime_by_pool_depth_df.empty:
    runtime_merge_cols = [
        "pool_depth",
        "reranker",
        "rerank_runtime_sec",
        "rerank_runtime_sec_per_query",
        "rerank_runtime_sec_per_candidate",
    ]
    summary_by_pool_depth_df = (
        summary_by_pool_depth_df.merge(
            runtime_by_pool_depth_df[runtime_merge_cols],
            left_on=["pool_depth", "rerank_method"],
            right_on=["pool_depth", "reranker"],
            how="left",
        )
        .drop(columns=["reranker"], errors="ignore")
    )

if "regime" in per_query_report_df.columns:
    summary_by_regime_df = summarize_metrics(per_query_report_df, ["regime", "rerank_method"])
else:
    summary_by_regime_df = pd.DataFrame(columns=["regime", "rerank_method", "n_queries"])

if "regime" in per_query_final_eval_df.columns:
    summary_by_regime_pool_depth_df = summarize_metrics(
        per_query_final_eval_df,
        ["regime", "pool_depth", "rerank_method"],
    )
else:
    summary_by_regime_pool_depth_df = pd.DataFrame(columns=["regime", "pool_depth", "rerank_method", "n_queries"])

def add_summary_scope_metadata(df: pd.DataFrame, summary_scope: str) -> pd.DataFrame:
    work = df.copy()
    work["report_pool_depth"] = REPORT_POOL_DEPTH
    work["summary_scope"] = summary_scope
    work["condition"] = EXPERIMENT_CONDITION
    work["candidate_source"] = CANDIDATE_SOURCE_LABEL
    work["candidate_pool_type"] = CANDIDATE_POOL_TYPE
    work["retrieval_method"] = RETRIEVAL_METHOD
    work["retrieval_method_label"] = RETRIEVAL_METHOD_LABEL
    work["reranking_method"] = RERANKING_METHOD
    work["comparison_set"] = "native"
    work["category_id"] = CATEGORY_ID
    return work

summary_overall_df = add_summary_scope_metadata(summary_overall_df, f"pool_depth_{REPORT_POOL_DEPTH}")
summary_by_regime_df = add_summary_scope_metadata(summary_by_regime_df, f"pool_depth_{REPORT_POOL_DEPTH}")
summary_by_pool_depth_df = add_summary_scope_metadata(summary_by_pool_depth_df, "all_pool_depths")
summary_by_regime_pool_depth_df = add_summary_scope_metadata(summary_by_regime_pool_depth_df, "all_pool_depths")

if summary_overall_df.empty or not summary_overall_df["summary_scope"].eq(f"pool_depth_{REPORT_POOL_DEPTH}").all():
    raise RuntimeError(f"results_overall.csv must be generated from the pool_depth {REPORT_POOL_DEPTH} report dataframe.")

if not summary_by_regime_df.empty and not summary_by_regime_df["summary_scope"].eq(f"pool_depth_{REPORT_POOL_DEPTH}").all():
    raise RuntimeError(f"results_by_regime.csv must be generated from the pool_depth {REPORT_POOL_DEPTH} report dataframe.")

summary_pool_values = (
    sorted(pd.to_numeric(summary_by_pool_depth_df["pool_depth"], errors="coerce").dropna().astype(int).unique().tolist())
    if "pool_depth" in summary_by_pool_depth_df.columns
    else []
)
if summary_pool_values != EXPECTED_POOL_DEPTHS:
    raise RuntimeError(
        f"results_by_pool_depth.csv must preserve pool_depth values {EXPECTED_POOL_DEPTHS}; found {summary_pool_values}."
    )

summary_regime_pool_values = (
    sorted(
        pd.to_numeric(summary_by_regime_pool_depth_df["pool_depth"], errors="coerce")
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    if "pool_depth" in summary_by_regime_pool_depth_df.columns
    else []
)
if "regime" in per_query_final_eval_df.columns and summary_regime_pool_values != EXPECTED_POOL_DEPTHS:
    raise RuntimeError(
        f"results_by_regime_pool_depth.csv must preserve pool_depth values {EXPECTED_POOL_DEPTHS} "
        f"when regime exists; found {summary_regime_pool_values}."
    )

def pool_depth_value_string(df: pd.DataFrame) -> str:
    if df is None or df.empty or "pool_depth" not in df.columns:
        return ""
    values = sorted(pd.to_numeric(df["pool_depth"], errors="coerce").dropna().astype(int).unique().tolist())
    return ",".join(str(value) for value in values)

def qc_row(
    output_file: str,
    expected_scope: str,
    actual_scope: str,
    source_dataframe: str,
    df: pd.DataFrame,
    warning_message: str = "",
) -> dict:
    return {
        "notebook_name": NOTEBOOK_NAME,
        "output_file": output_file,
        "expected_scope": expected_scope,
        "actual_scope": actual_scope,
        "source_dataframe": source_dataframe,
        "pool_depth_values": pool_depth_value_string(df),
        "n_rows": int(len(df)) if df is not None else 0,
        "n_queries": (
            int(df["query_id"].nunique())
            if df is not None and "query_id" in df.columns
            else int(pd.to_numeric(df["n_queries"], errors="coerce").max())
            if df is not None and "n_queries" in df.columns and len(df)
            else 0
        ),
        "check_passed": bool(not warning_message),
        "warning_message": warning_message,
    }

report_summary_scope_qc_df = pd.DataFrame(
    [
        qc_row("results_overall.csv", f"pool_depth_{REPORT_POOL_DEPTH}", f"pool_depth_{REPORT_POOL_DEPTH}", "per_query_report_df", per_query_report_df),
        qc_row(
            "results_by_regime.csv",
            f"pool_depth_{REPORT_POOL_DEPTH}",
            f"pool_depth_{REPORT_POOL_DEPTH}",
            "per_query_report_df",
            per_query_report_df if "regime" in per_query_report_df.columns else summary_by_regime_df,
        ),
        qc_row("results_by_pool_depth.csv", "all_pool_depths", "all_pool_depths", "per_query_final_eval_df", per_query_final_eval_df),
        qc_row(
            "results_by_regime_pool_depth.csv",
            "all_pool_depths",
            "all_pool_depths",
            "per_query_final_eval_df",
            per_query_final_eval_df if "regime" in per_query_final_eval_df.columns else summary_by_regime_pool_depth_df,
        ),
        qc_row("per_query_metrics.parquet", "all_pool_depths", "all_pool_depths", "per_query_results_df", per_query_results_df),
    ]
)

print(f"results_overall.csv basis: pool_depth {REPORT_POOL_DEPTH}")
print(f"results_by_regime.csv basis: pool_depth {REPORT_POOL_DEPTH}")
print("results_by_pool_depth.csv basis: all pool depths")
print("results_by_regime_pool_depth.csv basis: all pool depths")

print("n_queries by rerank_method in per_query_report_df:")
display(per_query_report_df.groupby("rerank_method", dropna=False)["query_id"].nunique().reset_index(name="n_queries"))

print("pool_depth value counts in per_query_report_df:")
display(per_query_report_df["pool_depth"].value_counts(dropna=False).sort_index().reset_index(name="row_count"))

print("pool_depth value counts in the full long-format per-query dataframe:")
display(per_query_results_df["pool_depth"].value_counts(dropna=False).sort_index().reset_index(name="row_count"))

def build_uplift_summary(df, group_cols):
    group_cols = list(group_cols)
    key_cols = group_cols + ["query_id"]
    value_cols = metric_cols + supp_metric_cols_existing

    baseline_ref = df.loc[
        df["rerank_method"].astype(str).eq(BASELINE_RERANK_METHOD),
        key_cols + value_cols,
    ]
    tuned_ref = df.loc[
        df["rerank_method"].astype(str).eq(OUTPUT_RERANK_METHOD),
        key_cols + value_cols,
    ]

    if "pool_depth" not in group_cols:
        baseline_ref = baseline_ref.groupby(key_cols, dropna=False, as_index=False)[value_cols].mean()
        tuned_ref = tuned_ref.groupby(key_cols, dropna=False, as_index=False)[value_cols].mean()

    merged = tuned_ref.merge(
        baseline_ref,
        on=key_cols,
        how="inner",
        validate="one_to_one",
        suffixes=("_tuned", "_baseline"),
    )

    delta_cols = []
    for col in value_cols:
        delta_col = f"delta_{col}"
        merged[delta_col] = (
            pd.to_numeric(merged[f"{col}_tuned"], errors="coerce").fillna(0.0)
            - pd.to_numeric(merged[f"{col}_baseline"], errors="coerce").fillna(0.0)
        )
        delta_cols.append(delta_col)

    if group_cols:
        uplift_df = summarize_delta(merged[key_cols + delta_cols], group_cols)
    else:
        uplift_row = {"n_queries": int(merged["query_id"].nunique())}
        for col in delta_cols:
            uplift_row[col] = float(pd.to_numeric(merged[col], errors="coerce").fillna(0.0).mean())
        uplift_df = pd.DataFrame([uplift_row])

    del baseline_ref, tuned_ref, merged
    gc.collect()

    return uplift_df

uplift_overall_df = build_uplift_summary(per_query_report_df, [])
uplift_by_pool_depth_df = build_uplift_summary(per_query_final_eval_df, ["pool_depth"])
uplift_by_regime_df = build_uplift_summary(per_query_report_df, ["regime"])
uplift_by_regime_pool_depth_df = build_uplift_summary(per_query_final_eval_df, ["regime", "pool_depth"])

expected_rerank_methods = list(EXPECTED_RERANK_METHODS)
executed_methods = sorted(per_query_results_df["rerank_method"].astype(str).unique().tolist())
if executed_methods != sorted(expected_rerank_methods):
    raise RuntimeError(f"Unexpected rerank methods found: {executed_methods}")

print("Final evaluation overall summary:")
display(summary_overall_df)

print("Final evaluation by pool depth:")
display(summary_by_pool_depth_df)

print("Final evaluation by regime and pool depth:")
display(summary_by_regime_pool_depth_df.head(40))

print("Uplift vs baseline by pool depth:")
display(uplift_by_pool_depth_df)

results_overall.csv basis: pool_depth 1000
results_by_regime.csv basis: pool_depth 1000
results_by_pool_depth.csv basis: all pool depths
results_by_regime_pool_depth.csv basis: all pool depths
n_queries by rerank_method in per_query_report_df:


,rerank_method,n_queries
0,lightgbm_rerank,2288
1,stage1_baseline,2288


pool_depth value counts in per_query_report_df:


,pool_depth,row_count
0,1000,4576


pool_depth value counts in the full long-format per-query dataframe:


...

,pool_depth,rerank_method,n_queries,HitRate@1,MRR@1,NDCG@1,HitRate@5,MRR@5,NDCG@5,HitRate@10,MRR@10,NDCG@10,HitRate@100,MRR@100,NDCG@100,HitRate@300,MRR@300,NDCG@300,HitRate@500,MRR@500,NDCG@500,HitRate@700,MRR@700,NDCG@700,HitRate@1000,MRR@1000,NDCG@1000,weighted_facet_overlap_at_5,brand_match_at_5,concern_match_at_5,ingredient_match_at_5,skin_type_match_at_5,form_match_at_5,benefit_match_at_5,claim_match_at_5,scent_match_at_5,category_or_product_type_match_at_5,rerank_runtime_sec,rerank_runtime_sec_per_query,rerank_runtime_sec_per_candidate,report_pool_depth,summary_scope,condition,candidate_source,candidate_pool_type,retrieval_method,retrieval_method_label,reranking_method,comparison_set,category_id
0,100,lightgbm_rerank,2288,0.026661,0.026661,0.026661,0.050262,0.035242,0.038957,0.057692,0.036267,0.041396,0.080857,0.037442,0.046599,0.080857,0.037442,0.046599,0.080857,0.037442,0.046599,0.080857,0.037442,0.046599,0.080857,0.037442,0.046599,0.342951,0.030130,0.744472,0.303557,0.503552,0.324552,NaN,NaN,0.138107,0.574872,0.610091,0.000267,0.000003,1000,all_pool_depths,full_pipeline_personalization,notebook09_personalized_winner_top1000_face,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,lightgbm,native,face
1,100,stage1_baseline,2288,0.005682,0.005682,0.005682,0.014423,0.008661,0.010081,0.018794,0.009162,0.011412,0.080857,0.011039,0.023294,0.080857,0.011039,0.023294,0.080857,0.011039,0.023294,0.080857,0.011039,0.023294,0.080857,0.011039,0.023294,0.309024,0.009969,0.693585,0.255624,0.443439,0.334907,NaN,NaN,0.099913,0.580196,NaN,NaN,NaN,1000,all_pool_depths,full_pipeline_personalization,notebook09_personalized_winner_top1000_face,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,lightgbm,native,face
2,300,lightgbm_rerank,2288,0.024476,0.024476,0.024476,0.057255,0.036218,0.041446,0.071241,0.038120,0.046004,0.121941,0.040263,0.056683,0.132430,0.040328,0.058102,0.132430,0.040328,0.058102,0.132430,0.040328,0.058102,0.132430,0.040328,0.058102,0.345681,0.035576,0.750525,0.289293,0.508965,0.324592,NaN,NaN,0.141350,0.579355,1.686876,0.000737,0.000002,1000,all_pool_depths,full_pipeline_personalization,notebook09_personalized_winner_top1000_face,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,lightgbm,native,face
3,300,stage1_baseline,2288,0.005682,0.005682,0.005682,0.014423,0.008661,0.010081,0.018794,0.009162,0.011412,0.080857,0.011039,0.023294,0.132430,0.011341,0.030198,0.132430,0.011341,0.030198,0.132430,0.011341,0.030198,0.132430,0.011341,0.030198,0.309024,0.009969,0.693585,0.255624,0.443439,0.334907,NaN,NaN,0.099913,0.580196,NaN,NaN,NaN,1000,all_pool_depths,full_pipeline_personalization,notebook09_personalized_winner_top1000_face,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,lightgbm,native,face
4,500,lightgbm_rerank,2288,0.030157,0.030157,0.030157,0.062063,0.041594,0.046669,0.077797,0.043649,0.051712,0.145542,0.046303,0.065647,0.169143,0.046454,0.068863,0.172640,0.046463,0.069270,0.172640,0.046463,0.069270,0.172640,0.046463,0.069270,0.354894,0.036589,0.763557,0.283414,0.530641,0.326499,NaN,NaN,0.134096,0.576832,3.561982,0.001557,0.000003,1000,all_pool_depths,full_pipeline_personalization,notebook09_personalized_winner_top1000_face,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,lightgbm,native,face
5,500,stage1_baseline,2288,0.005682,0.005682,0.005682,0.014423,0.008661,0.010081,0.018794,0.009162,0.011412,0.080857,0.011039,0.023294,0.132430,0.011341,0.030198,0.172640,0.011443,0.034852,0.172640,0.011443,0.034852,0.172640,0.011443,0.034852,0.309024,0.009969,0.693585,0.255624,0.443439,0.334907,NaN,NaN,0.099913,0.580196,NaN,NaN,NaN,1000,all_pool_depths,full_pipeline_personalization,notebook09_personalized_winner_top1000_face,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,lightgbm,native,face
6,700,lightgbm_rerank,2288,0.027972,0.027972,0.027972,0.066434,0.042526,0.048496,0.083479,0.044765,0.053971,0.168706,0.047972,0.071333,0.197552,0.048156,0.075256,0.202797,0.0481

Final evaluation by regime and pool depth:


,regime,pool_depth,rerank_method,n_queries,HitRate@1,MRR@1,NDCG@1,HitRate@5,MRR@5,NDCG@5,HitRate@10,MRR@10,NDCG@10,HitRate@100,MRR@100,NDCG@100,HitRate@300,MRR@300,NDCG@300,HitRate@500,MRR@500,NDCG@500,HitRate@700,MRR@700,NDCG@700,HitRate@1000,MRR@1000,NDCG@1000,weighted_facet_overlap_at_5,brand_match_at_5,concern_match_at_5,ingredient_match_at_5,skin_type_match_at_5,form_match_at_5,benefit_match_at_5,claim_match_at_5,scent_match_at_5,category_or_product_type_match_at_5,report_pool_depth,summary_scope,condition,candidate_source,candidate_pool_type,retrieval_method,retrieval_method_label,reranking_method,comparison_set,category_id
0,cold,100,lightgbm_rerank,572,0.013986,0.013986,0.013986,0.024476,0.017890,0.019522,0.027972,0.018334,0.020631,0.050699,0.019489,0.025718,0.050699,0.019489,0.025718,0.050699,0.019489,0.025718,0.050699,0.019489,0.025718,0.050699,0.019489,0.025718,0.330919,0.015552,0.737483,0.274025,0.492324,0.317684,NaN,NaN,0.142369,0.578860,1000,all_pool_depths,full_pipeline_personalization,notebook09_personalized_winner_top1000_face,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,lightgbm,native,face
1,cold,100,stage1_baseline,572,0.000000,0.000000,0.000000,0.003497,0.001224,0.001779,0.006993,0.001573,0.002790,0.050699,0.003185,0.011675,0.050699,0.003185,0.011675,0.050699,0.003185,0.011675,0.050699,0.003185,0.011675,0.050699,0.003185,0.011675,0.308425,0.008499,0.696888,0.245562,0.430612,0.322179,NaN,NaN,0.092463,0.589474,1000,all_pool_depths,full_pipeline_personalization,notebook09_personalized_winner_top1000_face,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,lightgbm,native,face
2,cold,300,lightgbm_rerank,572,0.010490,0.010490,0.010490,0.027972,0.016230,0.019127,0.033217,0.017020,0.020915,0.087413,0.019143,0.032072,0.097902,0.019203,0.033472,0.097902,0.019203,0.033472,0.097902,0.019203,0.033472,0.097902,0.019203,0.033472,0.330655,0.014165,0.745917,0.244701,0.470799,0.311040,NaN,NaN,0.145758,0.581667,1000,all_pool_depths,full_pipeline_personalization,notebook09_personalized_winner_top1000_face,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,lightgbm,native,face
3,cold,300,stage1_baseline,572,0.000000,0.000000,0.000000,0.003497,0.001224,0.001779,0.006993,0.001573,0.002790,0.050699,0.003185,0.011675,0.097902,0.003451,0.017946,0.097902,0.003451,0.017946,0.097902,0.003451,0.017946,0.097902,0.003451,0.017946,0.308425,0.008499,0.696888,0.245562,0.430612,0.322179,NaN,NaN,0.092463,0.589474,1000,all_pool_depths,full_pipeline_personalization,notebook09_personalized_winner_top1000_face,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,lightgbm,native,face
4,cold,500,lightgbm_rerank,572,0.013986,0.013986,0.013986,0.022727,0.015909,0.017521,0.033217,0.017256,0.020861,0.096154,0.019456,0.033339,0.125874,0.019656,0.037438,0.132867,0.019674,0.038252,0.132867,0.019674,0.038252,0.132867,0.019674,0.038252,0.337955,0.011664,0.754953,0.260646,0.482114,0.311935,NaN,NaN,0.132186,0.560000,1000,all_pool_depths,full_pipeline_personalization,notebook09_personalized_winner_top1000_face,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,lightgbm,native,face
5,cold,500,stage1_baseline,572,0.000000,0.000000,0.000000,0.003497,0.001224,0.001779,0.006993,0.001573,0.002790,0.050699,0.003185,0.011675,0.097902,0.003451,0.017946,0.132867,0.003536,0.021970,0.132867,0.003536,0.021970,0.132867,0.003536,0.021970,0.308425,0.008499,0.696888,0.245562,0.430612,0.322179,NaN,NaN,0.092463,0.589474,1000,all_pool_depths,full_pipeline_personalization,notebook09_personalized_winner_top1000_face,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,lightgbm,native,face
6,cold,700,lightgbm_rerank,572,0.008741,0.008741,0.008741,0.020979,0.013607,0.015457,0.027972,0.014409,0.017587,0.113636,0.017373,0.034602,0.159091,0.017673,0.040836,0.162587,0.017683,0.041248,0.167832,0.017692,0.041819,0.167832,0.017692,0.041819,0.345995,0.013472,0.762784,0.289267,0.471053,0.337650,NaN,NaN,0.135454,0.584737,1000,all_pool_dep

Uplift vs baseline by pool depth:


,pool_depth,n_queries,delta_HitRate@1,delta_MRR@1,delta_NDCG@1,delta_HitRate@5,delta_MRR@5,delta_NDCG@5,delta_HitRate@10,delta_MRR@10,delta_NDCG@10,delta_HitRate@100,delta_MRR@100,delta_NDCG@100,delta_HitRate@300,delta_MRR@300,delta_NDCG@300,delta_HitRate@500,delta_MRR@500,delta_NDCG@500,delta_HitRate@700,delta_MRR@700,delta_NDCG@700,delta_HitRate@1000,delta_MRR@1000,delta_NDCG@1000,delta_weighted_facet_overlap_at_5,delta_brand_match_at_5,delta_concern_match_at_5,delta_ingredient_match_at_5,delta_skin_type_match_at_5,delta_form_match_at_5,delta_benefit_match_at_5,delta_claim_match_at_5,delta_scent_match_at_5,delta_category_or_product_type_match_at_5
0,100,2288,0.020979,0.020979,0.020979,0.035839,0.026581,0.028876,0.038899,0.027104,0.029984,0.000000,0.026403,0.023304,0.000000,0.026403,0.023304,0.000000,0.026403,0.023304,0.000000,0.026403,0.023304,0.0,0.026403,0.023304,0.033541,0.019712,0.045127,0.027346,0.022174,-0.005966,0.0,0.0,0.027127,-0.005303
1,300,2288,0.018794,0.018794,0.018794,0.042832,0.027557,0.031364,0.052448,0.028957,0.034592,0.041084,0.029224,0.033389,0.000000,0.028986,0.027904,0.000000,0.028986,0.027904,0.000000,0.028986,0.027904,0.0,0.028986,0.027904,0.036240,0.025036,0.050823,0.022866,0.030456,-0.005937,0.0,0.0,0.029203,-0.000838
2,500,2288,0.024476,0.024476,0.024476,0.047640,0.032933,0.036588,0.059003,0.034486,0.040300,0.064685,0.035264,0.042353,0.036713,0.035113,0.038665,0.000000,0.035020,0.034419,0.000000,0.035020,0.034419,0.0,0.035020,0.034419,0.045349,0.026027,0.062719,0.020498,0.044464,-0.004684,0.0,0.0,0.024636,-0.003351
3,700,2288,0.022290,0.022290,0.022290,0.052010,0.033865,0.038415,0.064685,0.035603,0.042559,0.087850,0.036934,0.048038,0.065122,0.036814,0.045059,0.030157,0.036726,0.041012,0.000000,0.036677,0.037755,0.0,0.036677,0.037755,0.053130,0.028649,0.071525,0.028606,0.043371,-0.001173,0.0,0.0,0.027528,0.014175
4,1000,2288,0.020542,0.020542,0.020542,0.055070,0.033661,0.038993,0.072115,0.036198,0.044768,0.107517,0.037836,0.052550,0.097465,0.037794,0.051283,0.065559,0.037715,0.047605,0.036276,0.037667,0.044442,0.0,0.037623,0.040700,0.060581,0.028322,0.077666,0.028482,0.051967,0.005143,0.0,0.0,0.029662,0.013716


## 10. QC / Sanity Checks

In [49]:
# ==== Minimal QC and Diagnostics Export ====
query_diag_cols = [
    "query_id",
    "case_id",
    "user_id",
    "regime",
    "sampling_bracket",
    "gt_item_id",
    "query_text",
    "query_token_len",
    "removed_token_count",
    "prior_review_n",
    "prior_item_n",
    "user_total_reviews",
]

query_diagnostics_df = (
    query_meta_df[query_diag_cols]
    .drop_duplicates("query_id")
    .merge(
        candidate_counts_df.rename(columns={"candidate_n": "candidate_n_max"}),
        on="query_id",
        how="left",
    )
    .merge(
        user_profiles_df[
            [
                "user_id",
                "profile_history_review_n",
                "profile_history_item_n",
                "profile_regime",
                "user_entropy_norm_mean",
                "user_top_share_mean",
            ]
        ],
        on="user_id",
        how="left",
    )
)
query_diagnostics_df = apply_regime_order(query_diagnostics_df, "regime")

loaded_paths = {
    "candidate": CANDIDATE_POOL_PATH,
    "query_cache": QUERY_CACHE_PARQUET,
    "target_case_metadata": TARGET_CASE_METADATA_PARQUET,
    "item_schema": ITEM_SCHEMA_PARQUET,
    "items_facets": ITEMS_FACETS_PARQUET,
    "item_docs": ITEM_DOCS_PARQUET,
    "prior_history": PRIOR_HISTORY_PARQUET,
}

regime_counts_df = pd.Series(
    ordered_regime_counts(query_diagnostics_df["regime"]),
    name="n_queries",
).to_frame()

method_check = sorted(per_query_results_df["rerank_method"].astype(str).unique().tolist())
if method_check != sorted(EXPECTED_RERANK_METHODS):
    raise RuntimeError(f"Unexpected rerank methods found: {method_check}")

print("Loaded paths:")
for label, path_obj in loaded_paths.items():
    print(f"- {label}: {path_obj}")

print("Candidate row counts:")
print("- loaded:", candidate_row_count_loaded)
print("- prepared:", candidate_row_count_prepared)
print("executed rerank methods:", method_check)

print("Regime counts:")
display(regime_counts_df)

print("LightGBM CV summary:")
display(model_cv_summary_df)

print("LightGBM feature importance summary:")
display(feature_importance_summary_df.head(25))

print("Ground-truth presence by pool depth:")
display(gt_presence_df)

print("Non-null rate of key score columns:")
display(score_nonnull_df)

Loaded paths:
- candidate: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage1_candidate_pools/by_method/personalized_winner_top1000_face.parquet
- query_cache: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/query_cache/face_queries.parquet
- target_case_metadata: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/user_sampling/face_target_case_metadata.parquet
- item_schema: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/items/face_item_schema_full.parquet
- items_facets: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/items/face_items_facets.parquet
- item_docs: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/items/face_item_docs.parquet
- prior_history: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/user_sampling/face_user_prior_review_history.parquet
Candidate row counts:
- loade

,n_queries
cold,572
weak,572
moderate,572
strong,572


LightGBM CV summary:


,pool_depth,fold,n_train_queries_all,n_cold_queries_excluded_from_training,n_train_queries_positive,n_fit_queries_positive,n_valid_queries_positive,n_test_queries,n_test_queries_positive,pred_nonnull_rate,training_row_hash,validation_row_hash,fit_runtime_sec,score_runtime_sec,best_iteration,reused_valid_checkpoint
0,100,1,128,472,128,109,19,458,35,1.0,bbbc43faca9b6fb213c96c53493560bab2658ed080fbb8...,6c07f3a76e9f31b5edc6d46e986467644d3157c491c3ea...,0.292939,0.024372,37,False
1,100,2,119,461,119,101,18,458,41,1.0,b6107b7a5aafcd000aed7379cee50f0d6ff5f5bc2932dc...,8d121472c94beea6c86aca84d998a6c782678cfe236a8b...,0.712693,0.024046,14,False
2,100,3,122,460,122,104,18,458,36,1.0,c2ee5a209969ca4fa4c10081f55cb2c798c772c1de3aa8...,7d529d487c7c7d72403d619e6688314abb97b31dc3d9b7...,0.434539,0.053337,99,False
3,100,4,122,443,122,104,18,457,40,1.0,238d465607b95f0d11fbed82abd8e4dd5a372804b2a9d8...,c25939b8d4e57d292c029543ebc5c52c7563d190729fd9...,0.939596,0.046461,84,False
4,100,5,133,452,133,113,20,457,33,1.0,1d605d842469c7a50327fa30b44b2420c8604afd177cb3...,2e5d8da091c9c60e5a1f1d592ce0c67ced8ab85e97a9eb...,0.405334,0.051427,99,False
5,300,1,199,472,199,169,30,458,60,1.0,8d2ff80b3b6d6d4b6ad4a6d907f4e3eedc4886182006dd...,bacaf6943183675a0103b766c6de09809f7b1c4713ea48...,1.113809,0.151060,92,False
6,300,2,194,461,194,165,29,458,63,1.0,d3ff710e7ac228a16cb740b4704196a9613299a908071d...,f8d14a87e1cc6ddbe85e9d0b1fbc2e3a177433b301b4a2...,0.598432,0.053499,27,False
7,300,3,189,460,189,161,28,458,64,1.0,dc5edd0446471b95b656ff992efdc5a2881cdc63ead0b8...,852548093683cd1c46d4f56054c5d80d15f27b921b5aaa...,0.516857,0.036857,11,False
8,300,4,200,443,200,170,30,457,60,1.0,7484909b8f86617688f60d6aa6d97f67df9635333dcb4e...,1407247b4ed48e8175fdc070967e4e41f07d0ae3f12371...,0.878014,0.095906,58,False
9,300,5,206,452,206,175,31,457,56,1.0,3f625fd6173b0530c44bb7590b1f4bdb62402cddb5d39f...,193bc00399cc591dfb14c98a08bc69aa84102bd5d7908b...,2.586466,0.092183,56,False


LightGBM feature importance summary:


,feature,gain_mean,gain_sum,split_sum,depth_n,fold_n
0,item_last_review_gap_days,9281.585809,232039.645226,4628,5,5
1,item_recent_review_share_180d,2874.248042,71856.201057,2755,5,5
2,item_prequery_review_count,1875.635704,46890.892590,3692,5,5
3,item_title_token_overlap,1848.628175,46215.704377,3662,5,5
4,item_first_review_age_days,1807.480584,45187.014590,3507,5,5
5,candidate_rank,1501.905776,37547.644401,3007,5,5
6,candidate_score_norm_pool,1446.579315,36164.482870,3228,5,5
7,item_doc_token_overlap,1256.253058,31406.326453,3558,5,5
8,candidate_brand_prior_share,1181.578233,29539.455831,1637,5,5
9,user_last_interaction_gap_days,821.967738,20549.193451,2100,5,5


Ground-truth presence by pool depth:


,pool_depth,gt_presence_rate
0,100,0.080857
1,300,0.132430
2,500,0.172640
3,700,0.207168
4,1000,0.246941


Non-null rate of key score columns:


,column,nonnull_rate
0,candidate_score,1.0


## 11. Export Outputs

In [50]:
# ==== Save Outputs ====
results_overall_path = OUT_DIR / "results_overall.csv"
results_by_pool_depth_path = OUT_DIR / "results_by_pool_depth.csv"
results_by_regime_path = OUT_DIR / "results_by_regime.csv"
results_by_regime_pool_depth_path = OUT_DIR / "results_by_regime_pool_depth.csv"
runtime_by_pool_depth_path = OUT_DIR / "runtime_by_pool_depth.csv"

uplift_summary_path = OUT_DIR / "uplift_summary.csv"
uplift_by_pool_depth_path = OUT_DIR / "uplift_by_pool_depth.csv"
uplift_by_regime_path = OUT_DIR / "uplift_by_regime.csv"
uplift_by_regime_pool_depth_path = OUT_DIR / "uplift_by_regime_pool_depth.csv"

per_query_results_path = OUT_DIR / "per_query_metrics.parquet"
reranked_candidates_path = OUT_DIR / "reranked_candidates.parquet"
reranked_candidates_export_qc_path = OUT_DIR / "reranked_candidates_export_qc.csv"
query_profiles_path = OUT_DIR / "user_profiles.parquet"

model_cv_summary_path = OUT_DIR / "lightgbm_model_cv_summary.csv"
feature_importance_path = OUT_DIR / "feature_importance.csv"
feature_importance_summary_path = OUT_DIR / "feature_importance_summary.csv"

query_diagnostics_path = OUT_DIR / "query_diagnostics.csv"
run_manifest_path = OUT_DIR / "run_manifest.json"
config_snapshot_path = OUT_DIR / "config_snapshot.json"
diagnostics_summary_path = OUT_DIR / "diagnostics_summary.csv"
report_summary_scope_qc_path = OUT_DIR / "report_summary_scope_qc.csv"
cold_rerank_fallback_qc_path = OUT_DIR / "cold_rerank_fallback_qc.csv"

summary_overall_df.to_csv(results_overall_path, index=False)
brand_contract_diagnostics_df.to_csv(OUT_DIR / "brand_contract_diagnostics.csv", index=False)
summary_by_pool_depth_df.to_csv(results_by_pool_depth_path, index=False)
runtime_by_pool_depth_df.to_csv(runtime_by_pool_depth_path, index=False)
summary_by_regime_df.to_csv(results_by_regime_path, index=False)
summary_by_regime_pool_depth_df.to_csv(results_by_regime_pool_depth_path, index=False)
report_summary_scope_qc_df.to_csv(report_summary_scope_qc_path, index=False)
cold_rerank_fallback_qc_df.to_csv(cold_rerank_fallback_qc_path, index=False)

uplift_overall_df.to_csv(uplift_summary_path, index=False)
uplift_by_pool_depth_df.to_csv(uplift_by_pool_depth_path, index=False)
uplift_by_regime_df.to_csv(uplift_by_regime_path, index=False)
uplift_by_regime_pool_depth_df.to_csv(uplift_by_regime_pool_depth_path, index=False)

per_query_results_df.to_parquet(per_query_results_path, index=False)
reranked_candidates_export_qc_df = build_reranked_candidates_export_qc(reranked_candidates_df, reranked_candidates_path)
reranked_candidates_df.to_parquet(reranked_candidates_path, index=False)
reranked_candidates_export_qc_df.to_csv(reranked_candidates_export_qc_path, index=False)
user_profiles_df.to_parquet(query_profiles_path, index=False)

model_cv_summary_df.to_csv(model_cv_summary_path, index=False)
feature_importance_df.to_csv(feature_importance_path, index=False)
feature_importance_summary_df.to_csv(feature_importance_summary_path, index=False)

query_diagnostics_df.to_csv(query_diagnostics_path, index=False)

config_snapshot = {
    "notebook_name": NOTEBOOK_NAME,
    "stage": STAGE,
    "project_root": str(PROJECT_ROOT),
    "output_dir": str(OUT_DIR),
    "candidate_pool_dir": str(CANDIDATE_POOL_DIR),
    "candidate_pool_path": str(CANDIDATE_POOL_PATH),
    "candidate_pool_manifest_path": str(CANDIDATE_POOL_MANIFEST_PATH),
    "candidate_pool_summary_path": str(CANDIDATE_POOL_SUMMARY_PATH),
    "category_id": CATEGORY_ID,
    "category_folder": CATEGORY_FOLDER,
    "category_label": CATEGORY_LABEL,
    "experiment_condition": EXPERIMENT_CONDITION,
    "candidate_source": CANDIDATE_SOURCE_LABEL,
    "candidate_pool_type": CANDIDATE_POOL_TYPE,
    "retrieval_method": RETRIEVAL_METHOD,
    "retrieval_method_label": RETRIEVAL_METHOD_LABEL,
    "reranking_method": RERANKING_METHOD,
    "rerank_method": OUTPUT_RERANK_METHOD,
    "query_method": STAGE1_QUERY_METHOD,
    "pool_k": int(POOL_K),
    "pool_depths": [int(x) for x in POOL_DEPTHS],
    "actual_regimes": list(actual_regimes),
    "expected_regimes": list(EXPECTED_REGIMES),
    "notebook09_personalized_winner_candidate_pool_source": True,
    "regime_personalization_weight_policy": REGIME_PERSONALIZATION_WEIGHT_POLICY,
    "regime_personalization_weight": make_jsonable_dict(REGIME_PERSONALIZATION_WEIGHT),
    "use_explicit_regime_personalization_weights": bool(USE_EXPLICIT_REGIME_PERSONALIZATION_WEIGHTS),
    "eval_ks": [int(x) for x in EVAL_KS],
    "primary_metrics": PRIMARY_STAGE2_METRICS,
    "secondary_metrics": SECONDARY_STAGE2_METRICS,
    "common_comparison_metrics": COMMON_STAGE2_COMPARISON_METRICS,
    "supplementary_metrics": SUPPLEMENTARY_STAGE2_METRICS,
    "preference_alignment_k": int(PREFERENCE_ALIGNMENT_K),
    "expected_rerank_methods": list(EXPECTED_RERANK_METHODS),
    "cold_reranking_fallback_policy": "use_same_family_p2q_oof_predictions",
    "transport": False,
    "trained_on_pool": "personalized_winner_long",
    "outcome_data_used": False,
    "cold_users_in_prior_aware_training": False,
    "cold_prediction_source": "p2q_cold_fallback",
    "cold_metric_identity_passed": bool(cold_rerank_fallback_qc_df["cold_metric_identity_passed"].all()),
    "history_eligible_regimes": ["weak", "moderate", "strong"],
    "lightgbm_config": {
        "random_seed": int(RANDOM_SEED),
        "n_folds": int(N_FOLDS),
        "valid_frac_within_train": float(VALID_FRAC_WITHIN_TRAIN),
        "objective": LIGHTGBM_OBJECTIVE,
        "metric": LIGHTGBM_METRIC,
        "eval_at": [int(x) for x in LIGHTGBM_EVAL_AT],
        "num_boost_round": int(LIGHTGBM_NUM_BOOST_ROUND),
        "early_stopping_rounds": int(LIGHTGBM_EARLY_STOPPING_ROUNDS),
        "use_candidate_score_feature": bool(USE_CANDIDATE_SCORE_FEATURE),
        "dedup_history_by_user_item": bool(DEDUP_HISTORY_BY_USER_ITEM),
        "feature_col_count": int(len(static_feature_cols)),
        "optional_feature_col_map": make_jsonable_dict(optional_feature_col_map),
        "regime_personalization_weight_policy": REGIME_PERSONALIZATION_WEIGHT_POLICY,
    },
}
Path(config_snapshot_path).write_text(
    json.dumps(make_jsonable_dict(config_snapshot), ensure_ascii=False, indent=2),
    encoding="utf-8",
)

saved_paths = [
    results_overall_path,
    results_by_pool_depth_path,
    runtime_by_pool_depth_path,
    results_by_regime_path,
    results_by_regime_pool_depth_path,
    report_summary_scope_qc_path,
    cold_rerank_fallback_qc_path,
    uplift_summary_path,
    uplift_by_pool_depth_path,
    uplift_by_regime_path,
    uplift_by_regime_pool_depth_path,
    per_query_results_path,
    reranked_candidates_path,
    reranked_candidates_export_qc_path,
    query_profiles_path,
    model_cv_summary_path,
    feature_importance_path,
    feature_importance_summary_path,
    config_snapshot_path,
]

saved_paths.append(query_diagnostics_path)

In [51]:
# ==== Feature Contract, Temporal Summary, and Leakage QC Export ====
feature_manifest_path = OUT_DIR / "feature_manifest.json"
used_model_features_path = OUT_DIR / "used_model_features.csv"
feature_leakage_qc_path = OUT_DIR / "feature_leakage_qc.csv"
temporal_feature_summary_path = OUT_DIR / "temporal_feature_summary.csv"
used_model_features_df = used_model_features_df.copy()
used_model_features_df.columns = used_model_features_df.columns.astype(str)

used_model_features_df["is_temporal_feature"] = (
    used_model_features_df["feature"]
    .astype(str)
    .isin(set(COMMON_TEMPORAL_FEATURE_COLS))
)

used_model_features_df.to_csv(used_model_features_path, index=False)
feature_leakage_qc_df.to_csv(feature_leakage_qc_path, index=False)
temporal_feature_summary_df.to_csv(temporal_feature_summary_path, index=False)
Path(feature_manifest_path).write_text(
    json.dumps(make_jsonable_dict(feature_manifest) if "make_jsonable_dict" in globals() else feature_manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

if "config_snapshot" in globals():
    config_snapshot["common_feature_registry_version"] = COMMON_FEATURE_REGISTRY_VERSION
    config_snapshot["temporal_feature_version"] = TEMPORAL_FEATURE_VERSION
    config_snapshot["temporal_leakage_rule"] = TEMPORAL_LEAKAGE_RULE
    config_snapshot["feature_manifest_path"] = str(feature_manifest_path)
    config_snapshot["used_model_features_path"] = str(used_model_features_path)
    config_snapshot["feature_leakage_qc_path"] = str(feature_leakage_qc_path)
    config_snapshot["temporal_feature_summary_path"] = str(temporal_feature_summary_path)
    config_snapshot["used_model_feature_count"] = int(len(used_model_features_df))
    config_snapshot["used_temporal_feature_count"] = int(
        used_model_features_df["is_temporal_feature"].sum()
    )
    if "config_snapshot_path" in globals():
        Path(config_snapshot_path).write_text(
            json.dumps(make_jsonable_dict(config_snapshot) if "make_jsonable_dict" in globals() else config_snapshot, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

if "run_manifest" in globals():
    run_manifest["feature_parity"] = {
        "common_feature_registry_version": COMMON_FEATURE_REGISTRY_VERSION,
        "temporal_feature_version": TEMPORAL_FEATURE_VERSION,
        "temporal_leakage_rule": TEMPORAL_LEAKAGE_RULE,
        "used_model_feature_count": int(len(used_model_features_df)),
        "used_temporal_feature_count": int(used_model_features_df["is_temporal_feature"].sum()),
        "raw_timestamp_columns_excluded": bool(feature_leakage_qc_df.loc[feature_leakage_qc_df["check_name"].eq("raw_timestamp_columns_excluded"), "passed"].all()),
        "temporal_recency_features_included": bool(used_model_features_df["is_temporal_feature"].any()),
    }
    run_manifest.setdefault("output_paths", {})
    run_manifest["output_paths"].update({
        "feature_manifest": str(feature_manifest_path),
        "used_model_features": str(used_model_features_path),
        "feature_leakage_qc": str(feature_leakage_qc_path),
        "temporal_feature_summary": str(temporal_feature_summary_path),
    })
    run_manifest["created_outputs"] = list(dict.fromkeys(run_manifest.get("created_outputs", []) + [
        str(feature_manifest_path),
        str(used_model_features_path),
        str(feature_leakage_qc_path),
        str(temporal_feature_summary_path),
    ]))
    if "run_manifest_path" in globals():
        Path(run_manifest_path).write_text(
            json.dumps(make_jsonable_dict(run_manifest) if "make_jsonable_dict" in globals() else run_manifest, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

if "saved_paths" in globals():
    for path_obj in [feature_manifest_path, used_model_features_path, feature_leakage_qc_path, temporal_feature_summary_path]:
        if path_obj not in saved_paths:
            saved_paths.append(path_obj)

print("Feature contract outputs saved:")
for path_obj in [feature_manifest_path, used_model_features_path, feature_leakage_qc_path, temporal_feature_summary_path]:
    print("-", path_obj)

Feature contract outputs saved:
- /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/feature_manifest.json
- /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/used_model_features.csv
- /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/feature_leakage_qc.csv
- /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/temporal_feature_summary.csv


In [52]:
# ==== Export Outputs — Part 3 ====
with runtime_step(
    "export_outputs",
    pool_depth=RUNTIME_REPORT_POOL_DEPTH,
    runtime_measurement_type="wall_clock_step",
    derived_from_existing_runtime=False,
):
    run_manifest = {
        "notebook_name": NOTEBOOK_NAME,
        "stage": STAGE,
        "project_root": str(PROJECT_ROOT),
        "output_dir": str(OUT_DIR),
        "host": socket.gethostname(),
        "platform": platform.platform(),
        "category_id": CATEGORY_ID,
        "category_folder": CATEGORY_FOLDER,
        "category_label": CATEGORY_LABEL,
        "experiment_condition": EXPERIMENT_CONDITION,
        "candidate_source": CANDIDATE_SOURCE_LABEL,
        "candidate_pool_type": CANDIDATE_POOL_TYPE,
        "retrieval_method": RETRIEVAL_METHOD,
        "retrieval_method_label": RETRIEVAL_METHOD_LABEL,
        "reranking_method": RERANKING_METHOD,
        "rerank_method": OUTPUT_RERANK_METHOD,
        "candidate_pool_dir": str(CANDIDATE_POOL_DIR),
        "candidate_pool_path": str(CANDIDATE_POOL_PATH),
        "candidate_pool_manifest_path": str(CANDIDATE_POOL_MANIFEST_PATH),
        "candidate_pool_summary_path": str(CANDIDATE_POOL_SUMMARY_PATH),
        "p2q_oof_prediction_path": str(P2Q_OOF_PREDICTION_PATH),
        "cold_reranking_fallback_policy": "use_same_family_p2q_oof_predictions",
    "transport": False,
    "trained_on_pool": "personalized_winner_long",
    "outcome_data_used": False,
        "cold_users_in_prior_aware_training": False,
        "cold_prediction_source": "p2q_cold_fallback",
        "cold_metric_identity_passed": bool(cold_rerank_fallback_qc_df["cold_metric_identity_passed"].all()),
        "history_eligible_regimes": ["weak", "moderate", "strong"],
        "query_method": STAGE1_QUERY_METHOD,
        "notebook09_personalized_winner_candidate_pool_source": True,
        "regime_personalization_weight_policy": REGIME_PERSONALIZATION_WEIGHT_POLICY,
        "regime_personalization_weight": make_jsonable_dict(REGIME_PERSONALIZATION_WEIGHT),
        "use_explicit_regime_personalization_weights": bool(USE_EXPLICIT_REGIME_PERSONALIZATION_WEIGHTS),
        "primary_metrics": PRIMARY_STAGE2_METRICS,
        "secondary_metrics": SECONDARY_STAGE2_METRICS,
        "common_comparison_metrics": COMMON_STAGE2_COMPARISON_METRICS,
        "key_input_paths": {
            "candidate_pool_path": str(CANDIDATE_POOL_PATH),
            "candidate_pool_manifest_path": str(CANDIDATE_POOL_MANIFEST_PATH),
            "candidate_pool_summary_path": str(CANDIDATE_POOL_SUMMARY_PATH),
            "query_cache_parquet": str(QUERY_CACHE_PARQUET),
            "target_case_metadata_parquet": str(TARGET_CASE_METADATA_PARQUET),
            "query_cache_summary_path": str(QUERY_CACHE_SUMMARY_PATH),
            "query_cache_config_path": str(QUERY_CACHE_CONFIG_PATH),
            "item_schema_parquet": str(ITEM_SCHEMA_PARQUET),
            "items_facets_parquet": str(ITEMS_FACETS_PARQUET),
            "item_docs_parquet": str(ITEM_DOCS_PARQUET),
            "prior_history_parquet": str(PRIOR_HISTORY_PARQUET),
        },
        "supplementary_metrics": SUPPLEMENTARY_STAGE2_METRICS,
        "preference_alignment_k": int(PREFERENCE_ALIGNMENT_K),
        "candidate_row_count_loaded": int(candidate_row_count_loaded),
        "candidate_row_count_prepared": int(candidate_row_count_prepared),
        "unique_case_count": int(unique_case_count),
        "unique_query_count": int(unique_query_count),
        "query_cache_row_count": int(len(query_cache_meta_df)),
        "query_meta_row_count": int(len(query_meta_df)),
        "reranked_candidate_row_count": int(len(reranked_candidates_df)),
        "per_query_result_row_count": int(len(per_query_results_df)),
        "runtime_by_pool_depth": make_jsonable_dict(runtime_by_pool_depth_df.to_dict(orient="list")),
        "executed_rerank_methods": list(executed_methods),
        "created_outputs": [str(p) for p in saved_paths + [run_manifest_path, diagnostics_summary_path]],
        "candidate_schema_map": make_jsonable_dict(candidate_schema_map),
        "query_cache_schema_map": make_jsonable_dict(query_cache_schema_map),
        "regime_column_used": regime_column_used,
        "regime_counts": make_jsonable_dict(ordered_regime_counts(query_meta_df["regime"])),
        "status": "completed",
    }

    Path(run_manifest_path).write_text(
        json.dumps(make_jsonable_dict(run_manifest), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    if run_manifest_path not in saved_paths:
        saved_paths.append(run_manifest_path)

    diagnostics_rows = []

    for label, path_obj in loaded_paths.items():
        diagnostics_rows.append(
            {
                "notebook_name": NOTEBOOK_NAME,
                "category": CATEGORY_ID,
                "category_folder": CATEGORY_FOLDER,
                "category_label": CATEGORY_LABEL,
                "stage": STAGE,
                "artifact_name": label,
                "artifact_path": str(path_obj),
                "file_exists": Path(path_obj).exists(),
                "row_count": None,
                "note": "input",
            }
        )

    for path_obj in saved_paths:
        diagnostics_rows.append(
            {
                "notebook_name": NOTEBOOK_NAME,
                "category": CATEGORY_ID,
                "category_folder": CATEGORY_FOLDER,
                "category_label": CATEGORY_LABEL,
                "stage": STAGE,
                "artifact_name": Path(path_obj).name,
                "artifact_path": str(path_obj),
                "file_exists": Path(path_obj).exists(),
                "row_count": None,
                "note": "output",
            }
        )

    diagnostics_summary_df = pd.DataFrame(diagnostics_rows)
    diagnostics_summary_df.to_csv(diagnostics_summary_path, index=False)

    if diagnostics_summary_path not in saved_paths:
        saved_paths.append(diagnostics_summary_path)


runtime_steps_df, runtime_notebook_summary_df, runtime_method_summary_at1000_df, runtime_pool_depth_diagnostic_df, runtime_method_components_at1000_df = export_runtime_logs(
    OUT_DIR,
    report_pool_depth=RUNTIME_REPORT_POOL_DEPTH,
)

runtime_output_paths = [
    OUT_DIR / "runtime_steps.csv",
    OUT_DIR / "runtime_notebook_summary.csv",
    OUT_DIR / "runtime_method_summary_at1000.csv",
    OUT_DIR / "runtime_pool_depth_diagnostic.csv",
    OUT_DIR / "runtime_method_components_at1000_face.csv",
]

for runtime_path in runtime_output_paths:
    if runtime_path not in saved_paths:
        saved_paths.append(runtime_path)

print("Runtime summary at @1000:")
display(runtime_method_summary_at1000_df)

print("Total notebook runtime:")
display(runtime_notebook_summary_df[["method", "total_notebook_runtime_sec", "sample_scope", "common_sample_filtering_introduced"]])

print("Runtime by step:")
display(runtime_steps_df[["method", "step_name", "pool_depth", "runtime_sec", "n_queries", "n_candidates", "runtime_measurement_type", "derived_from_existing_runtime"]])

print("Runtime method components at @1000:")
display(runtime_method_components_at1000_df)

print("n_queries and n_candidates at @1000:")
display(runtime_method_summary_at1000_df[["method", "n_queries", "n_candidates", "runtime_sec_per_query", "runtime_sec_per_candidate"]])

print("runtime sample scope: native")
print("common-sample filtering introduced: False")

print("Saved outputs:")
for path_obj in saved_paths:
    print("-", path_obj)

print("Completion summary:")
print("- output_dir:", OUT_DIR)
print("- candidate rows prepared:", int(candidate_row_count_prepared))
print("- per-query metric rows:", int(len(per_query_results_df)))
print("- primary metrics:", PRIMARY_STAGE2_METRICS)
print("- secondary metrics:", SECONDARY_STAGE2_METRICS)
print("- supplementary metrics:", SUPPLEMENTARY_STAGE2_METRICS)

del per_query_results_df, reranked_candidates_df
gc.collect()

Runtime summary at @1000:


,category,category_id,notebook_name,branch,method,rerank_method,pool_depth,n_queries,n_candidates,online_operation_runtime_sec,offline_preparation_runtime_sec,evaluation_export_runtime_sec,total_notebook_runtime_sec,runtime_sec_per_query,runtime_sec_per_candidate,queries_per_second,candidates_per_second,runtime_scope,candidate_pool_type,retrieval_method,retrieval_method_label,reranking_method,sample_scope,common_sample_filtering_introduced,primary_metric,ndcg_at_5,hitrate_at_5,mrr_at_5,baseline_ndcg_at_5,baseline_hitrate_at_5,delta_ndcg_at_5_vs_baseline,delta_hitrate_at_5_vs_baseline,delta_ndcg5_per_100sec_online,delta_hitrate5_per_100sec_online
0,facial_skincare,face,11c_personalized_rerank_lightgbm_face.ipynb,personalized_winner_lightgbm_prior,lightgbm,lightgbm_rerank,1000,2288.0,2288000.0,936.452469,32.059093,403.298743,8836.106789,0.409289,0.000409,2.443263,2443.263353,stage2_online_reranking_excluding_stage1_retri...,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,lightgbm,native,False,NDCG@5,0.049074,0.069493,0.042322,0.010081,0.014423,0.038993,0.05507,0.004164,0.005881


Total notebook runtime:


,method,total_notebook_runtime_sec,sample_scope,common_sample_filtering_introduced
0,lightgbm,8836.106789,native,False


Runtime by step:


...

,category_id,category_folder,method,rerank_method,pool_depth,n_queries,feature_preparation_runtime_sec,model_fit_or_tuning_runtime_sec,model_scoring_runtime_sec,ranking_sorting_runtime_sec,evaluation_runtime_sec,export_outputs_runtime_sec,online_operation_runtime_sec,runtime_scope,candidate_pool_type,retrieval_method,retrieval_method_label,reranking_method
0,face,facial_skincare,lightgbm,lightgbm_rerank,1000,2288.0,928.546391,32.059093,2.669569,5.236509,403.236823,0.06192,936.452469,stage2_online_reranking_excluding_stage1_retri...,Profile Sparse QCHA,Profile Sparse QCHA,Profile Sparse QCHA,lightgbm


n_queries and n_candidates at @1000:


,method,n_queries,n_candidates,runtime_sec_per_query,runtime_sec_per_candidate
0,lightgbm,2288.0,2288000.0,0.409289,0.000409


runtime sample scope: native
common-sample filtering introduced: False
Saved outputs:
- /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/results_overall.csv
- /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/results_by_pool_depth.csv
- /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/runtime_by_pool_depth.csv
- /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/results_by_regime.csv
- /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/results_by_regime_pool_depth.csv
- /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/report_summary_scope_qc.csv
- /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/out

31

In [53]:
# ==== Candidate Source Naming QC ====
retrieval_method_values = sorted(
    set(
        str(value)
        for frame_name in ["results_overall_df", "per_query_results_df", "reranked_candidates_df"]
        if frame_name in globals()
        for value in globals()[frame_name].get("retrieval_method", pd.Series(dtype=str)).dropna().astype(str).unique()
    )
)
expected_retrieval_method_values = [RETRIEVAL_METHOD]
if retrieval_method_values and retrieval_method_values != expected_retrieval_method_values:
    raise RuntimeError(
        f"Expected only {RETRIEVAL_METHOD} in retrieval_method outputs; found {retrieval_method_values}"
    )
candidate_pool_manifest_qc = load_json_if_exists(CANDIDATE_POOL_MANIFEST_PATH)
manifest_candidate_path_text = str(
    candidate_pool_manifest_qc.get("output_paths", {}).get(CANDIDATE_POOL_MANIFEST_OUTPUT_KEY, "")
).strip()
if not manifest_candidate_path_text:
    raise RuntimeError("Notebook 09 manifest is missing the selected candidate output path.")
if Path(manifest_candidate_path_text) != CANDIDATE_POOL_PATH:
    raise RuntimeError("Active candidate path differs from the Notebook 09 selected output path.")

print("Candidate source naming QC: PASS")
print("Retrieval method:", RETRIEVAL_METHOD)
print("Candidate path:", CANDIDATE_POOL_PATH)

Candidate source naming QC: PASS
Retrieval method: Profile Sparse QCHA
Candidate path: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage1_candidate_pools/by_method/personalized_winner_top1000_face.parquet


In [54]:
# ==== Runtime Depth QC ====
expected_runtime_depths = sorted(int(depth) for depth in POOL_DEPTHS)
observed_runtime_depths = (
    sorted(pd.to_numeric(runtime_by_pool_depth_df["pool_depth"], errors="coerce").dropna().astype(int).unique().tolist())
    if "runtime_by_pool_depth_df" in globals() and isinstance(runtime_by_pool_depth_df, pd.DataFrame) and "pool_depth" in runtime_by_pool_depth_df.columns
    else []
)
runtime_depths_complete = observed_runtime_depths == expected_runtime_depths

runtime_required_cols = [
    "pool_depth",
    "candidate_pool_depth",
    "n_queries",
    "n_candidates",
    "feature_preparation_runtime_sec",
    "model_scoring_runtime_sec",
    "ranking_sorting_runtime_sec",
    "online_operation_runtime_sec",
    "runtime_sec_per_query",
    "runtime_sec_per_candidate",
    "queries_per_second",
    "candidates_per_second",
]
runtime_missing_cols = [
    col for col in runtime_required_cols
    if "runtime_by_pool_depth_df" not in globals()
    or not isinstance(runtime_by_pool_depth_df, pd.DataFrame)
    or col not in runtime_by_pool_depth_df.columns
]

runtime_duplicate_rows = 0
if "runtime_by_pool_depth_df" in globals() and isinstance(runtime_by_pool_depth_df, pd.DataFrame) and not runtime_by_pool_depth_df.empty:
    duplicate_key_cols = [col for col in ["pool_depth", "method", "reranker", "reranking_method"] if col in runtime_by_pool_depth_df.columns]
    if duplicate_key_cols:
        runtime_duplicate_rows = int(runtime_by_pool_depth_df.duplicated(duplicate_key_cols).sum())

runtime_output_candidates = []
for name in [
    "runtime_by_pool_depth_path",
    "runtime_method_summary_at1000_path",
    "runtime_steps_path",
    "runtime_notebook_summary_path",
]:
    if name in globals():
        runtime_output_candidates.append(globals()[name])
if "OUT_DIR" in globals():
    for filename in [
        "runtime_by_pool_depth.csv",
        "runtime_method_summary_at1000.csv",
        "runtime_steps.csv",
        "runtime_notebook_summary.csv",
    ]:
        path = OUT_DIR / filename
        if path not in runtime_output_candidates:
            runtime_output_candidates.append(path)
runtime_output_files_created = [str(path) for path in runtime_output_candidates if Path(path).exists()]

print("Candidate pool depths evaluated:", expected_runtime_depths)
print("Candidate pool depths in runtime_by_pool_depth.csv:", observed_runtime_depths)
print("Every evaluated pool depth has a runtime row:", runtime_depths_complete)
print("Output runtime files created:")
for path in runtime_output_files_created:
    print(path)
runtime_all_null_components = []
if "runtime_by_pool_depth_df" in globals() and isinstance(runtime_by_pool_depth_df, pd.DataFrame) and not runtime_by_pool_depth_df.empty:
    for col in [
        "feature_preparation_runtime_sec",
        "model_scoring_runtime_sec",
        "ranking_sorting_runtime_sec",
        "online_operation_runtime_sec",
    ]:
        if col in runtime_by_pool_depth_df.columns and pd.to_numeric(runtime_by_pool_depth_df[col], errors="coerce").notna().sum() == 0:
            runtime_all_null_components.append(col)
if runtime_missing_cols:
    print("Runtime QC warning: missing runtime columns:", runtime_missing_cols)
else:
    print("Runtime QC missing columns: none")
if runtime_all_null_components:
    print("Runtime QC warning: runtime components not measured per depth:", runtime_all_null_components)
else:
    print("Runtime QC unmeasured components: none")
if runtime_duplicate_rows:
    print("Runtime QC warning: duplicate runtime rows:", runtime_duplicate_rows)
else:
    print("Runtime QC duplicate rows: none")


Candidate pool depths evaluated: [100, 300, 500, 700, 1000]
Candidate pool depths in runtime_by_pool_depth.csv: [100, 300, 500, 700, 1000]
Every evaluated pool depth has a runtime row: True
Output runtime files created:
/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/runtime_by_pool_depth.csv
/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/runtime_method_summary_at1000.csv
/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/runtime_steps.csv
/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/runtime_notebook_summary.csv
Runtime QC missing columns: none
Runtime QC warning: runtime components not measured per depth: ['feature_preparation_runtime_sec']
Runtime QC duplicate rows: none


In [55]:
# ==== Metric Depth QC ====
expected_pool_depths = [int(depth) for depth in POOL_DEPTHS]
expected_eval_ks = [int(k) for k in EVAL_KS]
required_metric_cols = [
    f"{metric}@{k}"
    for metric in ["HitRate", "NDCG", "MRR"]
    for k in expected_eval_ks
]

metric_source_df = None
for candidate_name in [
    "per_query_results_df",
    "per_query_final_eval_df",
    "per_query_metrics_df",
    "summary_by_pool_depth_df",
    "results_by_pool_depth_df",
]:
    candidate_df = globals().get(candidate_name)
    if isinstance(candidate_df, pd.DataFrame) and not candidate_df.empty:
        metric_source_df = candidate_df
        metric_source_name = candidate_name
        break
if metric_source_df is None:
    metric_source_df = pd.DataFrame()
    metric_source_name = "none"

available_hitrate_cols = [col for col in required_metric_cols if col.startswith("HitRate@") and col in metric_source_df.columns]
available_ndcg_cols = [col for col in required_metric_cols if col.startswith("NDCG@") and col in metric_source_df.columns]
available_mrr_cols = [col for col in required_metric_cols if col.startswith("MRR@") and col in metric_source_df.columns]
missing_metric_cols = [col for col in required_metric_cols if col not in metric_source_df.columns]

pool_metric_df = globals().get("summary_by_pool_depth_df")
if not isinstance(pool_metric_df, pd.DataFrame) or pool_metric_df.empty:
    pool_metric_df = globals().get("results_by_pool_depth_df")
if not isinstance(pool_metric_df, pd.DataFrame) or pool_metric_df.empty:
    pool_metric_df = metric_source_df

observed_pool_depths = (
    sorted(pd.to_numeric(pool_metric_df["pool_depth"], errors="coerce").dropna().astype(int).unique().tolist())
    if isinstance(pool_metric_df, pd.DataFrame) and "pool_depth" in pool_metric_df.columns
    else []
)
pool_depth_metric_missing = {}
if isinstance(pool_metric_df, pd.DataFrame) and "pool_depth" in pool_metric_df.columns:
    for depth in expected_pool_depths:
        depth_rows = pool_metric_df[pd.to_numeric(pool_metric_df["pool_depth"], errors="coerce").eq(depth)]
        missing_for_depth = [col for col in required_metric_cols if col not in depth_rows.columns]
        if depth_rows.empty or missing_for_depth:
            pool_depth_metric_missing[int(depth)] = missing_for_depth if missing_for_depth else ["no rows"]
else:
    pool_depth_metric_missing = {int(depth): ["pool_depth column missing"] for depth in expected_pool_depths}
all_pool_depths_have_metrics = not pool_depth_metric_missing and observed_pool_depths == expected_pool_depths

output_file_candidates = []
if "saved_paths" in globals():
    output_file_candidates.extend(saved_paths)
if isinstance(globals().get("expected_outputs"), (list, tuple)):
    output_file_candidates.extend(globals()["expected_outputs"])
if isinstance(globals().get("output_paths"), dict):
    output_file_candidates.extend(globals()["output_paths"].values())
for name, value in list(globals().items()):
    if name.endswith("_path") and any(token in name for token in ["results", "runtime", "manifest", "summary", "metrics", "candidates"]):
        output_file_candidates.append(value)

output_files_created = []
for path_value in output_file_candidates:
    try:
        path_obj = Path(path_value)
    except TypeError:
        continue
    if path_obj.exists() and str(path_obj) not in output_files_created:
        output_files_created.append(str(path_obj))

print("POOL_DEPTHS:", expected_pool_depths)
print("EVAL_KS:", expected_eval_ks)
print("Metric source dataframe:", metric_source_name)
print("Available HitRate columns:", available_hitrate_cols)
print("Available NDCG columns:", available_ndcg_cols)
print("Available MRR columns:", available_mrr_cols)
print("Every evaluated pool depth has all required EVAL_KS metrics:", all_pool_depths_have_metrics)
if missing_metric_cols:
    print("Missing metric columns:", missing_metric_cols)
if pool_depth_metric_missing:
    print("Pool-depth metric gaps:", pool_depth_metric_missing)
print("Output files created:")
for path in output_files_created:
    print(path)


POOL_DEPTHS: [100, 300, 500, 700, 1000]
EVAL_KS: [1, 5, 10, 100, 300, 500, 700, 1000]
Metric source dataframe: per_query_final_eval_df
Available HitRate columns: ['HitRate@1', 'HitRate@5', 'HitRate@10', 'HitRate@100', 'HitRate@300', 'HitRate@500', 'HitRate@700', 'HitRate@1000']
Available NDCG columns: ['NDCG@1', 'NDCG@5', 'NDCG@10', 'NDCG@100', 'NDCG@300', 'NDCG@500', 'NDCG@700', 'NDCG@1000']
Available MRR columns: ['MRR@1', 'MRR@5', 'MRR@10', 'MRR@100', 'MRR@300', 'MRR@500', 'MRR@700', 'MRR@1000']
Every evaluated pool depth has all required EVAL_KS metrics: True
Output files created:
/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/results_overall.csv
/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/results_by_pool_depth.csv
/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/runtime_by_pool_depth.c

In [56]:
# ==== Held-Out Feature Interpretation Export ====
model_feature_columns = list(EXPECTED_MODEL_FEATURE_COLUMNS)
if len(model_feature_columns) != len(set(model_feature_columns)):
    raise RuntimeError("Model feature names must be unique.")
if used_model_feature_cols != model_feature_columns:
    raise RuntimeError("Executed model feature order differs from the interpretation contract.")

feature_interpretation_fold_assignments_df = pd.concat(
    feature_interpretation_fold_assignment_frames,
    ignore_index=True,
)
feature_interpretation_fold_assignments_df["fold_id"] = pd.to_numeric(
    feature_interpretation_fold_assignments_df["fold_id"], errors="raise"
).astype(np.int16)
if feature_interpretation_fold_assignments_df.duplicated(["pool_depth", "case_id"]).any():
    raise RuntimeError("Each case must have exactly one fold assignment per pool depth.")

expected_case_ids = set(candidate_work["case_id"].astype(str))
for pool_depth in POOL_DEPTHS:
    depth_assignments = feature_interpretation_fold_assignments_df.loc[
        feature_interpretation_fold_assignments_df["pool_depth"].eq(int(pool_depth))
    ]
    if set(depth_assignments["case_id"].astype(str)) != expected_case_ids:
        raise RuntimeError(f"Fold assignments do not cover every case at pool_depth={pool_depth}.")
feature_interpretation_fold_assignments_df.to_parquet(
    FEATURE_INTERPRETATION_FOLD_ASSIGNMENT_PATH,
    index=False,
)

brand_affinity_feature_columns = [
    column for column in model_feature_columns if column in set(USER_BRAND_FEATURE_COLUMNS)
]
user_history_feature_columns = [
    column for column in model_feature_columns
    if column in set(P2Q_FORBIDDEN_USER_PRIOR_FEATURE_COLUMNS)
]
candidate_side_brand_feature_columns = [
    column for column in model_feature_columns if column == "candidate_brand_present"
]

if CONDITION_NAME == "P2-Q":
    if user_history_feature_columns:
        raise RuntimeError(f"P2-Q contains user-history model features: {user_history_feature_columns}")
    if brand_affinity_feature_columns:
        raise RuntimeError(f"P2-Q contains brand-affinity model features: {brand_affinity_feature_columns}")
else:
    if HISTORY_SOURCE != "all_prior":
        raise RuntimeError(f"{CONDITION_NAME} must use All Prior history.")
    if feature_columns_P2P != feature_columns_Full or feature_dtypes_P2P != feature_dtypes_Full:
        raise RuntimeError("P2-P and Full feature schemas must be identical.")

retrieval_features = {
    "candidate_rank", "candidate_rank_log1p", "candidate_rank_inv", "candidate_score",
    "candidate_score_norm_pool", "candidate_rank_pct_in_pool", "candidate_rank_from_bottom",
}
query_item_features = {
    "query_token_len", "query_token_len_log1p", "query_unique_token_len",
    "query_unique_token_len_log1p", "removed_token_count", "removed_token_count_log1p",
    "item_title_token_overlap", "item_doc_token_overlap", "query_item_structured_match",
}
candidate_item_features = set(COMMON_TEMPORAL_ITEM_FEATURE_COLS) | {"candidate_brand_present"}
functional_facet_preference_features = (
    set(COMMON_FAMILY_USER_FEATURE_COLS)
    | set(COMMON_TEMPORAL_FAMILY_FEATURE_COLS)
    | {"user_entropy_norm_mean", "user_top_share_mean"}
)
history_depth_recency_features = (
    set(COMMON_USER_BASE_FEATURE_COLS)
    | set(COMMON_TEMPORAL_USER_FEATURE_COLS)
    | {
        "regime_cold", "regime_weak", "regime_moderate", "regime_strong",
        "profile_regime_cold", "profile_regime_weak", "profile_regime_moderate",
        "profile_regime_strong", "regime_personalization_weight",
        "user_item_affinity", "user_item_seen_strength",
    }
)


def interpretation_group_for_feature(feature: str) -> str:
    if feature in set(USER_BRAND_FEATURE_COLUMNS):
        return "brand_affinity"
    if feature in retrieval_features or feature.endswith("__pool_norm"):
        return "retrieval"
    if feature in query_item_features or feature.startswith("qmatch__"):
        return "query_item"
    if feature in candidate_item_features or feature.startswith("item_label_n__"):
        return "candidate_item"
    if feature in functional_facet_preference_features:
        return "functional_facet_preference"
    if feature in history_depth_recency_features:
        return "history_depth_recency"
    raise RuntimeError(f"Feature lacks an interpretation group: {feature}")


feature_group_mapping_df = pd.DataFrame({
    "feature_position": np.arange(len(model_feature_columns), dtype=np.int32),
    "feature": model_feature_columns,
    "dtype": ["float32"] * len(model_feature_columns),
    "interpretation_group": [
        interpretation_group_for_feature(column) for column in model_feature_columns
    ],
})
feature_group_mapping_df.to_csv(FEATURE_GROUP_MAPPING_PATH, index=False)

candidate_source_output_key = (
    "personalized_winner_long" if CONDITION_NAME == "Full" else "query_only_winner_long"
)
required_notebook09_contract_flags = {
    "candidate_budget_policy": "exact_k_all_methods",
    "candidate_budget_k": int(POOL_K),
    "variable_candidate_count_allowed": False,
    "candidate_scores_preserved": True,
    "retrieval_recomputed": False,
    "complete_case_universe_preserved": True,
    "brand_in_candidate_output": True,
    "historical_review_reputation_used_as_profile_evidence": False,
    "brand_query_matching_enabled": False,
}
for key, expected_value in required_notebook09_contract_flags.items():
    if candidate_pool_manifest.get(key) != expected_value:
        raise RuntimeError(
            f"Notebook 09 candidate contract mismatch for {key}: "
            f"expected={expected_value!r}, actual={candidate_pool_manifest.get(key)!r}"
        )
notebook09_candidate_path = candidate_pool_manifest.get("output_paths", {}).get(
    candidate_source_output_key
)
if not notebook09_candidate_path or Path(notebook09_candidate_path) != CANDIDATE_POOL_PATH:
    raise RuntimeError(
        f"Notebook 09 {candidate_source_output_key} path does not match CANDIDATE_POOL_PATH."
    )
if int(candidate_pool_manifest.get("query_count", -1)) != int(candidate_work["case_id"].nunique()):
    raise RuntimeError("Notebook 09 query count does not match the reranking case universe.")

notebook09_required_compatibility_columns = [
    "category_id", "case_id", "query_id", "user_id", "regime", "sampling_bracket",
    "target_selection_mode", "query_method", "active_query_method", "gt_item_id",
    "target_parent_asin", "query_text", "prior_history_n",
    "baseline_retrieval_method_key", "baseline_retrieval_method", "method",
    "method_slug", "retrieval_method", "retrieval_method_label",
    "candidate_parent_asin", "candidate_rank", "candidate_score", "is_target", "is_gt",
]
candidate_source_contract = {
    "source_notebook": "Notebook 09",
    "source_notebook_name": candidate_pool_manifest.get("notebook_name"),
    "candidate_source": CANDIDATE_SOURCE_LABEL,
    "candidate_source_output_key": candidate_source_output_key,
    "candidate_pool_path": str(CANDIDATE_POOL_PATH),
    "candidate_pool_manifest_path": str(CANDIDATE_POOL_MANIFEST_PATH),
    "candidate_pool_summary_path": str(CANDIDATE_POOL_SUMMARY_PATH),
    "candidate_pool_type": CANDIDATE_POOL_TYPE,
    "retrieval_method": RETRIEVAL_METHOD,
    "candidate_source_format": INTERPRETATION_CANDIDATE_SOURCE_FORMAT,
    "candidate_score_policy": INTERPRETATION_CANDIDATE_SCORE_POLICY,
    "notebook09_required_compatibility_columns": notebook09_required_compatibility_columns,
    "stage2_required_columns": list(REQUIRED_CANDIDATE_COLUMNS),
    "candidate_brand_column": "candidate_brand_facet_text",
    "candidate_score_source_column": "candidate_score_source",
    "candidate_budget_policy": candidate_pool_manifest.get("candidate_budget_policy"),
    "candidate_budget_k": int(candidate_pool_manifest.get("candidate_budget_k")),
    "effective_candidate_count_per_query": int(
        candidate_pool_manifest.get("effective_candidate_count_per_query")
    ),
    "variable_candidate_count_allowed": bool(
        candidate_pool_manifest.get("variable_candidate_count_allowed")
    ),
    "candidate_scores_preserved": bool(candidate_pool_manifest.get("candidate_scores_preserved")),
    "retrieval_recomputed": bool(candidate_pool_manifest.get("retrieval_recomputed")),
    "complete_case_universe_preserved": bool(
        candidate_pool_manifest.get("complete_case_universe_preserved")
    ),
    "brand_in_candidate_output": bool(candidate_pool_manifest.get("brand_in_candidate_output")),
    "query_only_winner_contract_version": candidate_pool_manifest.get(
        "query_only_winner_contract_version"
    ),
    "personalized_winner_contract_version": candidate_pool_manifest.get(
        "personalized_winner_contract_version"
    ),
    "baseline_retrieval_winner_method_key": candidate_pool_manifest.get(
        "baseline_retrieval_winner_method_key"
    ),
    "baseline_retrieval_winner_method_label": candidate_pool_manifest.get(
        "baseline_retrieval_winner_method_label"
    ),
    "selected_personalized_method_slug": candidate_pool_manifest.get(
        "selected_personalized_method_slug"
    ),
    "selected_personalized_method_label": candidate_pool_manifest.get(
        "selected_personalized_method_label"
    ),
    "pool_depths": [int(depth) for depth in POOL_DEPTHS],
}
brand_all_prior_feature_contract_flags = {
    "brand_query_enabled": bool(BRAND_QUERY_ENABLED),
    "brand_candidate_visible": bool(BRAND_CANDIDATE_VISIBLE),
    "brand_reranking_enabled": bool(BRAND_RERANKING_ENABLED),
    "user_brand_affinity_enabled": bool(USER_BRAND_AFFINITY_ENABLED),
    "candidate_side_brand_feature_columns": candidate_side_brand_feature_columns,
    "candidate_side_brand_feature_count": int(len(candidate_side_brand_feature_columns)),
    "brand_affinity_feature_columns": brand_affinity_feature_columns,
    "brand_affinity_feature_count": int(len(brand_affinity_feature_columns)),
    "user_history_feature_columns": user_history_feature_columns,
    "user_history_feature_count": int(len(user_history_feature_columns)),
    "model_history_source": "none" if CONDITION_NAME == "P2-Q" else "all_prior",
    "diagnostic_history_source": HISTORY_SOURCE,
    "historical_population_review_signals_in_user_profile": False,
    "historical_population_review_signals_candidate_side_only": True,
}

model_contracts = []
model_artifacts = []
for pool_depth in POOL_DEPTHS:
    if CONDITION_NAME == "P2-Q":
        model_contract_path = OUT_DIR / f"p2q_model_contract_pool{pool_depth}.json"
        expected_model_source_condition = "P2-Q"
    else:
        model_contract_path = MODEL_SOURCE_ARTIFACT_DIR / f"full_model_contract_pool{pool_depth}.json"
        expected_model_source_condition = "Full"
    if not model_contract_path.exists():
        raise FileNotFoundError(f"Missing fold-model contract: {model_contract_path}")
    with open(model_contract_path, "r", encoding="utf-8") as file:
        model_contract = json.load(file)
    if model_contract.get("condition_name") != expected_model_source_condition:
        raise RuntimeError(f"Unexpected model-source condition in {model_contract_path}.")
    if model_contract.get("feature_columns") != model_feature_columns:
        raise RuntimeError(f"Model feature order mismatch in {model_contract_path}.")
    if model_contract.get("feature_dtypes") != {column: "float32" for column in model_feature_columns}:
        raise RuntimeError(f"Model feature dtype mismatch in {model_contract_path}.")
    if model_contract.get("feature_preprocessing_contract") != FEATURE_PREPROCESSING_CONTRACT:
        raise RuntimeError(f"Model preprocessing mismatch in {model_contract_path}.")
    if model_contract.get("ranker_params_base") != make_jsonable_dict(LIGHTGBM_RANKER_PARAMS_BASE):
        raise RuntimeError(f"LightGBM parameter mismatch in {model_contract_path}.")
    if int(model_contract.get("random_seed_base", -1)) != int(RANDOM_SEED):
        raise RuntimeError(f"Random-seed mismatch in {model_contract_path}.")
    model_contracts.append({
        "pool_depth": int(pool_depth),
        "path": str(model_contract_path),
        "fold_assignment_path": model_contract.get("fold_assignment_path"),
        "fallback_reason": model_contract.get("fallback_reason"),
    })
    for fold_contract in model_contract.get("folds", []):
        model_filename = fold_contract["model_filename"]
        model_path = MODEL_SOURCE_ARTIFACT_DIR / model_filename
        if not model_path.exists():
            raise FileNotFoundError(f"Missing fold-specific LightGBM model: {model_path}")
        model_artifacts.append({
            "pool_depth": int(pool_depth),
            "fold_id": int(fold_contract["fold"]),
            "path": str(model_path),
            "random_seed": int(fold_contract["random_seed"]),
        })

feature_interpretation_manifest = {
    "artifact_version": "lightgbm_heldout_feature_interpretation_v1",
    "contract_version": LIGHTGBM_MODEL_CONTRACT_VERSION,
    "model_contract_version": LIGHTGBM_MODEL_CONTRACT_VERSION,
    "condition_name": CONDITION_NAME,
    "category_id": CATEGORY_ID,
    "model_family": "lightgbm",
    "task_scope": "query_conditioned_next_novel_item_ranking",
    "specification_role": SPECIFICATION_ROLE,
    "primary_registry_policy_version": PRIMARY_REGISTRY_POLICY_VERSION,
    "exact_item_familiarity_enabled": ENABLE_EXACT_ITEM_FAMILIARITY,
    "exact_item_familiarity_feature_columns": list(EXACT_ITEM_FAMILIARITY_FEATURE_COLS),
    "model_source_condition": CONDITION_NAME,
    "feature_names": model_feature_columns,
    "feature_dtypes": {column: "float32" for column in model_feature_columns},
    "feature_count": int(len(model_feature_columns)),
    "feature_group_mapping_path": str(FEATURE_GROUP_MAPPING_PATH),
    "feature_preprocessing_contract": FEATURE_PREPROCESSING_CONTRACT,
    "ranker_params_base": make_jsonable_dict(LIGHTGBM_RANKER_PARAMS_BASE),
    "random_seed_base": int(RANDOM_SEED),
    "candidate_source_contract": candidate_source_contract,
    "brand_all_prior_feature_contract_flags": brand_all_prior_feature_contract_flags,
    "fold_assignment_path": str(FEATURE_INTERPRETATION_FOLD_ASSIGNMENT_PATH),
    "fold_assignment_row_count": int(len(feature_interpretation_fold_assignments_df)),
    "candidate_feature_oof_artifacts": feature_interpretation_artifact_rows,
    "model_contracts": model_contracts,
    "fold_models": model_artifacts,
    "oof_prediction_semantics": "final score used by the evaluated condition",
    "model_oof_prediction_semantics": "raw fold-model score before any P2-Q cold fallback",
    "p2p_full_feature_schema_equal": bool(feature_columns_P2P == feature_columns_Full),
    "p2p_full_feature_dtype_equal": bool(feature_dtypes_P2P == feature_dtypes_Full),
    "p2p_full_preprocessing_equal": True if CONDITION_NAME in {"P2-P", "Full"} else None,
    "p2p_full_fold_assignment_equal": None,  # retired with transport; Full self-trains on its own pool
    "shap_computed": False,
    "permutation_importance_computed": False,
}
FEATURE_INTERPRETATION_MANIFEST_PATH.write_text(
    json.dumps(make_jsonable_dict(feature_interpretation_manifest), ensure_ascii=False, indent=2),
    encoding="utf-8",
)

interpretation_output_paths = [
    FEATURE_INTERPRETATION_FOLD_ASSIGNMENT_PATH,
    FEATURE_GROUP_MAPPING_PATH,
    FEATURE_INTERPRETATION_MANIFEST_PATH,
] + [Path(row["path"]) for row in feature_interpretation_artifact_rows]
if "saved_paths" in globals():
    for path_obj in interpretation_output_paths:
        if path_obj not in saved_paths:
            saved_paths.append(path_obj)
if "run_manifest" in globals():
    run_manifest.setdefault("output_paths", {})
    run_manifest["output_paths"].update({
        "feature_interpretation_manifest": str(FEATURE_INTERPRETATION_MANIFEST_PATH),
        "feature_group_mapping": str(FEATURE_GROUP_MAPPING_PATH),
        "feature_interpretation_fold_assignments": str(FEATURE_INTERPRETATION_FOLD_ASSIGNMENT_PATH),
        "feature_interpretation_candidates": [
            row["path"] for row in feature_interpretation_artifact_rows
        ],
    })
    run_manifest["created_outputs"] = list(dict.fromkeys(
        run_manifest.get("created_outputs", []) + [str(path) for path in interpretation_output_paths]
    ))
    Path(run_manifest_path).write_text(
        json.dumps(make_jsonable_dict(run_manifest), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

print("Feature-interpretation artifacts saved:")
for path_obj in interpretation_output_paths:
    print("-", path_obj)


Feature-interpretation artifacts saved:
- /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/feature_interpretation_fold_assignments.parquet
- /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/feature_group_mapping.csv
- /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/feature_interpretation_manifest.json
- /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/heldout_feature_interpretation_pool100.parquet
- /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/heldout_feature_interpretation_pool300.parquet
- /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage2_personalized_rerank/lightgbm_full/heldout_feature_interpretation_pool500.parquet
- /content/drive/M

In [57]:
# ==== Matched LightGBM Contract QC ====
matched_training_contract_rows = []
for _depth in POOL_DEPTHS:
    if LIGHTGBM_VARIANT == "a":
        _model_contract_path = OUT_DIR / f"p2q_model_contract_pool{int(_depth)}.json"
        _fold_assignment_path = OUT_DIR / f"p2q_fold_assignment_pool{int(_depth)}.parquet"
    else:
        _model_contract_path = SHARED_MODEL_ARTIFACT_DIR / f"full_model_contract_pool{int(_depth)}.json"
        _fold_assignment_path = SHARED_MODEL_ARTIFACT_DIR / f"full_fold_assignment_pool{int(_depth)}.parquet"
    if not _model_contract_path.exists():
        raise FileNotFoundError(f"Missing Batch-2 LightGBM model contract: {_model_contract_path}")
    if not _fold_assignment_path.exists():
        raise FileNotFoundError(f"Missing Batch-2 LightGBM fold assignment: {_fold_assignment_path}")
    _contract = json.loads(_model_contract_path.read_text(encoding="utf-8"))
    if _contract.get("contract_version") != LIGHTGBM_MODEL_CONTRACT_VERSION:
        raise RuntimeError("Batch-2 LightGBM model-contract version mismatch.")
    if _contract.get("matched_fitting_policy") != LIGHTGBM_MATCHED_FITTING_POLICY:
        raise RuntimeError("Batch-2 matched fitting policy is not active.")
    if int(_contract.get("cold_queries_in_fitting", -1)) != 0:
        raise RuntimeError("Cold queries entered LightGBM fitting.")
    if int(_contract.get("n_folds", -1)) != 5:
        raise RuntimeError("LightGBM model contract does not use five folds.")
    _missing_fold_model_files = []
    for _fold_contract in _contract.get("folds", []):
        _model_filename = _fold_contract.get("model_filename")
        if not _model_filename:
            _missing_fold_model_files.append("<missing model_filename>")
            continue
        _model_path = _model_contract_path.parent / str(_model_filename)
        if not _model_path.exists():
            _missing_fold_model_files.append(str(_model_path))
    if _missing_fold_model_files:
        raise FileNotFoundError(
            "Missing Batch-2 LightGBM fold model files:\n"
            + "\n".join(_missing_fold_model_files[:20])
        )
    matched_training_contract_rows.append({
        "category_id": CATEGORY_ID,
        "variant": LIGHTGBM_VARIANT,
        "pool_depth": int(_depth),
        "training_plan_role": LIGHTGBM_TRAINING_PLAN_ROLE,
        "training_plan_hash": _contract.get("training_plan_hash", ""),
        "base_feature_contract_hash": _contract.get("base_feature_contract_hash", ""),
        "model_parameter_hash": _contract.get("model_parameter_hash", ""),
        "cold_queries_in_fitting": int(_contract.get("cold_queries_in_fitting", -1)),
        "fold_assignment_exists": bool(_fold_assignment_path.exists()),
        "missing_fold_model_files_n": int(len(_missing_fold_model_files)),
        "current_notebook_training_rows_used": 0 if LIGHTGBM_VARIANT == "c" else int(_contract.get("training_rows_used", 0)),
        "five_fold_contract_passed": int(_contract.get("n_folds", -1)) == 5,
        "full_load_only_passed": LIGHTGBM_VARIANT != "c" or all(
            bool(fold.get("model_sha256")) for fold in _contract.get("folds", [])
        ),
    })
matched_training_contract_qc_df = pd.DataFrame(matched_training_contract_rows)
if LIGHTGBM_VARIANT == "c" and not matched_training_contract_qc_df["current_notebook_training_rows_used"].eq(0).all():
    raise RuntimeError("Full LightGBM notebook performed model fitting.")
matched_training_contract_qc_path = OUT_DIR / "matched_training_contract_qc.csv"
matched_training_contract_qc_df.to_csv(matched_training_contract_qc_path, index=False)
if "saved_paths" in globals() and matched_training_contract_qc_path not in saved_paths:
    saved_paths.append(matched_training_contract_qc_path)
if "run_manifest" in globals():
    run_manifest.setdefault("output_paths", {})["matched_training_contract_qc"] = str(
        matched_training_contract_qc_path
    )
    run_manifest["matched_fitting_policy"] = LIGHTGBM_MATCHED_FITTING_POLICY
    run_manifest["lightgbm_training_plan_role"] = LIGHTGBM_TRAINING_PLAN_ROLE
    if "run_manifest_path" in globals():
        Path(run_manifest_path).write_text(
            json.dumps(make_jsonable_dict(run_manifest), ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
print("Batch-2 matched LightGBM common-framework validation: PASS")
display(matched_training_contract_qc_df)


Batch-2 matched LightGBM common-framework validation: PASS


,category_id,variant,pool_depth,training_plan_role,training_plan_hash,base_feature_contract_hash,model_parameter_hash,cold_queries_in_fitting,fold_assignment_exists,missing_fold_model_files_n,current_notebook_training_rows_used,five_fold_contract_passed,full_load_only_passed
0,face,b,100,write,ed716415a9cdd8e54563ae651ece41633a66788594f561...,0859c7adb488bd2f912b76f63759c047a0d58f811d75b1...,44a7a0af244b0abb963d418e590515f0c7a02bcabbad8e...,0,True,0,53100,True,True
1,face,b,300,write,797ea2fceb53d3caf53d7d7132bc684f71f4ed8a0c1dae...,0859c7adb488bd2f912b76f63759c047a0d58f811d75b1...,44a7a0af244b0abb963d418e590515f0c7a02bcabbad8e...,0,True,0,252000,True,True
2,face,b,500,write,38b9f76a843e92406205f7e14906a405d47313d6b6914b...,0859c7adb488bd2f912b76f63759c047a0d58f811d75b1...,44a7a0af244b0abb963d418e590515f0c7a02bcabbad8e...,0,True,0,542000,True,True
3,face,b,700,write,edb218d2dab69c0c8ba3a8c341e23ecf0afb3eb4089096...,0859c7adb488bd2f912b76f63759c047a0d58f811d75b1...,44a7a0af244b0abb963d418e590515f0c7a02bcabbad8e...,0,True,0,899500,True,True
4,face,b,1000,write,d0047d9aae7336ac669c0e8db214daf066c10cb6acf0a9...,0859c7adb488bd2f912b76f63759c047a0d58f811d75b1...,44a7a0af244b0abb963d418e590515f0c7a02bcabbad8e...,0,True,0,1513000,True,True


In [58]:
# ==== Primary Previously-Reviewed-Item Exclusion QC ====
_exact_item_columns = list(EXACT_ITEM_FAMILIARITY_FEATURE_COLS)
_primary_model_features = list(EXPECTED_MODEL_FEATURE_COLUMNS)
_exact_in_primary = sorted(set(_primary_model_features).intersection(_exact_item_columns))
if _exact_in_primary:
    raise RuntimeError(f"Previously-reviewed-item diagnostics entered the primary LightGBM model: {_exact_in_primary}")
if list(feature_interpretation_manifest.get("feature_names", [])) != _primary_model_features:
    raise RuntimeError("LightGBM interpretation feature order differs from the executed primary model order.")
if sorted(set(feature_interpretation_manifest.get("feature_names", [])).intersection(_exact_item_columns)):
    raise RuntimeError("LightGBM interpretation input includes previously-reviewed-item diagnostics.")
_seen_item_qc = {
    "task_scope": "query_conditioned_next_novel_item_ranking",
    "specification_role": SPECIFICATION_ROLE,
    "primary_model_features": _primary_model_features,
    "diagnostic_only_columns": _exact_item_columns,
    "diagnostic_columns_in_primary_model": _exact_in_primary,
    "candidate_export_may_retain_diagnostics": True,
    "model_input_excludes_diagnostics": True,
}
(OUT_DIR / "seen_item_primary_exclusion_qc.json").write_text(
    json.dumps(_seen_item_qc, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Seen-item primary exclusion QC passed.")


Seen-item primary exclusion QC passed.


In [59]:
# ==== Export Outputs — Part 10 ====
# Export the LightGBM early-stopping behavior summary.
try:
    _w25_lh = pd.DataFrame(LIGHTGBM_LEARNING_HISTORY_ROWS)
    if _w25_lh.empty:
        raise ValueError("empty in-memory history")
except Exception:
    _w25_lh = pd.read_csv(LIGHTGBM_LEARNING_HISTORY_PATH)
_w25_best = (
    _w25_lh.loc[_w25_lh["is_selected_best_iteration"].astype(bool)]
    .groupby(["category_id", "pool_depth", "outer_fold"], as_index=False)["iteration"].max()
    .rename(columns={"iteration": "best_iteration"})
)
lightgbm_earlystop_summary_df = _w25_best.groupby(["category_id", "pool_depth"], as_index=False).agg(
    n_folds=("best_iteration", "size"),
    mean_best_iteration=("best_iteration", "mean"),
    std_best_iteration=("best_iteration", "std"),
    min_best_iteration=("best_iteration", "min"),
    max_best_iteration=("best_iteration", "max"),
)
lightgbm_earlystop_summary_df.to_csv(OUT_DIR / "lightgbm_earlystop_summary.csv", index=False, encoding="utf-8-sig")
print("W2-5 lightgbm early-stopping summary rows:", len(lightgbm_earlystop_summary_df))
try:
    display(lightgbm_earlystop_summary_df)
except NameError:
    print(lightgbm_earlystop_summary_df)


W2-5 lightgbm early-stopping summary rows: 5


,category_id,pool_depth,n_folds,mean_best_iteration,std_best_iteration,min_best_iteration,max_best_iteration
0,face,100,5,66.6,38.875442,14,99
1,face,300,5,48.8,31.252200,11,92
2,face,500,5,70.4,53.868358,27,156
3,face,700,5,82.8,47.187922,31,139
4,face,1000,5,87.6,51.781271,10,135
